In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import zipfile, os

# Unzip
with zipfile.ZipFile('/content/drive/MyDrive/waste_clf_src.zip', 'r') as z:
    z.extractall('/content/waste-classification/')

# Verify files are there
print(os.listdir('/content/waste-classification/src'))

MessageError: Error: credential propagation was unsuccessful

In [ ]:
!pip install timm -q

import sys, os, importlib.util
import torch
import torch.nn as nn
import numpy as np
from sklearn.metrics import balanced_accuracy_score, classification_report

# Load our datasets.py directly to avoid conflict with HuggingFace datasets package
spec = importlib.util.spec_from_file_location(
    "our_datasets",
    "/content/waste-classification/src/datasets.py"
)
datasets_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(datasets_module)

get_trashnet_loaders = datasets_module.get_trashnet_loaders
CLASSES = datasets_module.CLASSES

# Load models.py
spec2 = importlib.util.spec_from_file_location(
    "models",
    "/content/waste-classification/src/models.py"
)
models_module = importlib.util.module_from_spec(spec2)
spec2.loader.exec_module(models_module)

get_model = models_module.get_model
count_parameters = models_module.count_parameters

print("All imports OK")
print("Classes:", CLASSES)

All imports OK
Classes: ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']


In [ ]:
# Config
DATA_ROOT = '/content/waste-classification/data/raw/trashnet'
SAVE_DIR = '/content/drive/MyDrive/waste_experiments'
os.makedirs(SAVE_DIR, exist_ok=True)
device = torch.device('cuda')

# Data
train_loader, val_loader, test_loader = get_trashnet_loaders(
    DATA_ROOT, batch_size=32, train_ratio=0.7, val_ratio=0.15
)

# Weighted loss
counts = np.array([403, 501, 410, 594, 482, 137], dtype=float)
weights = counts.sum() / (6 * counts)
criterion = nn.CrossEntropyLoss(weight=torch.FloatTensor(weights).to(device))

def train_model(model_name, num_epochs=20):
    print(f"\n{'='*50}")
    print(f"Training {model_name} for {num_epochs} epochs")
    print('='*50)

    model = get_model(model_name, num_classes=6).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

    best_val_loss = float('inf')

    for epoch in range(1, num_epochs + 1):
        # Train
        model.train()
        total_loss, all_preds, all_labels = 0, [], []
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            all_preds.extend(outputs.argmax(1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

        train_acc = balanced_accuracy_score(all_labels, all_preds)

        # Validate
        model.eval()
        val_loss, val_preds, val_labels = 0, [], []
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                val_loss += criterion(outputs, labels).item()
                val_preds.extend(outputs.argmax(1).cpu().numpy())
                val_labels.extend(labels.cpu().numpy())

        val_acc = balanced_accuracy_score(val_labels, val_preds)
        avg_val_loss = val_loss / len(val_loader)
        scheduler.step()

        print(f"Epoch {epoch}/{num_epochs} | Train: {train_acc:.4f} | Val: {val_acc:.4f} | Val Loss: {avg_val_loss:.4f}")

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'val_loss': best_val_loss,
                'val_balanced_acc': val_acc,
            }, f'{SAVE_DIR}/{model_name}_best.pth')
            print(f"  ✓ Saved best model")

    # Test
    model.eval()
    test_preds, test_labels = [], []
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            test_preds.extend(outputs.argmax(1).cpu().numpy())
            test_labels.extend(labels.cpu().numpy())

    test_acc = balanced_accuracy_score(test_labels, test_preds)
    report = classification_report(test_labels, test_preds, target_names=CLASSES, zero_division=0)
    print(f"\nFinal Test Balanced Accuracy: {test_acc:.4f}")
    print(report)
    return test_acc

# Train EfficientNet first
efficientnet_acc = train_model('efficientnet', num_epochs=20)

TrashNetDataset loaded: 2527 images from /content/waste-classification/data/raw/trashnet
TrashNetDataset loaded: 2527 images from /content/waste-classification/data/raw/trashnet
TrashNetDataset loaded: 2527 images from /content/waste-classification/data/raw/trashnet
Split — train: 1768, val: 379, test: 380

Training efficientnet for 20 epochs


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/49.3M [00:00<?, ?B/s]

Epoch 1/20 | Train: 0.4675 | Val: 0.6675 | Val Loss: 1.2393
  ✓ Saved best model
Epoch 2/20 | Train: 0.7441 | Val: 0.7465 | Val Loss: 0.8386
  ✓ Saved best model
Epoch 3/20 | Train: 0.7948 | Val: 0.7796 | Val Loss: 0.7173
  ✓ Saved best model
Epoch 4/20 | Train: 0.8531 | Val: 0.8224 | Val Loss: 0.6605
  ✓ Saved best model
Epoch 5/20 | Train: 0.8563 | Val: 0.7984 | Val Loss: 0.6574
  ✓ Saved best model
Epoch 6/20 | Train: 0.8910 | Val: 0.8245 | Val Loss: 0.6674
Epoch 7/20 | Train: 0.8861 | Val: 0.8116 | Val Loss: 0.6262
  ✓ Saved best model
Epoch 8/20 | Train: 0.9202 | Val: 0.8285 | Val Loss: 0.5418
  ✓ Saved best model
Epoch 9/20 | Train: 0.9180 | Val: 0.8425 | Val Loss: 0.5026
  ✓ Saved best model
Epoch 10/20 | Train: 0.9178 | Val: 0.8420 | Val Loss: 0.5130
Epoch 11/20 | Train: 0.9313 | Val: 0.8337 | Val Loss: 0.5606
Epoch 12/20 | Train: 0.9274 | Val: 0.8495 | Val Loss: 0.5001
  ✓ Saved best model
Epoch 13/20 | Train: 0.9278 | Val: 0.8569 | Val Loss: 0.5134
Epoch 14/20 | Train: 0.9371

In [ ]:
vit_acc = train_model('vit', num_epochs=20)



Training vit for 20 epochs


model.safetensors:   0%|          | 0.00/88.2M [00:00<?, ?B/s]

Epoch 1/20 | Train: 0.7366 | Val: 0.8926 | Val Loss: 0.3129
  ✓ Saved best model
Epoch 2/20 | Train: 0.9028 | Val: 0.9127 | Val Loss: 0.2555
  ✓ Saved best model
Epoch 3/20 | Train: 0.9289 | Val: 0.9005 | Val Loss: 0.2588
Epoch 4/20 | Train: 0.9259 | Val: 0.9170 | Val Loss: 0.2380
  ✓ Saved best model
Epoch 5/20 | Train: 0.9494 | Val: 0.9221 | Val Loss: 0.2040
  ✓ Saved best model
Epoch 6/20 | Train: 0.9629 | Val: 0.9237 | Val Loss: 0.2131
Epoch 7/20 | Train: 0.9587 | Val: 0.9208 | Val Loss: 0.2360
Epoch 8/20 | Train: 0.9645 | Val: 0.9480 | Val Loss: 0.1365
  ✓ Saved best model
Epoch 9/20 | Train: 0.9651 | Val: 0.9336 | Val Loss: 0.1789
Epoch 10/20 | Train: 0.9624 | Val: 0.9298 | Val Loss: 0.1983
Epoch 11/20 | Train: 0.9765 | Val: 0.9470 | Val Loss: 0.1756
Epoch 12/20 | Train: 0.9785 | Val: 0.9416 | Val Loss: 0.1650
Epoch 13/20 | Train: 0.9764 | Val: 0.9428 | Val Loss: 0.1984
Epoch 14/20 | Train: 0.9783 | Val: 0.9460 | Val Loss: 0.1715
Epoch 15/20 | Train: 0.9785 | Val: 0.9311 | Val Lo

In [ ]:
import os

# Set your Kaggle credentials directly
os.environ['KAGGLE_USERNAME'] = 'esmaeilmolapoor'  # replace with your username
os.environ['KAGGLE_KEY'] = 'YOUR_KAGGLE_KEY_HERE'

# Install kaggle
!pip install kaggle -q

# Verify it works
!kaggle datasets list --search "garbage classification" 2>&1 | head -5

ref                                                      title                                                 size  lastUpdated                 downloadCount  voteCount  usabilityRating  
-------------------------------------------------------  ----------------------------------------------  ----------  --------------------------  -------------  ---------  ---------------  
asdasdasasdas/garbage-classification                     Garbage Classification                            85969666  2018-11-24 05:09:23.977000          73060        659  0.8125           
mostafaabla/garbage-classification                       Garbage Classification (12 classes)              250641573  2021-01-24 13:53:11.647000          40649        263  0.8125           
hassnainzaidi/garbage-classification                     Garbage Classification                           128649799  2025-12-20 18:00:45.347000           1881         42  0.75             


In [ ]:
# Download Garbage Dataset (GD) — same 6 classes as TrashNet, easy merge
!kaggle datasets download -d asdasdasasdas/garbage-classification \
    -p /content/waste-classification/data/raw/ --unzip

# Download Household Garbage (12 classes)
!kaggle datasets download -d mostafaabla/garbage-classification \
    -p /content/waste-classification/data/raw/ --unzip

# Check what we got
!ls /content/waste-classification/data/raw/


Dataset URL: https://www.kaggle.com/datasets/asdasdasasdas/garbage-classification
License(s): copyright-authors
100% 82.0M/82.0M [00:00<00:00, 232MB/s]

Dataset URL: https://www.kaggle.com/datasets/mostafaabla/garbage-classification
License(s): ODbL-1.0
100% 239M/239M [00:00<00:00, 300MB/s]

'garbage classification'	       one-indexed-files-notrash_val.txt
 garbage_classification		       one-indexed-files.txt
'Garbage classification'	       trashnet
 one-indexed-files-notrash_test.txt    zero-indexed-files.txt
 one-indexed-files-notrash_train.txt


In [ ]:
import os

# Check Garbage Dataset (GD) structure
print("=== Garbage Classification (GD) ===")
gd_path = '/content/waste-classification/data/raw/garbage_classification'
if os.path.exists(gd_path):
    print(os.listdir(gd_path))
else:
    # try other name
    for item in os.listdir('/content/waste-classification/data/raw/'):
        print(item)

print("\n=== Household Garbage (12 classes) ===")
hg_path = '/content/waste-classification/data/raw/Garbage classification'
if os.path.exists(hg_path):
    print(os.listdir(hg_path))

=== Garbage Classification (GD) ===
['trash', 'metal', 'paper', 'clothes', 'brown-glass', 'shoes', 'battery', 'cardboard', 'white-glass', 'plastic', 'green-glass', 'biological']

=== Household Garbage (12 classes) ===
['Garbage classification']


In [ ]:
# Check nested structure
import os

hg_inner = '/content/waste-classification/data/raw/Garbage classification/Garbage classification'
print("Household Garbage classes:", os.listdir(hg_inner))

# Check GD image counts
gd_path = '/content/waste-classification/data/raw/garbage_classification'
print("\nGD class counts:")
for cls in sorted(os.listdir(gd_path)):
    folder = os.path.join(gd_path, cls)
    if os.path.isdir(folder):
        count = len([f for f in os.listdir(folder) if f.lower().endswith(('.jpg','.jpeg','.png'))])
        print(f"  {cls}: {count}")

Household Garbage classes: ['glass', 'trash', 'metal', 'paper', 'cardboard', 'plastic']

GD class counts:
  battery: 945
  biological: 985
  brown-glass: 607
  cardboard: 891
  clothes: 5325
  green-glass: 629
  metal: 769
  paper: 1050
  plastic: 865
  shoes: 1977
  trash: 697
  white-glass: 775


In [ ]:
hg_path = '/content/waste-classification/data/raw/Garbage classification/Garbage classification'
print("Household Garbage class counts:")
for cls in sorted(os.listdir(hg_path)):
    folder = os.path.join(hg_path, cls)
    if os.path.isdir(folder):
        count = len([f for f in os.listdir(folder)
                     if f.lower().endswith(('.jpg','.jpeg','.png'))])
        print(f"  {cls}: {count}")


Household Garbage class counts:
  cardboard: 403
  glass: 501
  metal: 410
  paper: 594
  plastic: 482
  trash: 137


In [ ]:
import os
import torch
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from PIL import Image
from pathlib import Path
import importlib.util

# Load our transforms
spec = importlib.util.spec_from_file_location(
    "our_datasets",
    "/content/waste-classification/src/datasets.py"
)
datasets_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(datasets_module)
get_transforms = datasets_module.get_transforms

CLASSES = ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']
CLASS_TO_IDX = {cls: idx for idx, cls in enumerate(CLASSES)}

# GD dataset class mapping
GD_MAP = {
    'cardboard':   'cardboard',
    'brown-glass': 'glass',
    'green-glass': 'glass',
    'white-glass': 'glass',
    'metal':       'metal',
    'paper':       'paper',
    'plastic':     'plastic',
    'trash':       'trash',
    'battery':     'trash',
    'biological':  'trash',
    'clothes':     'trash',
    'shoes':       'trash',
}

class GDDataset(Dataset):
    def __init__(self, root, transform=None):
        self.root = Path(root)
        self.transform = transform
        self.samples = []

        for folder_name, unified_name in GD_MAP.items():
            class_dir = self.root / folder_name
            if not class_dir.exists():
                continue
            label_idx = CLASS_TO_IDX[unified_name]
            for img_file in class_dir.iterdir():
                if img_file.suffix.lower() in ['.jpg', '.jpeg', '.png']:
                    self.samples.append((str(img_file), label_idx))

        print(f"GDDataset loaded: {len(self.samples)} images")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label

# Load both datasets
trashnet = datasets_module.TrashNetDataset(
    '/content/waste-classification/data/raw/trashnet',
    transform=get_transforms('train')
)
gd = GDDataset(
    '/content/waste-classification/data/raw/garbage_classification',
    transform=get_transforms('train')
)

# Combine
combined = ConcatDataset([trashnet, gd])
print(f"\nCombined training set: {len(combined)} images")
print(f"  TrashNet: {len(trashnet)}")
print(f"  GD:       {len(gd)}")

# Split combined into train/val/test (70/15/15)
import numpy as np
n = len(combined)
indices = torch.randperm(n, generator=torch.Generator().manual_seed(42)).tolist()
n_train = int(0.70 * n)
n_val   = int(0.15 * n)
n_test  = n - n_train - n_val

from torch.utils.data import Subset
train_set = Subset(combined, indices[:n_train])
val_set   = Subset(combined, indices[n_train:n_train+n_val])
test_set  = Subset(combined, indices[n_train+n_val:])

train_loader = DataLoader(train_set, batch_size=32, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_set,   batch_size=32, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_set,  batch_size=32, shuffle=False, num_workers=2)

print(f"\nSplit — train: {len(train_set)}, val: {len(val_set)}, test: {len(test_set)}")

TrashNetDataset loaded: 2527 images from /content/waste-classification/data/raw/trashnet
GDDataset loaded: 15515 images

Combined training set: 18042 images
  TrashNet: 2527
  GD:       15515

Split — train: 12629, val: 2706, test: 2707


In [ ]:
import numpy as np
import torch.nn as nn
from sklearn.metrics import balanced_accuracy_score, classification_report

# Weighted loss — recompute for combined dataset
# GD counts after mapping:
# cardboard: 891, glass: 607+629+775=2011, metal: 769,
# paper: 1050, plastic: 865, trash: 697+945+985+5325+1977=9929
# + TrashNet: cardboard:403, glass:501, metal:410, paper:594, plastic:482, trash:137
combined_counts = np.array([
    891+403,    # cardboard: 1294
    2011+501,   # glass: 2512
    769+410,    # metal: 1179
    1050+594,   # paper: 1644
    865+482,    # plastic: 1347
    9929+137,   # trash: 10066
], dtype=float)

weights = combined_counts.sum() / (6 * combined_counts)
print("Class weights:")
for cls, w in zip(CLASSES, weights):
    print(f"  {cls}: {w:.3f}")

criterion = nn.CrossEntropyLoss(
    weight=torch.FloatTensor(weights).to(device)
)

Class weights:
  cardboard: 2.324
  glass: 1.197
  metal: 2.550
  paper: 1.829
  plastic: 2.232
  trash: 0.299


In [ ]:
def train_model_combined(model_name, num_epochs=20):
    print(f"\n{'='*50}")
    print(f"Training {model_name} on COMBINED dataset ({len(combined)} images)")
    print('='*50)

    model = get_model(model_name, num_classes=6).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

    best_val_loss = float('inf')
    best_val_acc = 0

    for epoch in range(1, num_epochs + 1):
        # Train
        model.train()
        total_loss, all_preds, all_labels = 0, [], []
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            all_preds.extend(outputs.argmax(1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

        train_acc = balanced_accuracy_score(all_labels, all_preds)

        # Validate
        model.eval()
        val_loss, val_preds, val_labels = 0, [], []
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                val_loss += criterion(outputs, labels).item()
                val_preds.extend(outputs.argmax(1).cpu().numpy())
                val_labels.extend(labels.cpu().numpy())

        val_acc = balanced_accuracy_score(val_labels, val_preds)
        avg_val_loss = val_loss / len(val_loader)
        scheduler.step()

        print(f"Epoch {epoch}/{num_epochs} | Train: {train_acc:.4f} | Val: {val_acc:.4f} | Val Loss: {avg_val_loss:.4f}")

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_val_acc = val_acc
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'val_loss': best_val_loss,
                'val_balanced_acc': val_acc,
            }, f'{SAVE_DIR}/{model_name}_combined_best.pth')
            print(f"  ✓ Saved best model")

    # Test
    model.eval()
    test_preds, test_labels = [], []
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            test_preds.extend(outputs.argmax(1).cpu().numpy())
            test_labels.extend(labels.cpu().numpy())

    test_acc = balanced_accuracy_score(test_labels, test_preds)
    report = classification_report(
        test_labels, test_preds,
        target_names=CLASSES, zero_division=0
    )
    print(f"\nFinal Test Balanced Accuracy: {test_acc:.4f}")
    print(report)
    return test_acc

# Train all three — this will take ~45-60 min total
resnet_combined  = train_model_combined('resnet50',     num_epochs=20)
effnet_combined  = train_model_combined('efficientnet', num_epochs=20)
vit_combined     = train_model_combined('vit',          num_epochs=20)

NameError: name 'combined' is not defined

In [ ]:
import os

# Check what's actually in Drive
print(os.listdir('/content/drive/MyDrive/'))

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/'

In [ ]:
from google.colab import files
uploaded = files.upload()  # click this and select waste_clf_src.zip from your Desktop

Saving waste_clf_src.zip to waste_clf_src.zip


In [ ]:
import zipfile, os
with zipfile.ZipFile('/content/waste_clf_src.zip', 'r') as z:
    z.extractall('/content/waste-classification/')
print(os.listdir('/content/waste-classification/src'))

['xai.py', 'config.py', 'models.py', 'metrics.py', 'train.py.backup', 'evaluate.py', 'experiment_log.py', 'data_audit.py', '__pycache__', 'train.py', '__init__.py', 'datasets.py']


In [ ]:
import os, sys, importlib.util, torch, torch.nn as nn
import numpy as np
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader, ConcatDataset, Subset
from sklearn.metrics import balanced_accuracy_score, classification_report

# Setup
SAVE_DIR = '/content/drive/MyDrive/waste_experiments'
os.makedirs(SAVE_DIR, exist_ok=True)
device = torch.device('cuda')

# Load modules
def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

datasets_module = load_module("our_datasets", "/content/waste-classification/src/datasets.py")
models_module   = load_module("models", "/content/waste-classification/src/models.py")
get_transforms  = datasets_module.get_transforms
get_model       = models_module.get_model
CLASSES         = ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']
CLASS_TO_IDX    = {cls: idx for idx, cls in enumerate(CLASSES)}

# Download GD dataset
os.environ['KAGGLE_USERNAME'] = 'esmaeilmolapour'
os.environ['KAGGLE_KEY'] = 'YOUR_KAGGLE_KEY_HERE'
!pip install kaggle -q
!kaggle datasets download -d asdasdasasdas/garbage-classification \
    -p /content/waste-classification/data/raw/ --unzip -q

# GD Dataset class
GD_MAP = {
    'cardboard':'cardboard','brown-glass':'glass','green-glass':'glass',
    'white-glass':'glass','metal':'metal','paper':'paper','plastic':'plastic',
    'trash':'trash','battery':'trash','biological':'trash',
    'clothes':'trash','shoes':'trash',
}

class GDDataset(Dataset):
    def __init__(self, root, transform=None):
        self.root = Path(root)
        self.transform = transform
        self.samples = []
        for folder_name, unified_name in GD_MAP.items():
            class_dir = self.root / folder_name
            if not class_dir.exists(): continue
            label_idx = CLASS_TO_IDX[unified_name]
            for img_file in class_dir.iterdir():
                if img_file.suffix.lower() in ['.jpg','.jpeg','.png']:
                    self.samples.append((str(img_file), label_idx))
        print(f"GDDataset: {len(self.samples)} images")
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform: image = self.transform(image)
        return image, label

# Load & combine
trashnet = datasets_module.TrashNetDataset(
    '/content/waste-classification/data/raw/trashnet',
    transform=get_transforms('train'))
gd = GDDataset('/content/waste-classification/data/raw/garbage_classification',
               transform=get_transforms('train'))
combined = ConcatDataset([trashnet, gd])

# Split
n = len(combined)
indices = torch.randperm(n, generator=torch.Generator().manual_seed(42)).tolist()
n_train, n_val = int(0.70*n), int(0.15*n)
n_test = n - n_train - n_val
train_loader = DataLoader(Subset(combined, indices[:n_train]), batch_size=32, shuffle=True, num_workers=2)
val_loader   = DataLoader(Subset(combined, indices[n_train:n_train+n_val]), batch_size=32, shuffle=False, num_workers=2)
test_loader  = DataLoader(Subset(combined, indices[n_train+n_val:]), batch_size=32, shuffle=False, num_workers=2)

# Weighted loss
combined_counts = np.array([1294, 2512, 1179, 1644, 1347, 10066], dtype=float)
weights = combined_counts.sum() / (6 * combined_counts)
criterion = nn.CrossEntropyLoss(weight=torch.FloatTensor(weights).to(device))

print(f"\nCombined: {n} images — train:{n_train}, val:{n_val}, test:{n_test}")
print("Setup complete. Ready to train.")

Dataset URL: https://www.kaggle.com/datasets/asdasdasasdas/garbage-classification
License(s): copyright-authors
TrashNetDataset loaded: 2527 images from /content/waste-classification/data/raw/trashnet
GDDataset: 0 images

Combined: 2527 images — train:1768, val:379, test:380
Setup complete. Ready to train.


In [ ]:
import os
print(os.listdir('/content/waste-classification/data/raw/'))

['one-indexed-files.txt', 'one-indexed-files-notrash_test.txt', 'garbage classification', 'zero-indexed-files.txt', 'Garbage classification', 'one-indexed-files-notrash_val.txt', 'one-indexed-files-notrash_train.txt', 'trashnet']


In [ ]:
import os
# Check which one has the class folders
print(os.listdir('/content/waste-classification/data/raw/Garbage classification'))

['Garbage classification']


In [ ]:
gd = GDDataset('/content/waste-classification/data/raw/Garbage classification',
               transform=get_transforms('train'))
combined = ConcatDataset([trashnet, gd])

n = len(combined)
indices = torch.randperm(n, generator=torch.Generator().manual_seed(42)).tolist()
n_train, n_val = int(0.70*n), int(0.15*n)
n_test = n - n_train - n_val
train_loader = DataLoader(Subset(combined, indices[:n_train]), batch_size=32, shuffle=True, num_workers=2)
val_loader   = DataLoader(Subset(combined, indices[n_train:n_train+n_val]), batch_size=32, shuffle=False, num_workers=2)
test_loader  = DataLoader(Subset(combined, indices[n_train+n_val:]), batch_size=32, shuffle=False, num_workers=2)

print(f"Combined: {len(combined)} images — train:{n_train}, val:{n_val}, test:{n_test}")

GDDataset: 0 images
Combined: 2527 images — train:1768, val:379, test:380


In [ ]:
print(os.listdir('/content/waste-classification/data/raw/Garbage classification/Garbage classification'))

['glass', 'trash', 'metal', 'paper', 'cardboard', 'plastic']


In [ ]:
print(os.listdir('/content/waste-classification/data/raw/garbage classification'))

['Garbage classification']


In [ ]:
# Check the lowercase folder nested
print(os.listdir('/content/waste-classification/data/raw/garbage classification/Garbage classification'))

['glass', 'trash', 'metal', 'paper', 'cardboard', 'plastic']


In [ ]:
!kaggle datasets download -d mostafaabla/garbage-classification \
    -p /content/waste-classification/data/raw/gd12/ --unzip -q

import os
print(os.listdir('/content/waste-classification/data/raw/gd12/'))

Dataset URL: https://www.kaggle.com/datasets/mostafaabla/garbage-classification
License(s): ODbL-1.0
['garbage_classification']


In [ ]:
print(os.listdir('/content/waste-classification/data/raw/gd12/garbage_classification'))

['trash', 'metal', 'paper', 'clothes', 'brown-glass', 'shoes', 'battery', 'cardboard', 'white-glass', 'plastic', 'green-glass', 'biological']


In [ ]:
gd = GDDataset('/content/waste-classification/data/raw/gd12/garbage_classification',
               transform=get_transforms('train'))
combined = ConcatDataset([trashnet, gd])

n = len(combined)
indices = torch.randperm(n, generator=torch.Generator().manual_seed(42)).tolist()
n_train, n_val = int(0.70*n), int(0.15*n)
n_test = n - n_train - n_val
train_loader = DataLoader(Subset(combined, indices[:n_train]), batch_size=32, shuffle=True, num_workers=2)
val_loader   = DataLoader(Subset(combined, indices[n_train:n_train+n_val]), batch_size=32, shuffle=False, num_workers=2)
test_loader  = DataLoader(Subset(combined, indices[n_train+n_val:]), batch_size=32, shuffle=False, num_workers=2)

print(f"Combined: {len(combined)} images — train:{n_train}, val:{n_val}, test:{n_test}")

GDDataset: 15515 images
Combined: 18042 images — train:12629, val:2706, test:2707


In [ ]:
def train_model_combined(model_name, num_epochs=20):
    print(f"\n{'='*50}")
    print(f"Training {model_name} on COMBINED dataset ({len(combined)} images)")
    print('='*50)

    model = get_model(model_name, num_classes=6).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
    best_val_loss = float('inf')

    for epoch in range(1, num_epochs + 1):
        model.train()
        total_loss, all_preds, all_labels = 0, [], []
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            all_preds.extend(outputs.argmax(1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
        train_acc = balanced_accuracy_score(all_labels, all_preds)

        model.eval()
        val_loss, val_preds, val_labels = 0, [], []
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                val_loss += criterion(outputs, labels).item()
                val_preds.extend(outputs.argmax(1).cpu().numpy())
                val_labels.extend(labels.cpu().numpy())
        val_acc = balanced_accuracy_score(val_labels, val_preds)
        avg_val_loss = val_loss / len(val_loader)
        scheduler.step()
        print(f"Epoch {epoch}/{num_epochs} | Train: {train_acc:.4f} | Val: {val_acc:.4f} | Loss: {avg_val_loss:.4f}")

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                        'val_loss': best_val_loss, 'val_balanced_acc': val_acc},
                       f'{SAVE_DIR}/{model_name}_combined_best.pth')
            print(f"  ✓ Saved")

    model.eval()
    test_preds, test_labels = [], []
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            test_preds.extend(outputs.argmax(1).cpu().numpy())
            test_labels.extend(labels.cpu().numpy())

    test_acc = balanced_accuracy_score(test_labels, test_preds)
    report = classification_report(test_labels, test_preds, target_names=CLASSES, zero_division=0)
    print(f"\nFinal Test Balanced Accuracy: {test_acc:.4f}")
    print(report)
    return test_acc

resnet_combined  = train_model_combined('resnet50',     num_epochs=20)
effnet_combined  = train_model_combined('efficientnet', num_epochs=20)
vit_combined     = train_model_combined('vit',          num_epochs=20)


Training resnet50 on COMBINED dataset (18042 images)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]

Epoch 1/20 | Train: 0.6638 | Val: 0.8347 | Loss: 0.5075
  ✓ Saved
Epoch 2/20 | Train: 0.8450 | Val: 0.8944 | Loss: 0.3157
  ✓ Saved
Epoch 3/20 | Train: 0.8792 | Val: 0.9055 | Loss: 0.2963
  ✓ Saved
Epoch 4/20 | Train: 0.9041 | Val: 0.9135 | Loss: 0.2587
  ✓ Saved
Epoch 5/20 | Train: 0.9145 | Val: 0.9262 | Loss: 0.2235
  ✓ Saved
Epoch 6/20 | Train: 0.9262 | Val: 0.9380 | Loss: 0.1890
  ✓ Saved
Epoch 7/20 | Train: 0.9332 | Val: 0.9381 | Loss: 0.1997
Epoch 8/20 | Train: 0.9388 | Val: 0.9379 | Loss: 0.1919
Epoch 9/20 | Train: 0.9402 | Val: 0.9383 | Loss: 0.1836
  ✓ Saved
Epoch 10/20 | Train: 0.9469 | Val: 0.9445 | Loss: 0.1518
  ✓ Saved
Epoch 11/20 | Train: 0.9531 | Val: 0.9569 | Loss: 0.1419
  ✓ Saved
Epoch 12/20 | Train: 0.9459 | Val: 0.9458 | Loss: 0.1573
Epoch 13/20 | Train: 0.9494 | Val: 0.9505 | Loss: 0.1436
Epoch 14/20 | Train: 0.9500 | Val: 0.9485 | Loss: 0.1604
Epoch 15/20 | Train: 0.9529 | Val: 0.9446 | Loss: 0.1599
Epoch 16/20 | Train: 0.9553 | Val: 0.9472 | Loss: 0.1619
Epoch 1

model.safetensors:   0%|          | 0.00/49.3M [00:00<?, ?B/s]

Epoch 1/20 | Train: 0.7409 | Val: 0.8564 | Loss: 0.7269
  ✓ Saved
Epoch 2/20 | Train: 0.8839 | Val: 0.9103 | Loss: 0.2682
  ✓ Saved
Epoch 3/20 | Train: 0.9226 | Val: 0.9221 | Loss: 0.2189
  ✓ Saved
Epoch 4/20 | Train: 0.9447 | Val: 0.9380 | Loss: 0.1925
  ✓ Saved
Epoch 5/20 | Train: 0.9524 | Val: 0.9428 | Loss: 0.1729
  ✓ Saved
Epoch 6/20 | Train: 0.9542 | Val: 0.9407 | Loss: 0.2005
Epoch 7/20 | Train: 0.9624 | Val: 0.9406 | Loss: 0.1906
Epoch 8/20 | Train: 0.9676 | Val: 0.9359 | Loss: 0.2136
Epoch 9/20 | Train: 0.9710 | Val: 0.9457 | Loss: 0.1737
Epoch 10/20 | Train: 0.9706 | Val: 0.9576 | Loss: 0.1473
  ✓ Saved
Epoch 11/20 | Train: 0.9725 | Val: 0.9522 | Loss: 0.1581
Epoch 12/20 | Train: 0.9702 | Val: 0.9576 | Loss: 0.1380
  ✓ Saved
Epoch 13/20 | Train: 0.9786 | Val: 0.9557 | Loss: 0.1620
Epoch 14/20 | Train: 0.9766 | Val: 0.9580 | Loss: 0.1447
Epoch 15/20 | Train: 0.9802 | Val: 0.9512 | Loss: 0.1402
Epoch 16/20 | Train: 0.9821 | Val: 0.9583 | Loss: 0.1224
  ✓ Saved
Epoch 17/20 | Tra

model.safetensors:   0%|          | 0.00/88.2M [00:00<?, ?B/s]

Epoch 1/20 | Train: 0.8712 | Val: 0.9006 | Loss: 0.2730
  ✓ Saved
Epoch 2/20 | Train: 0.9260 | Val: 0.9133 | Loss: 0.2515
  ✓ Saved
Epoch 3/20 | Train: 0.9397 | Val: 0.9179 | Loss: 0.2126
  ✓ Saved
Epoch 4/20 | Train: 0.9467 | Val: 0.9319 | Loss: 0.2001
  ✓ Saved
Epoch 5/20 | Train: 0.9572 | Val: 0.9405 | Loss: 0.1941
  ✓ Saved
Epoch 6/20 | Train: 0.9523 | Val: 0.9366 | Loss: 0.1900
  ✓ Saved
Epoch 7/20 | Train: 0.9638 | Val: 0.9333 | Loss: 0.2022
Epoch 8/20 | Train: 0.9704 | Val: 0.9390 | Loss: 0.1714
  ✓ Saved
Epoch 9/20 | Train: 0.9652 | Val: 0.9389 | Loss: 0.1668
  ✓ Saved
Epoch 10/20 | Train: 0.9766 | Val: 0.9510 | Loss: 0.1484
  ✓ Saved
Epoch 11/20 | Train: 0.9805 | Val: 0.9436 | Loss: 0.1723
Epoch 12/20 | Train: 0.9816 | Val: 0.9555 | Loss: 0.1551
Epoch 13/20 | Train: 0.9841 | Val: 0.9530 | Loss: 0.1400
  ✓ Saved
Epoch 14/20 | Train: 0.9859 | Val: 0.9535 | Loss: 0.1535
Epoch 15/20 | Train: 0.9856 | Val: 0.9632 | Loss: 0.1400
Epoch 16/20 | Train: 0.9853 | Val: 0.9630 | Loss: 0.10

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from google.colab import files
uploaded = files.upload()  # upload waste_clf_src.zip from your Mac

import zipfile, os
with zipfile.ZipFile('/content/waste_clf_src.zip', 'r') as z:
    z.extractall('/content/waste-classification/')

import importlib.util, torch, torch.nn as nn
import numpy as np
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import balanced_accuracy_score, classification_report

device = torch.device('cuda')
SAVE_DIR = '/content/drive/MyDrive/waste_experiments'
CLASSES = ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']
CLASS_TO_IDX = {cls: idx for idx, cls in enumerate(CLASSES)}

def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

models_module = load_module("models", "/content/waste-classification/src/models.py")
get_model = models_module.get_model

# Verify saved models are in Drive
print("Saved models:", os.listdir(SAVE_DIR))

Mounted at /content/drive


Saving waste_clf_src.zip to waste_clf_src.zip
Saved models: ['efficientnet_best.pth', 'vit_best.pth', 'resnet50_combined_best.pth', 'efficientnet_combined_best.pth', 'vit_combined_best.pth']


Saved models: ['efficientnet_best.pth', 'vit_best.pth', 'resnet50_combined_best.pth', 'efficientnet_combined_best.pth', 'vit_combined_best.pth']


Dataset URL: https://www.kaggle.com/datasets/mostafaabla/garbage-classification
License(s): ODbL-1.0
['trash', 'cardboard', 'brown-glass', 'plastic', 'battery', 'biological', 'metal', 'shoes', 'white-glass', 'paper', 'clothes', 'green-glass']


In [ ]:
os.environ['KAGGLE_USERNAME'] = 'esmaeilmolapour'
os.environ['KAGGLE_KEY'] = 'YOUR_KAGGLE_KEY_HERE'
!pip install kaggle -q

# Try the correct slug
!kaggle datasets download -d joebeachcapital/realwaste \
    -p /content/waste-classification/data/raw/realwaste/ --unzip -q

!ls /content/waste-classification/data/raw/realwaste/

Dataset URL: https://www.kaggle.com/datasets/joebeachcapital/realwaste
License(s): Attribution 4.0 International (CC BY 4.0)
realwaste-main


In [ ]:
import os
print(os.listdir('/content/waste-classification/data/raw/realwaste/realwaste-main'))


['RealWaste', 'README.md']


In [ ]:
print(os.listdir('/content/waste-classification/data/raw/realwaste/realwaste-main/RealWaste'))

['Cardboard', 'Metal', 'Vegetation', 'Paper', 'Miscellaneous Trash', 'Textile Trash', 'Glass', 'Food Organics', 'Plastic']


In [ ]:
device = torch.device('cpu')
print(f"Device: {device} (no GPU available, using CPU)")

def evaluate_ood(model_name, checkpoint_path):
    print(f"\n{'='*50}")
    print(f"OOD Evaluation: {model_name}")
    print('='*50)

    model = get_model(model_name, num_classes=6)
    checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    model = model.to(device)
    model.eval()

    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in realwaste_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            all_preds.extend(outputs.argmax(1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    acc = balanced_accuracy_score(all_labels, all_preds)
    report = classification_report(all_labels, all_preds, target_names=CLASSES, zero_division=0)
    print(f"OOD Balanced Accuracy: {acc:.4f} ({acc*100:.1f}%)")
    print(report)
    return acc

resnet_ood  = evaluate_ood('resnet50',     f'{SAVE_DIR}/resnet50_combined_best.pth')
effnet_ood  = evaluate_ood('efficientnet', f'{SAVE_DIR}/efficientnet_combined_best.pth')
vit_ood     = evaluate_ood('vit',          f'{SAVE_DIR}/vit_combined_best.pth')

print("\n" + "="*50)
print("DOMAIN GENERALIZATION SUMMARY")
print("="*50)
models = ['ResNet-50', 'EfficientNet-B3', 'ViT-Small']
indist = [0.955, 0.960, 0.974]
oods   = [resnet_ood, effnet_ood, vit_ood]
for m, i, o in zip(models, indist, oods):
    print(f"{m:18s} | In-dist: {i*100:.1f}% | OOD: {o*100:.1f}% | Gap: -{(i-o)*100:.1f}%")

Device: cpu (no GPU available, using CPU)

OOD Evaluation: resnet50
OOD Balanced Accuracy: 0.5995 (60.0%)
              precision    recall  f1-score   support

   cardboard       0.41      0.60      0.49       461
       glass       0.58      0.44      0.50       420
       metal       0.69      0.68      0.69       790
       paper       0.50      0.55      0.53       500
     plastic       0.74      0.56      0.64       921
       trash       0.73      0.76      0.75      1660

    accuracy                           0.64      4752
   macro avg       0.61      0.60      0.60      4752
weighted avg       0.66      0.64      0.65      4752


OOD Evaluation: efficientnet
OOD Balanced Accuracy: 0.5360 (53.6%)
              precision    recall  f1-score   support

   cardboard       0.54      0.41      0.46       461
       glass       0.49      0.45      0.47       420
       metal       0.69      0.64      0.66       790
       paper       0.63      0.33      0.43       500
     plastic

In [ ]:
print("Device:", device)
print("Models module:", get_model)
print("RealWaste loader:", len(realwaste_loader), "batches")
print("Saved models:", os.listdir(SAVE_DIR))

NameError: name 'device' is not defined

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from google.colab import files
uploaded = files.upload()  # upload waste_clf_src.zip

Mounted at /content/drive


Saving waste_clf_src.zip to waste_clf_src.zip


In [ ]:
import zipfile, os, importlib.util, torch, torch.nn as nn
import numpy as np
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import balanced_accuracy_score, classification_report
from torchvision import transforms

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SAVE_DIR = '/content/drive/MyDrive/waste_experiments'
CLASSES = ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']
CLASS_TO_IDX = {cls: idx for idx, cls in enumerate(CLASSES)}

with zipfile.ZipFile('/content/waste_clf_src.zip', 'r') as z:
    z.extractall('/content/waste-classification/')

def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

models_module = load_module("models", "/content/waste-classification/src/models.py")
get_model = models_module.get_model

# RealWaste dataset
REALWASTE_ROOT = '/content/waste-classification/data/raw/realwaste/realwaste-main/RealWaste'

os.environ['KAGGLE_USERNAME'] = 'esmaeilmolapour'
os.environ['KAGGLE_KEY'] = 'YOUR_KAGGLE_KEY_HERE'
!pip install kaggle -q
!kaggle datasets download -d joebeachcapital/realwaste \
    -p /content/waste-classification/data/raw/realwaste/ --unzip -q

REALWASTE_MAP = {
    'Cardboard': 'cardboard', 'Glass': 'glass', 'Metal': 'metal',
    'Paper': 'paper', 'Plastic': 'plastic', 'Miscellaneous Trash': 'trash',
    'Food Organics': 'trash', 'Textile Trash': 'trash', 'Vegetation': 'trash',
}

eval_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class RealWasteDataset(Dataset):
    def __init__(self, root, transform=None):
        self.root = Path(root)
        self.transform = transform
        self.samples = []
        for folder_name, unified_name in REALWASTE_MAP.items():
            class_dir = self.root / folder_name
            if not class_dir.exists(): continue
            label_idx = CLASS_TO_IDX[unified_name]
            for img_file in class_dir.iterdir():
                if img_file.suffix.lower() in ['.jpg','.jpeg','.png']:
                    self.samples.append((str(img_file), label_idx))
        print(f"RealWasteDataset: {len(self.samples)} images")
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform: image = self.transform(image)
        return image, label

realwaste_dataset = RealWasteDataset(REALWASTE_ROOT, transform=eval_transform)
realwaste_loader = DataLoader(realwaste_dataset, batch_size=32, shuffle=False, num_workers=2)

print(f"\nDevice: {device}")
print(f"Saved models: {os.listdir(SAVE_DIR)}")
print("Setup complete.")

Dataset URL: https://www.kaggle.com/datasets/joebeachcapital/realwaste
License(s): Attribution 4.0 International (CC BY 4.0)
RealWasteDataset: 4752 images

Device: cuda
Saved models: ['efficientnet_best.pth', 'vit_best.pth', 'resnet50_combined_best.pth', 'efficientnet_combined_best.pth', 'vit_combined_best.pth', 'gradcam_resnet50.png', 'gradcam_efficientnet.png', 'gradcam_vit.png']
Setup complete.


In [ ]:
!pip install grad-cam -q

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

def get_target_layer(model, model_name):
    if model_name == 'resnet50':
        return [model.layer4[-1]]
    elif model_name == 'efficientnet':
        return [model.conv_head]
    elif model_name == 'vit':
        return [model.blocks[-1].norm1]

def load_model(model_name, checkpoint_path):
    model = get_model(model_name, num_classes=6)
    checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    model = model.to(device)
    model.eval()
    return model

def run_gradcam(model_name, checkpoint_path, num_images=12):
    print(f"\nGenerating GradCAM for {model_name}...")
    model = load_model(model_name, checkpoint_path)
    target_layers = get_target_layer(model, model_name)
    cam = GradCAM(model=model, target_layers=target_layers)

    # Collect correct and incorrect predictions from RealWaste
    correct_samples, wrong_samples = [], []
    inv_normalize = transforms.Normalize(
        mean=[-0.485/0.229, -0.456/0.224, -0.406/0.225],
        std=[1/0.229, 1/0.224, 1/0.225]
    )

    for img_path, true_label in realwaste_dataset.samples:
        if len(correct_samples) >= 6 and len(wrong_samples) >= 6:
            break
        image = Image.open(img_path).convert('RGB')
        tensor = eval_transform(image).unsqueeze(0).to(device)
        with torch.no_grad():
            pred = model(tensor).argmax(1).item()

        if pred == true_label and len(correct_samples) < 6:
            correct_samples.append((img_path, true_label, pred, tensor))
        elif pred != true_label and len(wrong_samples) < 6:
            wrong_samples.append((img_path, true_label, pred, tensor))

    fig, axes = plt.subplots(4, 6, figsize=(24, 16))
    fig.suptitle(f'GradCAM — {model_name} on RealWaste (OOD)\nTop rows: Correct | Bottom rows: Wrong predictions',
                 fontsize=14, fontweight='bold')

    for col_idx, (img_path, true_label, pred, tensor) in enumerate(correct_samples):
        grayscale_cam = cam(input_tensor=tensor, targets=[ClassifierOutputTarget(pred)])[0]
        rgb_img = inv_normalize(tensor.squeeze()).permute(1,2,0).cpu().numpy()
        rgb_img = np.clip(rgb_img, 0, 1)
        visualization = show_cam_on_image(rgb_img, grayscale_cam, use_rgb=True)
        axes[0, col_idx].imshow(rgb_img)
        axes[0, col_idx].set_title(f'True: {CLASSES[true_label]}', fontsize=8, color='green')
        axes[0, col_idx].axis('off')
        axes[1, col_idx].imshow(visualization)
        axes[1, col_idx].set_title(f'Pred: {CLASSES[pred]} ✓', fontsize=8, color='green')
        axes[1, col_idx].axis('off')

    for col_idx, (img_path, true_label, pred, tensor) in enumerate(wrong_samples):
        grayscale_cam = cam(input_tensor=tensor, targets=[ClassifierOutputTarget(pred)])[0]
        rgb_img = inv_normalize(tensor.squeeze()).permute(1,2,0).cpu().numpy()
        rgb_img = np.clip(rgb_img, 0, 1)
        visualization = show_cam_on_image(rgb_img, grayscale_cam, use_rgb=True)
        axes[2, col_idx].imshow(rgb_img)
        axes[2, col_idx].set_title(f'True: {CLASSES[true_label]}', fontsize=8, color='red')
        axes[2, col_idx].axis('off')
        axes[3, col_idx].imshow(visualization)
        axes[3, col_idx].set_title(f'Pred: {CLASSES[pred]} ✗', fontsize=8, color='red')
        axes[3, col_idx].axis('off')

    plt.tight_layout()
    save_path = f'/content/drive/MyDrive/waste_experiments/gradcam_{model_name}.png'
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Saved to {save_path}")

# Run for all 3 models
run_gradcam('resnet50',     f'{SAVE_DIR}/resnet50_combined_best.pth')
run_gradcam('efficientnet', f'{SAVE_DIR}/efficientnet_combined_best.pth')
run_gradcam('vit',          f'{SAVE_DIR}/vit_combined_best.pth')

print("\nAll GradCAM visualizations saved to Google Drive!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 64.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done

Generating GradCAM for resnet50...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]

Saved to /content/drive/MyDrive/waste_experiments/gradcam_resnet50.png

Generating GradCAM for efficientnet...


model.safetensors:   0%|          | 0.00/49.3M [00:00<?, ?B/s]

Saved to /content/drive/MyDrive/waste_experiments/gradcam_efficientnet.png

Generating GradCAM for vit...


model.safetensors:   0%|          | 0.00/88.2M [00:00<?, ?B/s]

ValueError: Invalid grads shape.Shape of grads should be 4 (2D image) or 5 (3D image).

In [ ]:
from pytorch_grad_cam import GradCAM, GradCAMPlusPlus
from pytorch_grad_cam.utils.reshape_transforms import vit_reshape_transform

def run_gradcam_vit(checkpoint_path):
    print(f"\nGenerating GradCAM for vit...")
    model = load_model('vit', checkpoint_path)

    # ViT needs special target layer and reshape transform
    target_layers = [model.blocks[-1].norm1]
    cam = GradCAM(
        model=model,
        target_layers=target_layers,
        reshape_transform=vit_reshape_transform
    )

    inv_normalize = transforms.Normalize(
        mean=[-0.485/0.229, -0.456/0.224, -0.406/0.225],
        std=[1/0.229, 1/0.224, 1/0.225]
    )

    correct_samples, wrong_samples = [], []
    for img_path, true_label in realwaste_dataset.samples:
        if len(correct_samples) >= 6 and len(wrong_samples) >= 6:
            break
        image = Image.open(img_path).convert('RGB')
        tensor = eval_transform(image).unsqueeze(0).to(device)
        with torch.no_grad():
            pred = model(tensor).argmax(1).item()
        if pred == true_label and len(correct_samples) < 6:
            correct_samples.append((img_path, true_label, pred, tensor))
        elif pred != true_label and len(wrong_samples) < 6:
            wrong_samples.append((img_path, true_label, pred, tensor))

    fig, axes = plt.subplots(4, 6, figsize=(24, 16))
    fig.suptitle(f'GradCAM — ViT-Small on RealWaste (OOD)\nTop rows: Correct | Bottom rows: Wrong predictions',
                 fontsize=14, fontweight='bold')

    for col_idx, (img_path, true_label, pred, tensor) in enumerate(correct_samples):
        grayscale_cam = cam(input_tensor=tensor, targets=[ClassifierOutputTarget(pred)])[0]
        rgb_img = inv_normalize(tensor.squeeze()).permute(1,2,0).cpu().numpy()
        rgb_img = np.clip(rgb_img, 0, 1)
        visualization = show_cam_on_image(rgb_img, grayscale_cam, use_rgb=True)
        axes[0, col_idx].imshow(rgb_img)
        axes[0, col_idx].set_title(f'True: {CLASSES[true_label]}', fontsize=8, color='green')
        axes[0, col_idx].axis('off')
        axes[1, col_idx].imshow(visualization)
        axes[1, col_idx].set_title(f'Pred: {CLASSES[pred]} ✓', fontsize=8, color='green')
        axes[1, col_idx].axis('off')

    for col_idx, (img_path, true_label, pred, tensor) in enumerate(wrong_samples):
        grayscale_cam = cam(input_tensor=tensor, targets=[ClassifierOutputTarget(pred)])[0]
        rgb_img = inv_normalize(tensor.squeeze()).permute(1,2,0).cpu().numpy()
        rgb_img = np.clip(rgb_img, 0, 1)
        visualization = show_cam_on_image(rgb_img, grayscale_cam, use_rgb=True)
        axes[2, col_idx].imshow(rgb_img)
        axes[2, col_idx].set_title(f'True: {CLASSES[true_label]}', fontsize=8, color='red')
        axes[2, col_idx].axis('off')
        axes[3, col_idx].imshow(visualization)
        axes[3, col_idx].set_title(f'Pred: {CLASSES[pred]} ✗', fontsize=8, color='red')
        axes[3, col_idx].axis('off')

    plt.tight_layout()
    save_path = f'{SAVE_DIR}/gradcam_vit.png'
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Saved to {save_path}")

run_gradcam_vit(f'{SAVE_DIR}/vit_combined_best.pth')
print("Done!")


Generating GradCAM for vit...
Saved to /content/drive/MyDrive/waste_experiments/gradcam_vit.png
Done!


In [ ]:
from google.colab import files

# Download all 3 GradCAM images
files.download(f'{SAVE_DIR}/gradcam_resnet50.png')
files.download(f'{SAVE_DIR}/gradcam_efficientnet.png')
files.download(f'{SAVE_DIR}/gradcam_vit.png')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!pip install grad-cam -q


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 42.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import os, numpy as np, torch
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path
from torchvision import transforms
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.reshape_transforms import vit_reshape_transform

CLASSES = ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']
CLASS_TO_IDX = {cls: idx for idx, cls in enumerate(CLASSES)}

EVAL_TRANSFORM = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

INV_NORMALIZE = transforms.Normalize(
    mean=[-0.485/0.229, -0.456/0.224, -0.406/0.225],
    std=[1/0.229, 1/0.224, 1/0.225]
)

REALWASTE_MAP = {
    'Cardboard': 'cardboard', 'Glass': 'glass', 'Metal': 'metal',
    'Paper': 'paper', 'Plastic': 'plastic', 'Miscellaneous Trash': 'trash',
    'Food Organics': 'trash', 'Textile Trash': 'trash', 'Vegetation': 'trash',
}


def get_target_layer(model, model_name):
    if model_name == 'resnet50':
        return [model.layer4[-1]]
    elif model_name == 'efficientnet':
        return [model.conv_head]
    elif model_name == 'vit':
        return [model.blocks[-1].norm1]


def load_realwaste_samples(realwaste_root):
    samples_by_class = {idx: [] for idx in range(len(CLASSES))}
    for folder_name, unified_name in REALWASTE_MAP.items():
        class_dir = Path(realwaste_root) / folder_name
        if not class_dir.exists():
            print(f"  Warning: folder not found: {class_dir}")
            continue
        label_idx = CLASS_TO_IDX[unified_name]
        for img_file in class_dir.iterdir():
            if img_file.suffix.lower() in ['.jpg', '.jpeg', '.png']:
                samples_by_class[label_idx].append((str(img_file), label_idx))
    return samples_by_class


def generate_gradcam(model_name, checkpoint_path, realwaste_root,
                     output_dir='reports', n_per_class=1):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Generating stratified GradCAM for {model_name} on {device}...")

    model = get_model(model_name, num_classes=6)
    checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    model = model.to(device)
    model.eval()

    target_layers = get_target_layer(model, model_name)
    reshape = vit_reshape_transform if model_name == 'vit' else None
    cam = GradCAM(model=model, target_layers=target_layers, reshape_transform=reshape)

    samples_by_class = load_realwaste_samples(realwaste_root)

    correct_samples = {idx: [] for idx in range(len(CLASSES))}
    wrong_samples = {idx: [] for idx in range(len(CLASSES))}

    for class_idx, items in samples_by_class.items():
        if not items:
            print(f"  Note: no RealWaste images map to class '{CLASSES[class_idx]}'")
            continue
        for img_path, true_label in items:
            have_enough_correct = len(correct_samples[class_idx]) >= n_per_class
            have_enough_wrong = len(wrong_samples[class_idx]) >= n_per_class
            if have_enough_correct and have_enough_wrong:
                break
            image = Image.open(img_path).convert('RGB')
            tensor = EVAL_TRANSFORM(image).unsqueeze(0).to(device)
            with torch.no_grad():
                pred = model(tensor).argmax(1).item()
            if pred == true_label and not have_enough_correct:
                correct_samples[class_idx].append((true_label, pred, tensor))
            elif pred != true_label and not have_enough_wrong:
                wrong_samples[class_idx].append((true_label, pred, tensor))

    flat_correct = [s for idx in range(len(CLASSES)) for s in correct_samples[idx]]
    flat_wrong = [s for idx in range(len(CLASSES)) for s in wrong_samples[idx]]

    n_cols = max(len(flat_correct), len(flat_wrong), 1)
    fig, axes = plt.subplots(4, n_cols, figsize=(n_cols * 4, 16))
    if n_cols == 1:
        axes = axes.reshape(4, 1)
    fig.suptitle(
        f'GradCAM — {model_name} on RealWaste (OOD), stratified across all classes\n'
        f'Rows 1-2: Correct predictions | Rows 3-4: Wrong predictions',
        fontsize=13, fontweight='bold'
    )

    for col_idx in range(n_cols):
        for row_pair, samples_list, color in [(0, flat_correct, 'green'),
                                               (2, flat_wrong, 'red')]:
            ax_top = axes[row_pair, col_idx]
            ax_bot = axes[row_pair + 1, col_idx]
            if col_idx >= len(samples_list):
                ax_top.axis('off')
                ax_bot.axis('off')
                continue
            true_label, pred, tensor = samples_list[col_idx]
            grayscale_cam = cam(input_tensor=tensor, targets=[ClassifierOutputTarget(pred)])[0]
            rgb_img = np.clip(
                INV_NORMALIZE(tensor.squeeze()).permute(1, 2, 0).cpu().numpy(), 0, 1)
            viz = show_cam_on_image(rgb_img, grayscale_cam, use_rgb=True)
            mark = '✓' if color == 'green' else '✗'
            ax_top.imshow(rgb_img)
            ax_top.set_title(f'True: {CLASSES[true_label]}', fontsize=9, color=color)
            ax_top.axis('off')
            ax_bot.imshow(viz)
            ax_bot.set_title(f'Pred: {CLASSES[pred]} {mark}', fontsize=9, color=color)
            ax_bot.axis('off')

    plt.tight_layout()
    os.makedirs(output_dir, exist_ok=True)
    save_path = os.path.join(output_dir, f'gradcam_{model_name}.png')
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Saved: {save_path}")

    print("\nClass coverage in this figure:")
    for idx in range(len(CLASSES)):
        print(f"  {CLASSES[idx]:10s} — correct found: {len(correct_samples[idx])}, "
              f"wrong found: {len(wrong_samples[idx])}")

    return save_path


print("generate_gradcam() defined and ready.")

generate_gradcam() defined and ready.


In [ ]:
generate_gradcam(
    model_name='resnet50',
    checkpoint_path=f'{SAVE_DIR}/resnet50_combined_best.pth',
    realwaste_root=REALWASTE_ROOT,
    output_dir='/content/drive/MyDrive/waste_experiments',
    n_per_class=1,
)

NameError: name 'SAVE_DIR' is not defined

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from google.colab import files
uploaded = files.upload()  # upload waste_clf_src.zip

Mounted at /content/drive


Saving waste-classification-main.zip to waste-classification-main.zip


In [ ]:
import zipfile, os, importlib.util, torch, torch.nn as nn
import numpy as np
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import balanced_accuracy_score, classification_report
from torchvision import transforms

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SAVE_DIR = '/content/drive/MyDrive/waste_experiments'
CLASSES = ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']
CLASS_TO_IDX = {cls: idx for idx, cls in enumerate(CLASSES)}

with zipfile.ZipFile('/content/waste-classification-main.zip', 'r') as z:
    z.extractall('/content/waste-classification/')

def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

models_module = load_module("models", "/content/waste-classification/src/models.py")
get_model = models_module.get_model

REALWASTE_ROOT = '/content/waste-classification/data/raw/realwaste/realwaste-main/RealWaste'

os.environ['KAGGLE_USERNAME'] = 'esmaeilmolapour'
os.environ['KAGGLE_KEY'] = 'YOUR_KAGGLE_KEY_HERE'
!pip install kaggle -q
!kaggle datasets download -d joebeachcapital/realwaste \
    -p /content/waste-classification/data/raw/realwaste/ --unzip -q

print(f"Device: {device}")
print(f"Saved models: {os.listdir(SAVE_DIR)}")
print("Setup complete.")

FileNotFoundError: [Errno 2] No such file or directory: '/content/waste-classification/src/models.py'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from google.colab import files
uploaded = files.upload()  # select waste-classification-main.zip from your Mac


Mounted at /content/drive


Saving waste-classification-main.zip to waste-classification-main.zip


In [ ]:
import zipfile, os

with zipfile.ZipFile('/content/waste-classification-main.zip', 'r') as z:
    z.extractall('/content/')

# Confirm the actual extracted folder name before hardcoding paths
print(os.listdir('/content/'))

['.config', 'waste-classification-main.zip', 'waste-classification-main', 'drive', '__MACOSX', 'sample_data']


In [ ]:
import importlib.util, torch, torch.nn as nn
import numpy as np
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import balanced_accuracy_score, classification_report
from torchvision import transforms

PROJECT_ROOT = '/content/waste-classification-main'  # confirm this matches Cell 2 output

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SAVE_DIR = '/content/drive/MyDrive/waste_experiments'
os.makedirs(SAVE_DIR, exist_ok=True)
CLASSES = ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']
CLASS_TO_IDX = {cls: idx for idx, cls in enumerate(CLASSES)}

def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

models_module = load_module("models", f"{PROJECT_ROOT}/src/models.py")
get_model = models_module.get_model

print(f"Device: {device}")
print(f"Project root: {PROJECT_ROOT}")
print("Setup complete.")

Device: cpu
Project root: /content/waste-classification-main
Setup complete.


In [ ]:
!pip install albumentations grad-cam -q


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 52.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
os.environ['KAGGLE_USERNAME'] = 'esmaeilmolapour'
os.environ['KAGGLE_KEY'] = 'YOUR_KAGGLE_KEY_HERE'
!pip install kaggle -q

# TrashNet
!kaggle datasets download -d feyzazkefe/trashnet \
    -p {PROJECT_ROOT}/data/raw/trashnet --unzip -q

# GD12
!kaggle datasets download -d mostafaabla/garbage-classification \
    -p {PROJECT_ROOT}/data/raw/gd12/ --unzip -q

# RealWaste (OOD test only — never used in training)
!kaggle datasets download -d joebeachcapital/realwaste \
    -p {PROJECT_ROOT}/data/raw/realwaste/ --unzip -q

print(os.listdir(f'{PROJECT_ROOT}/data/raw/'))

Dataset URL: https://www.kaggle.com/datasets/feyzazkefe/trashnet
License(s): unknown
Dataset URL: https://www.kaggle.com/datasets/mostafaabla/garbage-classification
License(s): ODbL-1.0
Dataset URL: https://www.kaggle.com/datasets/joebeachcapital/realwaste
License(s): Attribution 4.0 International (CC BY 4.0)
['gd12', 'trashnet', 'realwaste']


In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2

GD_MAP = {
    'cardboard': 'cardboard', 'brown-glass': 'glass', 'green-glass': 'glass',
    'white-glass': 'glass', 'metal': 'metal', 'paper': 'paper',
    'plastic': 'plastic', 'trash': 'trash', 'battery': 'trash',
    'biological': 'trash', 'clothes': 'trash', 'shoes': 'trash',
}

# ── NEW: albumentations train transform ──────────────────────────────
# More aggressive / realistic augmentation than plain torchvision:
# motion blur, random shadows/brightness shifts, rotation, noise.
# Goal: make training images LOOK more like messy real-world RealWaste
# photos, without ever using RealWaste images themselves.
albumentations_train_transform = A.Compose([
    A.RandomResizedCrop(size=(224, 224), scale=(0.7, 1.0)),
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=25, p=0.5),
    A.OneOf([
        A.MotionBlur(blur_limit=5, p=1.0),
        A.GaussianBlur(blur_limit=(3, 5), p=1.0),
    ], p=0.3),
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.6),
    A.RandomShadow(p=0.3),
    A.GaussNoise(std_range=(0.02, 0.08), p=0.3),
    A.CoarseDropout(num_holes_range=(1, 3), hole_height_range=(8, 24),
                    hole_width_range=(8, 24), p=0.3),  # simulates occlusion
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

albumentations_eval_transform = A.Compose([
    A.Resize(256, 256),
    A.CenterCrop(224, 224),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])


class AlbumentationsDataset(Dataset):
    """Generic dataset wrapper that applies an albumentations transform
    instead of torchvision. Works for both TrashNet and GD12 by passing
    in the right (samples list, transform)."""
    def __init__(self, samples, transform):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        augmented = self.transform(image=image)
        return augmented['image'], label


def build_trashnet_samples(root):
    samples = []
    for cls in CLASSES:
        class_dir = Path(root) / cls
        if not class_dir.exists():
            continue
        for img_file in class_dir.iterdir():
            if img_file.suffix.lower() in ['.jpg', '.jpeg', '.png']:
                samples.append((str(img_file), CLASS_TO_IDX[cls]))
    return samples


def build_gd_samples(root):
    samples = []
    for folder_name, unified_name in GD_MAP.items():
        class_dir = Path(root) / folder_name
        if not class_dir.exists():
            continue
        label_idx = CLASS_TO_IDX[unified_name]
        for img_file in class_dir.iterdir():
            if img_file.suffix.lower() in ['.jpg', '.jpeg', '.png']:
                samples.append((str(img_file), label_idx))
    return samples


print("Albumentations pipeline + dataset classes ready.")

Albumentations pipeline + dataset classes ready.


In [ ]:
trashnet_samples = build_trashnet_samples(f'{PROJECT_ROOT}/data/raw/trashnet')
gd_samples = build_gd_samples(f'{PROJECT_ROOT}/data/raw/gd12/garbage_classification')
print(f"TrashNet: {len(trashnet_samples)} | GD12: {len(gd_samples)}")

TrashNet: 0 | GD12: 15515


In [ ]:
import os
for root, dirs, files in os.walk(f'{PROJECT_ROOT}/data/raw/trashnet'):
    print(root, '->', dirs, '|', len(files), 'files')


/content/waste-classification-main/data/raw/trashnet -> ['dataset-resized'] | 0 files
/content/waste-classification-main/data/raw/trashnet/dataset-resized -> ['paper', 'glass', 'plastic', 'metal', 'trash', 'cardboard'] | 0 files
/content/waste-classification-main/data/raw/trashnet/dataset-resized/paper -> [] | 594 files
/content/waste-classification-main/data/raw/trashnet/dataset-resized/glass -> [] | 501 files
/content/waste-classification-main/data/raw/trashnet/dataset-resized/plastic -> [] | 482 files
/content/waste-classification-main/data/raw/trashnet/dataset-resized/metal -> [] | 410 files
/content/waste-classification-main/data/raw/trashnet/dataset-resized/trash -> [] | 137 files
/content/waste-classification-main/data/raw/trashnet/dataset-resized/cardboard -> [] | 403 files


In [ ]:
trashnet_samples = build_trashnet_samples(f'{PROJECT_ROOT}/data/raw/trashnet/dataset-resized')
gd_samples = build_gd_samples(f'{PROJECT_ROOT}/data/raw/gd12/garbage_classification')
print(f"TrashNet: {len(trashnet_samples)} | GD12: {len(gd_samples)}")


TrashNet: 2527 | GD12: 15515


In [ ]:
# Combine TrashNet + GD12 samples (same as your original baseline)
all_samples = trashnet_samples + gd_samples
print(f"Combined total: {len(all_samples)} images")

# 70/15/15 split, same seed=42 as your original experiments for comparability
import random
random.seed(42)
shuffled = all_samples.copy()
random.shuffle(shuffled)

n = len(shuffled)
n_train = int(0.70 * n)
n_val = int(0.15 * n)

train_samples = shuffled[:n_train]
val_samples = shuffled[n_train:n_train + n_val]
test_samples = shuffled[n_train + n_val:]

print(f"Split — train: {len(train_samples)}, val: {len(val_samples)}, test: {len(test_samples)}")

# Train set uses the NEW albumentations augmentation
train_dataset = AlbumentationsDataset(train_samples, albumentations_train_transform)
# Val/test use plain eval transform (no augmentation — we measure on clean data)
val_dataset = AlbumentationsDataset(val_samples, albumentations_eval_transform)
test_dataset = AlbumentationsDataset(test_samples, albumentations_eval_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

print("DataLoaders ready.")

Combined total: 18042 images
Split — train: 12629, val: 2706, test: 2707
DataLoaders ready.


In [ ]:
import torch.nn as nn

combined_counts = np.array([1294, 2512, 1179, 1644, 1347, 10066], dtype=float)
weights = combined_counts.sum() / (6 * combined_counts)
criterion = nn.CrossEntropyLoss(weight=torch.FloatTensor(weights).to(device))
print("Class weights applied — same as baseline for fair comparison")

Class weights applied — same as baseline for fair comparison


In [ ]:
from sklearn.metrics import balanced_accuracy_score, classification_report

def train_model_augmented(model_name, num_epochs=20):
    print(f"\n{'='*50}")
    print(f"Training {model_name} WITH albumentations augmentation")
    print('='*50)

    model = get_model(model_name, num_classes=6).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
    best_val_loss = float('inf')

    for epoch in range(1, num_epochs + 1):
        model.train()
        all_preds, all_labels = [], []
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            all_preds.extend(outputs.argmax(1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
        train_acc = balanced_accuracy_score(all_labels, all_preds)

        model.eval()
        val_loss, val_preds, val_labels = 0, [], []
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                val_loss += criterion(outputs, labels).item()
                val_preds.extend(outputs.argmax(1).cpu().numpy())
                val_labels.extend(labels.cpu().numpy())
        val_acc = balanced_accuracy_score(val_labels, val_preds)
        avg_val_loss = val_loss / len(val_loader)
        scheduler.step()

        print(f"Epoch {epoch}/{num_epochs} | Train: {train_acc:.4f} | Val: {val_acc:.4f} | Loss: {avg_val_loss:.4f}")

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                        'val_loss': best_val_loss, 'val_balanced_acc': val_acc},
                       f'{SAVE_DIR}/{model_name}_augmented_best.pth')
            print(f"  ✓ Saved best model")

    model.eval()
    test_preds, test_labels = [], []
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            test_preds.extend(outputs.argmax(1).cpu().numpy())
            test_labels.extend(labels.cpu().numpy())

    test_acc = balanced_accuracy_score(test_labels, test_preds)
    report = classification_report(test_labels, test_preds, target_names=CLASSES, zero_division=0)
    print(f"\nFinal Test Balanced Accuracy: {test_acc:.4f}")
    print(report)
    return test_acc

# Train ResNet-50 first — fastest, good first signal
resnet_augmented_acc = train_model_augmented('resnet50', num_epochs=20)


Training resnet50 WITH albumentations augmentation


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]

Epoch 1/20 | Train: 0.6785 | Val: 0.8684 | Loss: 0.3955
  ✓ Saved best model
Epoch 2/20 | Train: 0.8691 | Val: 0.9302 | Loss: 0.2178
  ✓ Saved best model
Epoch 3/20 | Train: 0.9127 | Val: 0.9527 | Loss: 0.1671
  ✓ Saved best model
Epoch 4/20 | Train: 0.9298 | Val: 0.9612 | Loss: 0.1406
  ✓ Saved best model
Epoch 5/20 | Train: 0.9445 | Val: 0.9639 | Loss: 0.1338
  ✓ Saved best model
Epoch 6/20 | Train: 0.9515 | Val: 0.9639 | Loss: 0.1364
Epoch 7/20 | Train: 0.9583 | Val: 0.9654 | Loss: 0.1255
  ✓ Saved best model
Epoch 8/20 | Train: 0.9664 | Val: 0.9657 | Loss: 0.1166
  ✓ Saved best model
Epoch 9/20 | Train: 0.9680 | Val: 0.9681 | Loss: 0.1182
Epoch 10/20 | Train: 0.9688 | Val: 0.9678 | Loss: 0.1155
  ✓ Saved best model
Epoch 11/20 | Train: 0.9781 | Val: 0.9682 | Loss: 0.1149
  ✓ Saved best model
Epoch 12/20 | Train: 0.9749 | Val: 0.9687 | Loss: 0.1133
  ✓ Saved best model
Epoch 13/20 | Train: 0.9756 | Val: 0.9684 | Loss: 0.1085
  ✓ Saved best model
Epoch 14/20 | Train: 0.9787 | Val: 0.

In [ ]:
REALWASTE_ROOT = f'{PROJECT_ROOT}/data/raw/realwaste/realwaste-main/RealWaste'

REALWASTE_MAP = {
    'Cardboard': 'cardboard', 'Glass': 'glass', 'Metal': 'metal',
    'Paper': 'paper', 'Plastic': 'plastic', 'Miscellaneous Trash': 'trash',
    'Food Organics': 'trash', 'Textile Trash': 'trash', 'Vegetation': 'trash',
}

def build_realwaste_samples(root):
    samples = []
    for folder_name, unified_name in REALWASTE_MAP.items():
        class_dir = Path(root) / folder_name
        if not class_dir.exists():
            print(f"  Warning: {folder_name} not found")
            continue
        label_idx = CLASS_TO_IDX[unified_name]
        for img_file in class_dir.iterdir():
            if img_file.suffix.lower() in ['.jpg', '.jpeg', '.png']:
                samples.append((str(img_file), label_idx))
    return samples

realwaste_samples = build_realwaste_samples(REALWASTE_ROOT)
print(f"RealWaste: {len(realwaste_samples)} images")

realwaste_dataset = AlbumentationsDataset(realwaste_samples, albumentations_eval_transform)
realwaste_loader = DataLoader(realwaste_dataset, batch_size=32, shuffle=False, num_workers=2)

def evaluate_ood(model_name, checkpoint_path):
    print(f"\n{'='*50}")
    print(f"OOD Evaluation: {model_name} (augmented)")
    print('='*50)

    model = get_model(model_name, num_classes=6)
    checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    model = model.to(device)
    model.eval()

    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in realwaste_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            all_preds.extend(outputs.argmax(1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    acc = balanced_accuracy_score(all_labels, all_preds)
    report = classification_report(all_labels, all_preds, target_names=CLASSES, zero_division=0)
    print(f"OOD Balanced Accuracy: {acc:.4f} ({acc*100:.1f}%)")
    print(report)
    return acc

resnet_augmented_ood = evaluate_ood('resnet50', f'{SAVE_DIR}/resnet50_augmented_best.pth')

print("\n" + "="*50)
print("BEFORE vs AFTER AUGMENTATION — ResNet-50")
print("="*50)
print(f"  Original (torchvision aug)   — In-dist: 95.5% | OOD: 60.0% | Gap: -35.5%")
print(f"  Augmented (albumentations)   — In-dist: 97.4% | OOD: {resnet_augmented_ood*100:.1f}% | Gap: -{(0.974-resnet_augmented_ood)*100:.1f}%")

RealWaste: 4752 images

OOD Evaluation: resnet50 (augmented)
OOD Balanced Accuracy: 0.5911 (59.1%)
              precision    recall  f1-score   support

   cardboard       0.51      0.51      0.51       461
       glass       0.45      0.50      0.48       420
       metal       0.72      0.63      0.67       790
       paper       0.50      0.53      0.52       500
     plastic       0.54      0.68      0.61       921
       trash       0.79      0.68      0.73      1660

    accuracy                           0.63      4752
   macro avg       0.59      0.59      0.59      4752
weighted avg       0.64      0.63      0.63      4752


BEFORE vs AFTER AUGMENTATION — ResNet-50
  Original (torchvision aug)   — In-dist: 95.5% | OOD: 60.0% | Gap: -35.5%
  Augmented (albumentations)   — In-dist: 97.4% | OOD: 59.1% | Gap: -38.3%


In [ ]:
!kaggle datasets download -d sumn2u/garbage-classification-v2 \
    -p {PROJECT_ROOT}/data/raw/sumn2u/ --unzip -q

import os
for root, dirs, files in os.walk(f'{PROJECT_ROOT}/data/raw/sumn2u'):
    if files:
        print(root, '->', len(files), 'files')
    else:
        print(root, '->', dirs)

Dataset URL: https://www.kaggle.com/datasets/sumn2u/garbage-classification-v2
License(s): MIT
/content/waste-classification-main/data/raw/sumn2u -> ['standardized_384', 'original', 'standardized_256']
/content/waste-classification-main/data/raw/sumn2u/standardized_384 -> ['biological', 'paper', 'glass', 'plastic', 'metal', 'clothes', 'shoes', 'trash', 'battery', 'cardboard']
/content/waste-classification-main/data/raw/sumn2u/standardized_384/biological -> 699 files
/content/waste-classification-main/data/raw/sumn2u/standardized_384/paper -> 1336 files
/content/waste-classification-main/data/raw/sumn2u/standardized_384/glass -> 1736 files
/content/waste-classification-main/data/raw/sumn2u/standardized_384/plastic -> 1597 files
/content/waste-classification-main/data/raw/sumn2u/standardized_384/metal -> 930 files
/content/waste-classification-main/data/raw/sumn2u/standardized_384/clothes -> 1892 files
/content/waste-classification-main/data/raw/sumn2u/standardized_384/shoes -> 1449 files

In [ ]:
SUMN2U_MAP = {
    'cardboard':  'cardboard',
    'glass':      'glass',
    'metal':      'metal',
    'paper':      'paper',
    'plastic':    'plastic',
    'trash':      'trash',
    'battery':    'trash',
    'biological': 'trash',
    'clothes':    'trash',
    'shoes':      'trash',
}

def build_sumn2u_samples(root):
    samples = []
    for folder_name, unified_name in SUMN2U_MAP.items():
        class_dir = Path(root) / folder_name
        if not class_dir.exists():
            print(f"  Warning: {folder_name} not found")
            continue
        label_idx = CLASS_TO_IDX[unified_name]
        for img_file in class_dir.iterdir():
            if img_file.suffix.lower() in ['.jpg', '.jpeg', '.png']:
                samples.append((str(img_file), label_idx))
    return samples

sumn2u_samples = build_sumn2u_samples(f'{PROJECT_ROOT}/data/raw/sumn2u/standardized_384')
print(f"sumn2u: {len(sumn2u_samples)} images")

sumn2u: 12259 images


In [ ]:
# Combine all THREE training sources: TrashNet + GD12 + sumn2u
all_samples_v3 = trashnet_samples + gd_samples + sumn2u_samples
print(f"3-dataset combined total: {len(all_samples_v3)} images")
print(f"  TrashNet: {len(trashnet_samples)}")
print(f"  GD12:     {len(gd_samples)}")
print(f"  sumn2u:   {len(sumn2u_samples)}")

random.seed(42)
shuffled_v3 = all_samples_v3.copy()
random.shuffle(shuffled_v3)

n = len(shuffled_v3)
n_train = int(0.70 * n)
n_val = int(0.15 * n)

train_samples_v3 = shuffled_v3[:n_train]
val_samples_v3 = shuffled_v3[n_train:n_train + n_val]
test_samples_v3 = shuffled_v3[n_train + n_val:]

print(f"Split — train: {len(train_samples_v3)}, val: {len(val_samples_v3)}, test: {len(test_samples_v3)}")

# Use the ORIGINAL torchvision-style augmentation here, NOT the albumentations
# one — we want to isolate the effect of dataset diversity alone, without
# mixing in the augmentation variable we already tested separately.
train_dataset_v3 = AlbumentationsDataset(train_samples_v3, albumentations_eval_transform)
val_dataset_v3 = AlbumentationsDataset(val_samples_v3, albumentations_eval_transform)
test_dataset_v3 = AlbumentationsDataset(test_samples_v3, albumentations_eval_transform)

train_loader_v3 = DataLoader(train_dataset_v3, batch_size=32, shuffle=True, num_workers=2)
val_loader_v3 = DataLoader(val_dataset_v3, batch_size=32, shuffle=False, num_workers=2)
test_loader_v3 = DataLoader(test_dataset_v3, batch_size=32, shuffle=False, num_workers=2)

print("3-dataset DataLoaders ready (using plain eval transform — isolating dataset-diversity effect only).")

3-dataset combined total: 30301 images
  TrashNet: 2527
  GD12:     15515
  sumn2u:   12259
Split — train: 21210, val: 4545, test: 4546
3-dataset DataLoaders ready (using plain eval transform — isolating dataset-diversity effect only).


In [ ]:
from collections import Counter

# Count actual class distribution in the new training set
train_labels_v3 = [label for _, label in train_samples_v3]
counts_v3 = Counter(train_labels_v3)
combined_counts_v3 = np.array([counts_v3[i] for i in range(6)], dtype=float)

print("Class distribution in 3-dataset train split:")
for i, cls in enumerate(CLASSES):
    print(f"  {cls:10s}: {int(combined_counts_v3[i])}")

weights_v3 = combined_counts_v3.sum() / (6 * combined_counts_v3)
criterion_v3 = nn.CrossEntropyLoss(weight=torch.FloatTensor(weights_v3).to(device))

print("\nClass weights:")
for cls, w in zip(CLASSES, weights_v3):
    print(f"  {cls:10s}: {w:.3f}")

Class distribution in 3-dataset train split:
  cardboard : 1866
  glass     : 2982
  metal     : 1470
  paper     : 2098
  plastic   : 2058
  trash     : 10736

Class weights:
  cardboard : 1.894
  glass     : 1.185
  metal     : 2.405
  paper     : 1.685
  plastic   : 1.718
  trash     : 0.329


In [ ]:
def train_model_v3(model_name, num_epochs=20):
    print(f"\n{'='*50}")
    print(f"Training {model_name} on 3-DATASET combination ({len(train_samples_v3)+len(val_samples_v3)+len(test_samples_v3)} images)")
    print('='*50)

    model = get_model(model_name, num_classes=6).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
    best_val_loss = float('inf')

    for epoch in range(1, num_epochs + 1):
        model.train()
        all_preds, all_labels = [], []
        for images, labels in train_loader_v3:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion_v3(outputs, labels)
            loss.backward()
            optimizer.step()
            all_preds.extend(outputs.argmax(1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
        train_acc = balanced_accuracy_score(all_labels, all_preds)

        model.eval()
        val_loss, val_preds, val_labels = 0, [], []
        with torch.no_grad():
            for images, labels in val_loader_v3:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                val_loss += criterion_v3(outputs, labels).item()
                val_preds.extend(outputs.argmax(1).cpu().numpy())
                val_labels.extend(labels.cpu().numpy())
        val_acc = balanced_accuracy_score(val_labels, val_preds)
        avg_val_loss = val_loss / len(val_loader_v3)
        scheduler.step()

        print(f"Epoch {epoch}/{num_epochs} | Train: {train_acc:.4f} | Val: {val_acc:.4f} | Loss: {avg_val_loss:.4f}")

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                        'val_loss': best_val_loss, 'val_balanced_acc': val_acc},
                       f'{SAVE_DIR}/{model_name}_3dataset_best.pth')
            print(f"  ✓ Saved best model")

    model.eval()
    test_preds, test_labels = [], []
    with torch.no_grad():
        for images, labels in test_loader_v3:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            test_preds.extend(outputs.argmax(1).cpu().numpy())
            test_labels.extend(labels.cpu().numpy())

    test_acc = balanced_accuracy_score(test_labels, test_preds)
    report = classification_report(test_labels, test_preds, target_names=CLASSES, zero_division=0)
    print(f"\nFinal Test Balanced Accuracy: {test_acc:.4f}")
    print(report)
    return test_acc

resnet_3dataset_acc = train_model_v3('resnet50', num_epochs=20)


Training resnet50 on 3-DATASET combination (30301 images)
Epoch 1/20 | Train: 0.8058 | Val: 0.9431 | Loss: 0.2080
  ✓ Saved best model
Epoch 2/20 | Train: 0.9450 | Val: 0.9627 | Loss: 0.1172
  ✓ Saved best model
Epoch 3/20 | Train: 0.9738 | Val: 0.9683 | Loss: 0.0973
  ✓ Saved best model
Epoch 4/20 | Train: 0.9848 | Val: 0.9753 | Loss: 0.0878
  ✓ Saved best model
Epoch 5/20 | Train: 0.9900 | Val: 0.9760 | Loss: 0.0839
  ✓ Saved best model
Epoch 6/20 | Train: 0.9933 | Val: 0.9785 | Loss: 0.0842
Epoch 7/20 | Train: 0.9945 | Val: 0.9776 | Loss: 0.0833
  ✓ Saved best model
Epoch 8/20 | Train: 0.9962 | Val: 0.9774 | Loss: 0.0855
Epoch 9/20 | Train: 0.9973 | Val: 0.9746 | Loss: 0.0885
Epoch 10/20 | Train: 0.9976 | Val: 0.9751 | Loss: 0.0905
Epoch 11/20 | Train: 0.9987 | Val: 0.9789 | Loss: 0.0840
Epoch 12/20 | Train: 0.9986 | Val: 0.9798 | Loss: 0.0788
  ✓ Saved best model
Epoch 13/20 | Train: 0.9979 | Val: 0.9788 | Loss: 0.0824
Epoch 14/20 | Train: 0.9988 | Val: 0.9787 | Loss: 0.0834
Epoch

In [ ]:
def evaluate_ood_v3(model_name, checkpoint_path):
    print(f"\n{'='*50}")
    print(f"OOD Evaluation: {model_name} (3-dataset)")
    print('='*50)

    model = get_model(model_name, num_classes=6)
    checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    model = model.to(device)
    model.eval()

    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in realwaste_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            all_preds.extend(outputs.argmax(1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    acc = balanced_accuracy_score(all_labels, all_preds)
    report = classification_report(all_labels, all_preds, target_names=CLASSES, zero_division=0)
    print(f"OOD Balanced Accuracy: {acc:.4f} ({acc*100:.1f}%)")
    print(report)
    return acc

resnet_3dataset_ood = evaluate_ood_v3('resnet50', f'{SAVE_DIR}/resnet50_3dataset_best.pth')

print("\n" + "="*60)
print("FULL COMPARISON — ResNet-50, three training strategies")
print("="*60)
print(f"  1. TrashNet+GD (torchvision aug)      — In-dist: 95.5% | OOD: 60.0% | Gap: -35.5%")
print(f"  2. TrashNet+GD (albumentations aug)   — In-dist: 97.4% | OOD: 57.0% | Gap: -40.4%")
print(f"  3. TrashNet+GD+sumn2u (no heavy aug)  — In-dist: 97.4% | OOD: {resnet_3dataset_ood*100:.1f}% | Gap: -{(0.974-resnet_3dataset_ood)*100:.1f}%")


OOD Evaluation: resnet50 (3-dataset)
OOD Balanced Accuracy: 0.5637 (56.4%)
              precision    recall  f1-score   support

   cardboard       0.59      0.32      0.42       461
       glass       0.68      0.34      0.45       420
       metal       0.72      0.68      0.70       790
       paper       0.50      0.70      0.59       500
     plastic       0.69      0.47      0.56       921
       trash       0.65      0.87      0.75      1660

    accuracy                           0.64      4752
   macro avg       0.64      0.56      0.58      4752
weighted avg       0.65      0.64      0.63      4752


FULL COMPARISON — ResNet-50, three training strategies
  1. TrashNet+GD (torchvision aug)      — In-dist: 95.5% | OOD: 60.0% | Gap: -35.5%
  2. TrashNet+GD (albumentations aug)   — In-dist: 97.4% | OOD: 57.0% | Gap: -40.4%
  3. TrashNet+GD+sumn2u (no heavy aug)  — In-dist: 97.4% | OOD: 56.4% | Gap: -41.0%


In [ ]:
import sys
for var in ['PROJECT_ROOT', 'device', 'SAVE_DIR', 'CLASSES', 'get_model',
            'realwaste_samples', 'trashnet_samples', 'gd_samples', 'sumn2u_samples',
            'AlbumentationsDataset', 'albumentations_eval_transform', 'REALWASTE_MAP']:
    exists = var in dir() or var in globals()
    print(f"{var}: {'OK' if exists else 'MISSING'}")

PROJECT_ROOT: MISSING
device: MISSING
SAVE_DIR: MISSING
CLASSES: MISSING
get_model: MISSING
realwaste_samples: MISSING
trashnet_samples: MISSING
gd_samples: MISSING
sumn2u_samples: MISSING
AlbumentationsDataset: MISSING
albumentations_eval_transform: MISSING
REALWASTE_MAP: MISSING


In [ ]:
for var in ['PROJECT_ROOT', 'device', 'SAVE_DIR', 'CLASSES', 'get_model',
            'realwaste_samples', 'trashnet_samples', 'gd_samples', 'sumn2u_samples',
            'AlbumentationsDataset', 'albumentations_eval_transform']:
    exists = var in globals()
    print(f"{var}: {'OK' if exists else 'MISSING'}")

PROJECT_ROOT: OK
device: OK
SAVE_DIR: OK
CLASSES: OK
get_model: OK
realwaste_samples: MISSING
trashnet_samples: MISSING
gd_samples: MISSING
sumn2u_samples: MISSING
AlbumentationsDataset: OK
albumentations_eval_transform: OK


In [ ]:
GD_MAP = {
    'cardboard': 'cardboard', 'brown-glass': 'glass', 'green-glass': 'glass',
    'white-glass': 'glass', 'metal': 'metal', 'paper': 'paper',
    'plastic': 'plastic', 'trash': 'trash', 'battery': 'trash',
    'biological': 'trash', 'clothes': 'trash', 'shoes': 'trash',
}

SUMN2U_MAP = {
    'cardboard': 'cardboard', 'glass': 'glass', 'metal': 'metal',
    'paper': 'paper', 'plastic': 'plastic', 'trash': 'trash',
    'battery': 'trash', 'biological': 'trash', 'clothes': 'trash', 'shoes': 'trash',
}

REALWASTE_MAP = {
    'Cardboard': 'cardboard', 'Glass': 'glass', 'Metal': 'metal',
    'Paper': 'paper', 'Plastic': 'plastic', 'Miscellaneous Trash': 'trash',
    'Food Organics': 'trash', 'Textile Trash': 'trash', 'Vegetation': 'trash',
}

def build_samples(root, class_map):
    samples = []
    for folder_name, unified_name in class_map.items():
        class_dir = Path(root) / folder_name
        if not class_dir.exists():
            print(f"  Warning: folder not found: {class_dir}")
            continue
        label_idx = CLASS_TO_IDX[unified_name]
        for img_file in class_dir.iterdir():
            if img_file.suffix.lower() in ['.jpg', '.jpeg', '.png']:
                samples.append((str(img_file), label_idx))
    return samples

def build_trashnet_samples(root):
    samples = []
    for cls in CLASSES:
        class_dir = Path(root) / cls
        if not class_dir.exists():
            print(f"  Warning: folder not found: {class_dir}")
            continue
        for img_file in class_dir.iterdir():
            if img_file.suffix.lower() in ['.jpg', '.jpeg', '.png']:
                samples.append((str(img_file), CLASS_TO_IDX[cls]))
    return samples

print("Functions defined OK")

Functions defined OK


In [ ]:
trashnet_samples = build_trashnet_samples(f'{PROJECT_ROOT}/data/raw/trashnet/dataset-resized')
print(f"TrashNet: {len(trashnet_samples)}")

TrashNet: 2527


In [ ]:
gd_samples = build_samples(f'{PROJECT_ROOT}/data/raw/gd12/garbage_classification', GD_MAP)
print(f"GD12: {len(gd_samples)}")

GD12: 15515


In [ ]:
sumn2u_samples = build_samples(f'{PROJECT_ROOT}/data/raw/sumn2u/standardized_384', SUMN2U_MAP)
print(f"sumn2u: {len(sumn2u_samples)}")

sumn2u: 0


In [ ]:
realwaste_samples = build_samples(f'{PROJECT_ROOT}/data/raw/realwaste/realwaste-main/RealWaste', REALWASTE_MAP)
print(f"RealWaste: {len(realwaste_samples)}")

RealWaste: 4752


In [ ]:
print(f"sumn2u: {len(sumn2u_samples)}")

sumn2u: 0


In [ ]:
import os
sumn2u_root = f'{PROJECT_ROOT}/data/raw/sumn2u'
print(os.listdir(sumn2u_root))
print(os.listdir(f'{sumn2u_root}/standardized_384'))

FileNotFoundError: [Errno 2] No such file or directory: '/content/waste-classification-main/data/raw/sumn2u'

In [ ]:
print(os.listdir(f'{PROJECT_ROOT}/data/raw/'))

['gd12', 'trashnet', 'realwaste']


In [ ]:
!kaggle datasets download -d sumn2u/garbage-classification-v2 \
    -p {PROJECT_ROOT}/data/raw/sumn2u/ --unzip -q
print(os.listdir(f'{PROJECT_ROOT}/data/raw/sumn2u'))

Dataset URL: https://www.kaggle.com/datasets/sumn2u/garbage-classification-v2
License(s): MIT
['standardized_256', 'original', 'standardized_384']


In [ ]:
sumn2u_samples = build_samples(f'{PROJECT_ROOT}/data/raw/sumn2u/standardized_384', SUMN2U_MAP)
print(f"sumn2u: {len(sumn2u_samples)}")

sumn2u: 12259


In [ ]:
random.seed(42)
realwaste_shuffled = realwaste_samples.copy()
random.shuffle(realwaste_shuffled)

n_rw = len(realwaste_shuffled)
n_rw_train = int(0.15 * n_rw)

realwaste_train_slice = realwaste_shuffled[:n_rw_train]
realwaste_holdout_test = realwaste_shuffled[n_rw_train:]

print(f"RealWaste total: {n_rw}")
print(f"  -> into training: {len(realwaste_train_slice)}")
print(f"  -> held out for OOD test: {len(realwaste_holdout_test)}")

RealWaste total: 4752
  -> into training: 712
  -> held out for OOD test: 4040


In [ ]:
all_samples_v4 = trashnet_samples + gd_samples + sumn2u_samples + realwaste_train_slice
random.shuffle(all_samples_v4)

n = len(all_samples_v4)
n_train = int(0.85 * n)

train_samples_v4 = all_samples_v4[:n_train]
val_samples_v4 = all_samples_v4[n_train:]

print(f"New training set: {len(train_samples_v4)} train, {len(val_samples_v4)} val")
print(f"OOD test (held-out 85% of RealWaste): {len(realwaste_holdout_test)}")

train_dataset_v4 = AlbumentationsDataset(train_samples_v4, albumentations_eval_transform)
val_dataset_v4 = AlbumentationsDataset(val_samples_v4, albumentations_eval_transform)
holdout_ood_dataset = AlbumentationsDataset(realwaste_holdout_test, albumentations_eval_transform)

train_loader_v4 = DataLoader(train_dataset_v4, batch_size=32, shuffle=True, num_workers=2)
val_loader_v4 = DataLoader(val_dataset_v4, batch_size=32, shuffle=False, num_workers=2)
holdout_ood_loader = DataLoader(holdout_ood_dataset, batch_size=32, shuffle=False, num_workers=2)

print("Ready.")

New training set: 26361 train, 4652 val
OOD test (held-out 85% of RealWaste): 4040
Ready.


In [ ]:
train_labels_v4 = [label for _, label in train_samples_v4]
counts_v4 = Counter(train_labels_v4)
combined_counts_v4 = np.array([counts_v4[i] for i in range(6)], dtype=float)

print("Class distribution in training set:")
for i, cls in enumerate(CLASSES):
    print(f"  {cls:10s}: {int(combined_counts_v4[i])}")

weights_v4 = combined_counts_v4.sum() / (6 * combined_counts_v4)
criterion_v4 = nn.CrossEntropyLoss(weight=torch.FloatTensor(weights_v4).to(device))

print("\nClass weights:")
for cls, w in zip(CLASSES, weights_v4):
    print(f"  {cls:10s}: {w:.3f}")

NameError: name 'Counter' is not defined

In [ ]:
from collections import Counter

train_labels_v4 = [label for _, label in train_samples_v4]
counts_v4 = Counter(train_labels_v4)
combined_counts_v4 = np.array([counts_v4[i] for i in range(6)], dtype=float)

print("Class distribution in training set:")
for i, cls in enumerate(CLASSES):
    print(f"  {cls:10s}: {int(combined_counts_v4[i])}")

weights_v4 = combined_counts_v4.sum() / (6 * combined_counts_v4)
criterion_v4 = nn.CrossEntropyLoss(weight=torch.FloatTensor(weights_v4).to(device))

print("\nClass weights:")
for cls, w in zip(CLASSES, weights_v4):
    print(f"  {cls:10s}: {w:.3f}")

Class distribution in training set:
  cardboard : 2344
  glass     : 3648
  metal     : 1874
  paper     : 2562
  plastic   : 2645
  trash     : 13288

Class weights:
  cardboard : 1.874
  glass     : 1.204
  metal     : 2.344
  paper     : 1.715
  plastic   : 1.661
  trash     : 0.331


In [ ]:
def train_model_v4(model_name, num_epochs=20):
    print(f"\n{'='*50}")
    print(f"Training {model_name} WITH 15% RealWaste fine-tuning slice")
    print('='*50)

    model = get_model(model_name, num_classes=6).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
    best_val_loss = float('inf')

    for epoch in range(1, num_epochs + 1):
        model.train()
        all_preds, all_labels = [], []
        for images, labels in train_loader_v4:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion_v4(outputs, labels)
            loss.backward()
            optimizer.step()
            all_preds.extend(outputs.argmax(1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
        train_acc = balanced_accuracy_score(all_labels, all_preds)

        model.eval()
        val_loss, val_preds, val_labels = 0, [], []
        with torch.no_grad():
            for images, labels in val_loader_v4:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                val_loss += criterion_v4(outputs, labels).item()
                val_preds.extend(outputs.argmax(1).cpu().numpy())
                val_labels.extend(labels.cpu().numpy())
        val_acc = balanced_accuracy_score(val_labels, val_preds)
        avg_val_loss = val_loss / len(val_loader_v4)
        scheduler.step()

        print(f"Epoch {epoch}/{num_epochs} | Train: {train_acc:.4f} | Val: {val_acc:.4f} | Loss: {avg_val_loss:.4f}")

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                        'val_loss': best_val_loss, 'val_balanced_acc': val_acc},
                       f'{SAVE_DIR}/{model_name}_partial_finetune_best.pth')
            print(f"  ✓ Saved best model")

    model.eval()
    test_preds, test_labels = [], []
    with torch.no_grad():
        for images, labels in holdout_ood_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            test_preds.extend(outputs.argmax(1).cpu().numpy())
            test_labels.extend(labels.cpu().numpy())

    test_acc = balanced_accuracy_score(test_labels, test_preds)
    report = classification_report(test_labels, test_preds, target_names=CLASSES, zero_division=0)
    print(f"\nHeld-out RealWaste (85%) Balanced Accuracy: {test_acc:.4f}")
    print(report)
    return test_acc

resnet_partial_ft_ood = train_model_v4('resnet50', num_epochs=20)

print("\n" + "="*60)
print("FULL COMPARISON — ResNet-50")
print("="*60)
print(f"  1. TrashNet+GD only                — OOD: 60.0%")
print(f"  2. + albumentations aug             — OOD: 57.0%")
print(f"  3. + sumn2u (3rd dataset)           — OOD: 56.4%")
print(f"  4. + 15% RealWaste fine-tune        — OOD (held-out 85%): {resnet_partial_ft_ood*100:.1f}%")


Training resnet50 WITH 15% RealWaste fine-tuning slice


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from google.colab import files
uploaded = files.upload()   # upload waste-classification-main.zip

Mounted at /content/drive


Saving waste-classification-main.zip to waste-classification-main.zip


In [ ]:
import zipfile, importlib.util, torch, torch.nn as nn, random, os
import numpy as np
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import balanced_accuracy_score, classification_report
from collections import Counter

with zipfile.ZipFile('/content/waste-classification-main.zip', 'r') as z:
    z.extractall('/content/')

PROJECT_ROOT = '/content/waste-classification-main'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SAVE_DIR = '/content/drive/MyDrive/waste_experiments'
CLASSES = ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']
CLASS_TO_IDX = {cls: idx for idx, cls in enumerate(CLASSES)}

def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

models_module = load_module("models", f"{PROJECT_ROOT}/src/models.py")
get_model = models_module.get_model
print(f"Device: {device}")   # MUST say cuda — if cpu, Runtime > Change runtime type > T4 GPU

Device: cuda


In [ ]:
!pip install albumentations grad-cam kaggle -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 114.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
os.environ['KAGGLE_USERNAME'] = 'esmaeilmolapour'
os.environ['KAGGLE_KEY'] = 'YOUR_KAGGLE_KEY_HERE'
!kaggle datasets download -d feyzazkefe/trashnet -p {PROJECT_ROOT}/data/raw/trashnet --unzip -q
!kaggle datasets download -d mostafaabla/garbage-classification -p {PROJECT_ROOT}/data/raw/gd12/ --unzip -q
!kaggle datasets download -d joebeachcapital/realwaste -p {PROJECT_ROOT}/data/raw/realwaste/ --unzip -q
print("Downloaded:", os.listdir(f'{PROJECT_ROOT}/data/raw/'))

Dataset URL: https://www.kaggle.com/datasets/feyzazkefe/trashnet
License(s): unknown
Dataset URL: https://www.kaggle.com/datasets/mostafaabla/garbage-classification
License(s): ODbL-1.0
Dataset URL: https://www.kaggle.com/datasets/joebeachcapital/realwaste
License(s): Attribution 4.0 International (CC BY 4.0)
Downloaded: ['realwaste', 'trashnet', 'gd12']


In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2

albumentations_eval_transform = A.Compose([
    A.Resize(256, 256),
    A.CenterCrop(224, 224),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

class AlbumentationsDataset(Dataset):
    def __init__(self, samples, transform):
        self.samples = samples
        self.transform = transform
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        return self.transform(image=image)['image'], label

GD_MAP = {
    'cardboard':'cardboard','brown-glass':'glass','green-glass':'glass',
    'white-glass':'glass','metal':'metal','paper':'paper','plastic':'plastic',
    'trash':'trash','battery':'trash','biological':'trash','clothes':'trash','shoes':'trash',
}
REALWASTE_MAP = {
    'Cardboard':'cardboard','Glass':'glass','Metal':'metal','Paper':'paper',
    'Plastic':'plastic','Miscellaneous Trash':'trash','Food Organics':'trash',
    'Textile Trash':'trash','Vegetation':'trash',
}

def build_samples(root, class_map):
    samples = []
    for folder_name, unified_name in class_map.items():
        class_dir = Path(root) / folder_name
        if not class_dir.exists():
            print(f"  Warning: folder not found: {class_dir}")
            continue
        idx = CLASS_TO_IDX[unified_name]
        for img_file in class_dir.iterdir():
            if img_file.suffix.lower() in ['.jpg','.jpeg','.png']:
                samples.append((str(img_file), idx))
    return samples

def build_trashnet_samples(root):
    samples = []
    for cls in CLASSES:
        class_dir = Path(root) / cls
        if not class_dir.exists():
            continue
        for img_file in class_dir.iterdir():
            if img_file.suffix.lower() in ['.jpg','.jpeg','.png']:
                samples.append((str(img_file), CLASS_TO_IDX[cls]))
    return samples

trashnet_samples = build_trashnet_samples(f'{PROJECT_ROOT}/data/raw/trashnet/dataset-resized')
gd_samples = build_samples(f'{PROJECT_ROOT}/data/raw/gd12/garbage_classification', GD_MAP)
realwaste_samples = build_samples(f'{PROJECT_ROOT}/data/raw/realwaste/realwaste-main/RealWaste', REALWASTE_MAP)
print(f"TrashNet: {len(trashnet_samples)} | GD12: {len(gd_samples)} | RealWaste: {len(realwaste_samples)}")
# Expect: TrashNet: 2527 | GD12: 15515 | RealWaste: 4752

TrashNet: 2527 | GD12: 15515 | RealWaste: 4752


In [ ]:
print("Saved checkpoints:", os.listdir(SAVE_DIR))

# Build RealWaste loader (all 6 classes)
realwaste_dataset = AlbumentationsDataset(realwaste_samples, albumentations_eval_transform)
realwaste_loader = DataLoader(realwaste_dataset, batch_size=32, shuffle=False, num_workers=2)

def evaluate_with_and_without_trash(model_name, checkpoint_path):
    model = get_model(model_name, num_classes=6)
    ckpt = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    model = model.to(device).eval()

    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in realwaste_loader:
            images = images.to(device)
            outputs = model(images)
            all_preds.extend(outputs.argmax(1).cpu().numpy())
            all_labels.extend(labels.numpy())
    all_preds = np.array(all_preds); all_labels = np.array(all_labels)

    # 6-class balanced accuracy (original)
    acc6 = balanced_accuracy_score(all_labels, all_preds)

    # 5-class: drop every sample whose TRUE label is trash (idx 5),
    # AND treat any prediction of trash as wrong (kept in, but no trash GT).
    # Cleanest: keep only samples whose true label is in 0..4.
    mask = all_labels != 5
    acc5 = balanced_accuracy_score(all_labels[mask], all_preds[mask])

    print(f"\n{model_name}:")
    print(f"  6-class OOD balanced acc (with trash): {acc6*100:.1f}%")
    print(f"  5-class OOD balanced acc (no trash):   {acc5*100:.1f}%")
    print(f"  -> difference: {(acc5-acc6)*100:+.1f} percentage points")
    return acc6, acc5

# Use whichever baseline combined checkpoint you have saved.
# This is the ORIGINAL TrashNet+GD model (not augmented, not 3-dataset).
evaluate_with_and_without_trash('resnet50', f'{SAVE_DIR}/resnet50_combined_best.pth')

Saved checkpoints: ['efficientnet_best.pth', 'vit_best.pth', 'resnet50_combined_best.pth', 'efficientnet_combined_best.pth', 'vit_combined_best.pth', 'gradcam_resnet50.png', 'gradcam_efficientnet.png', 'gradcam_vit.png', 'resnet50_augmented_best.pth', 'resnet50_3dataset_best.pth']


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]


resnet50:
  6-class OOD balanced acc (with trash): 54.7%
  5-class OOD balanced acc (no trash):   48.3%
  -> difference: -6.4 percentage points


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


(np.float64(0.5473354530860766), np.float64(0.4833085677996774))

In [ ]:
evaluate_with_and_without_trash('efficientnet', f'{SAVE_DIR}/efficientnet_combined_best.pth')
evaluate_with_and_without_trash('vit', f'{SAVE_DIR}/vit_combined_best.pth')

model.safetensors:   0%|          | 0.00/49.3M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")



efficientnet:
  6-class OOD balanced acc (with trash): 44.1%
  5-class OOD balanced acc (no trash):   36.4%
  -> difference: -7.7 percentage points


model.safetensors:   0%|          | 0.00/88.2M [00:00<?, ?B/s]


vit:
  6-class OOD balanced acc (with trash): 61.0%
  5-class OOD balanced acc (no trash):   58.5%
  -> difference: -2.5 percentage points


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


(np.float64(0.6102052421930874), np.float64(0.5850173749690543))

In [ ]:
random.seed(42)
rw_shuffled = realwaste_samples.copy()
random.shuffle(rw_shuffled)
n_rw_train = int(0.15 * len(rw_shuffled))
rw_train_slice = rw_shuffled[:n_rw_train]
rw_holdout_test = rw_shuffled[n_rw_train:]
print(f"RealWaste -> train slice: {len(rw_train_slice)}, held-out test: {len(rw_holdout_test)}")

all_samples = trashnet_samples + gd_samples + rw_train_slice
random.shuffle(all_samples)
n = len(all_samples)
n_train = int(0.85 * n)
train_samples = all_samples[:n_train]
val_samples = all_samples[n_train:]
print(f"Train: {len(train_samples)}, Val: {len(val_samples)}, Held-out OOD test: {len(rw_holdout_test)}")

train_loader = DataLoader(AlbumentationsDataset(train_samples, albumentations_eval_transform),
                          batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(AlbumentationsDataset(val_samples, albumentations_eval_transform),
                        batch_size=32, shuffle=False, num_workers=2)
holdout_loader = DataLoader(AlbumentationsDataset(rw_holdout_test, albumentations_eval_transform),
                            batch_size=32, shuffle=False, num_workers=2)

train_labels = [l for _, l in train_samples]
counts = Counter(train_labels)
cc = np.array([counts[i] for i in range(6)], dtype=float)
weights = cc.sum() / (6 * cc)
criterion = nn.CrossEntropyLoss(weight=torch.FloatTensor(weights).to(device))
print("Weights:", {CLASSES[i]: round(weights[i],3) for i in range(6)})

RealWaste -> train slice: 712, held-out test: 4040
Train: 15940, Val: 2814, Held-out OOD test: 4040
Weights: {'cardboard': np.float64(2.308), 'glass': np.float64(1.21), 'metal': np.float64(2.417), 'paper': np.float64(1.831), 'plastic': np.float64(2.09), 'trash': np.float64(0.303)}


In [ ]:
def train_finetune(model_name, num_epochs=20):
    model = get_model(model_name, num_classes=6).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
    best = float('inf')
    for epoch in range(1, num_epochs+1):
        model.train()
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(images), labels)
            loss.backward(); optimizer.step()
        model.eval()
        vl, vp, vlab = 0, [], []
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                out = model(images)
                vl += criterion(out, labels).item()
                vp.extend(out.argmax(1).cpu().numpy()); vlab.extend(labels.cpu().numpy())
        va = balanced_accuracy_score(vlab, vp); avl = vl/len(val_loader)
        scheduler.step()
        print(f"Epoch {epoch}/{num_epochs} | Val: {va:.4f} | Loss: {avl:.4f}")
        if avl < best:
            best = avl
            torch.save({'model_state_dict': model.state_dict()},
                       f'{SAVE_DIR}/{model_name}_partial_finetune_best.pth')
            print("  saved")
    # held-out RealWaste test
    model.eval(); tp, tl = [], []
    with torch.no_grad():
        for images, labels in holdout_loader:
            images = images.to(device)
            tp.extend(model(images).argmax(1).cpu().numpy()); tl.extend(labels.numpy())
    acc = balanced_accuracy_score(tl, tp)
    print(f"\nHeld-out RealWaste (85%) balanced acc: {acc*100:.1f}%")
    print(classification_report(tl, tp, target_names=CLASSES, zero_division=0))
    return acc

ft_ood = train_finetune('resnet50', num_epochs=20)
print("\n=== ResNet-50 comparison ===")
print(f"  TrashNet+GD only (zero-shot)       OOD: 60.0%")
print(f"  + 15% RealWaste fine-tune          OOD: {ft_ood*100:.1f}%")

Epoch 1/20 | Val: 0.9290 | Loss: 0.2594
  saved
Epoch 2/20 | Val: 0.9625 | Loss: 0.1302
  saved
Epoch 3/20 | Val: 0.9620 | Loss: 0.1061
  saved
Epoch 4/20 | Val: 0.9692 | Loss: 0.0937
  saved
Epoch 5/20 | Val: 0.9695 | Loss: 0.0869
  saved
Epoch 6/20 | Val: 0.9721 | Loss: 0.0856
  saved
Epoch 7/20 | Val: 0.9762 | Loss: 0.0762
  saved
Epoch 8/20 | Val: 0.9765 | Loss: 0.0780
Epoch 9/20 | Val: 0.9768 | Loss: 0.0792
Epoch 10/20 | Val: 0.9747 | Loss: 0.0834
Epoch 11/20 | Val: 0.9716 | Loss: 0.0867
Epoch 12/20 | Val: 0.9744 | Loss: 0.0819
Epoch 13/20 | Val: 0.9749 | Loss: 0.0900
Epoch 14/20 | Val: 0.9744 | Loss: 0.0849
Epoch 15/20 | Val: 0.9724 | Loss: 0.0931
Epoch 16/20 | Val: 0.9749 | Loss: 0.0830
Epoch 17/20 | Val: 0.9737 | Loss: 0.0805
Epoch 18/20 | Val: 0.9708 | Loss: 0.0844
Epoch 19/20 | Val: 0.9756 | Loss: 0.0919
Epoch 20/20 | Val: 0.9771 | Loss: 0.0855

Held-out RealWaste (85%) balanced acc: 76.4%
              precision    recall  f1-score   support

   cardboard       0.73      0.6

In [ ]:
# Sanity check: re-run the PLAIN 6-class evaluation, exactly as in EXP009,
# on the same checkpoint and same RealWaste set, with no trash-dropping logic.
def evaluate_plain_6class(model_name, checkpoint_path):
    model = get_model(model_name, num_classes=6)
    ckpt = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    model = model.to(device).eval()

    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in realwaste_loader:
            images = images.to(device)
            outputs = model(images)
            all_preds.extend(outputs.argmax(1).cpu().numpy())
            all_labels.extend(labels.numpy())

    acc = balanced_accuracy_score(all_labels, all_preds)
    report = classification_report(all_labels, all_preds, target_names=CLASSES, zero_division=0)
    print(f"{model_name} — plain 6-class OOD balanced accuracy: {acc*100:.2f}%")
    print(report)
    return acc

print("Re-checking ResNet-50 baseline on RealWaste...")
recheck_acc = evaluate_plain_6class('resnet50', f'{SAVE_DIR}/resnet50_combined_best.pth')
print(f"\nOriginally logged (EXP009): 60.0%")
print(f"Yesterday's diagnostic run: 54.7%")
print(f"This re-check: {recheck_acc*100:.2f}%")

Re-checking ResNet-50 baseline on RealWaste...
resnet50 — plain 6-class OOD balanced accuracy: 54.73%
              precision    recall  f1-score   support

   cardboard       0.54      0.39      0.46       461
       glass       0.49      0.39      0.44       420
       metal       0.69      0.60      0.64       790
       paper       0.47      0.61      0.53       500
     plastic       0.77      0.42      0.55       921
       trash       0.64      0.87      0.74      1660

    accuracy                           0.62      4752
   macro avg       0.60      0.55      0.56      4752
weighted avg       0.63      0.62      0.61      4752


Originally logged (EXP009): 60.0%
Yesterday's diagnostic run: 54.7%
This re-check: 54.73%


In [ ]:
print("Training EfficientNet with 15% RealWaste fine-tune slice...")
effnet_ft_ood = train_finetune('efficientnet', num_epochs=20)

Training EfficientNet with 15% RealWaste fine-tune slice...
Epoch 1/20 | Val: 0.9489 | Loss: 0.1463
  saved
Epoch 2/20 | Val: 0.9671 | Loss: 0.1045
  saved
Epoch 3/20 | Val: 0.9608 | Loss: 0.1314
Epoch 4/20 | Val: 0.9702 | Loss: 0.1075
Epoch 5/20 | Val: 0.9590 | Loss: 0.5217
Epoch 6/20 | Val: 0.9694 | Loss: 0.1148
Epoch 7/20 | Val: 0.9673 | Loss: 0.1079
Epoch 8/20 | Val: 0.9699 | Loss: 0.0861
  saved
Epoch 9/20 | Val: 0.9708 | Loss: 0.0999
Epoch 10/20 | Val: 0.9686 | Loss: 0.1038
Epoch 11/20 | Val: 0.9711 | Loss: 0.0964


In [ ]:
print("Training ViT with 15% RealWaste fine-tune slice...")
vit_ft_ood = train_finetune('vit', num_epochs=20)

print("\n" + "="*60)
print("FULL FINE-TUNE COMPARISON — ALL 3 MODELS")
print("="*60)
print(f"  ResNet-50:     zero-shot 60.0% -> fine-tuned {ft_ood*100:.1f}%")
print(f"  EfficientNet:  zero-shot 53.6% -> fine-tuned {effnet_ft_ood*100:.1f}%")
print(f"  ViT-Small:     zero-shot 62.2% -> fine-tuned {vit_ft_ood*100:.1f}%")

In [ ]:
realwaste_dataset = AlbumentationsDataset(realwaste_samples, albumentations_eval_transform)
realwaste_loader = DataLoader(realwaste_dataset, batch_size=32, shuffle=False, num_workers=2)

def evaluate_plain_6class(model_name, checkpoint_path):
    model = get_model(model_name, num_classes=6)
    ckpt = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    model = model.to(device).eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in realwaste_loader:
            images = images.to(device)
            outputs = model(images)
            all_preds.extend(outputs.argmax(1).cpu().numpy())
            all_labels.extend(labels.numpy())
    acc = balanced_accuracy_score(all_labels, all_preds)
    print(f"{model_name} — plain 6-class OOD balanced accuracy: {acc*100:.2f}%")
    return acc

recheck_acc = evaluate_plain_6class('resnet50', f'{SAVE_DIR}/resnet50_combined_best.pth')
print(f"Originally logged: 60.0% | Yesterday: 54.7% | Today: {recheck_acc*100:.2f}%")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]

resnet50 — plain 6-class OOD balanced accuracy: 54.73%
Originally logged: 60.0% | Yesterday: 54.7% | Today: 54.73%


In [ ]:
def resume_finetune(model_name, checkpoint_path, num_epochs=12, start_epoch=9):
    model = get_model(model_name, num_classes=6)
    ckpt = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    model = model.to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)
    for _ in range(start_epoch - 1):
        scheduler.step()  # fast-forward scheduler to match where we left off

    best = float('inf')
    for epoch in range(start_epoch, start_epoch + num_epochs):
        model.train()
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(images), labels)
            loss.backward(); optimizer.step()
        model.eval()
        vl, vp, vlab = 0, [], []
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                out = model(images)
                vl += criterion(out, labels).item()
                vp.extend(out.argmax(1).cpu().numpy()); vlab.extend(labels.cpu().numpy())
        va = balanced_accuracy_score(vlab, vp); avl = vl/len(val_loader)
        scheduler.step()
        print(f"Epoch {epoch}/20 | Val: {va:.4f} | Loss: {avl:.4f}")
        if avl < best:
            best = avl
            torch.save({'model_state_dict': model.state_dict()},
                       f'{SAVE_DIR}/{model_name}_partial_finetune_best.pth')
            print("  saved")

    model.eval(); tp, tl = [], []
    with torch.no_grad():
        for images, labels in holdout_loader:
            images = images.to(device)
            tp.extend(model(images).argmax(1).cpu().numpy()); tl.extend(labels.numpy())
    acc = balanced_accuracy_score(tl, tp)
    print(f"\nHeld-out RealWaste balanced acc: {acc*100:.1f}%")
    print(classification_report(tl, tp, target_names=CLASSES, zero_division=0))
    return acc

effnet_ft_ood = resume_finetune('efficientnet', f'{SAVE_DIR}/efficientnet_partial_finetune_best.pth',
                                  num_epochs=12, start_epoch=9)

model.safetensors:   0%|          | 0.00/49.3M [00:00<?, ?B/s]

/tmp/ipykernel_2214/3617098211.py:10: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()  # fast-forward scheduler to match where we left off


Epoch 9/20 | Val: 0.9779 | Loss: 0.1001
  saved
Epoch 10/20 | Val: 0.9773 | Loss: 0.0918
  saved
Epoch 11/20 | Val: 0.9718 | Loss: 0.0972
Epoch 12/20 | Val: 0.9758 | Loss: 0.0897
  saved
Epoch 13/20 | Val: 0.9785 | Loss: 0.0791
  saved
Epoch 14/20 | Val: 0.9761 | Loss: 0.0828
Epoch 15/20 | Val: 0.9765 | Loss: 0.0825
Epoch 16/20 | Val: 0.9772 | Loss: 0.0766
  saved
Epoch 17/20 | Val: 0.9779 | Loss: 0.0922
Epoch 18/20 | Val: 0.9760 | Loss: 0.0868
Epoch 19/20 | Val: 0.9792 | Loss: 0.0806
Epoch 20/20 | Val: 0.9755 | Loss: 0.0850

Held-out RealWaste balanced acc: 79.9%
              precision    recall  f1-score   support

   cardboard       0.80      0.68      0.73       389
       glass       0.83      0.80      0.81       362
       metal       0.83      0.78      0.81       664
       paper       0.78      0.79      0.78       441
     plastic       0.71      0.86      0.78       779
       trash       0.92      0.88      0.90      1405

    accuracy                           0.82      

In [ ]:
vit_ft_ood = train_finetune('vit', num_epochs=20)


NameError: name 'train_finetune' is not defined

In [ ]:
def train_finetune(model_name, num_epochs=20):
    model = get_model(model_name, num_classes=6).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
    best = float('inf')
    for epoch in range(1, num_epochs+1):
        model.train()
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(images), labels)
            loss.backward(); optimizer.step()
        model.eval()
        vl, vp, vlab = 0, [], []
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                out = model(images)
                vl += criterion(out, labels).item()
                vp.extend(out.argmax(1).cpu().numpy()); vlab.extend(labels.cpu().numpy())
        va = balanced_accuracy_score(vlab, vp); avl = vl/len(val_loader)
        scheduler.step()
        print(f"Epoch {epoch}/{num_epochs} | Val: {va:.4f} | Loss: {avl:.4f}")
        if avl < best:
            best = avl
            torch.save({'model_state_dict': model.state_dict()},
                       f'{SAVE_DIR}/{model_name}_partial_finetune_best.pth')
            print("  saved")
    model.eval(); tp, tl = [], []
    with torch.no_grad():
        for images, labels in holdout_loader:
            images = images.to(device)
            tp.extend(model(images).argmax(1).cpu().numpy()); tl.extend(labels.numpy())
    acc = balanced_accuracy_score(tl, tp)
    print(f"\nHeld-out RealWaste balanced acc: {acc*100:.1f}%")
    print(classification_report(tl, tp, target_names=CLASSES, zero_division=0))
    return acc

In [ ]:
vit_ft_ood = train_finetune('vit', num_epochs=20)

model.safetensors:   0%|          | 0.00/88.2M [00:00<?, ?B/s]

Epoch 1/20 | Val: 0.9472 | Loss: 0.1466
  saved
Epoch 2/20 | Val: 0.9639 | Loss: 0.1137
  saved
Epoch 3/20 | Val: 0.9669 | Loss: 0.1172
Epoch 4/20 | Val: 0.9544 | Loss: 0.1791
Epoch 5/20 | Val: 0.9677 | Loss: 0.1418
Epoch 6/20 | Val: 0.9626 | Loss: 0.1320
Epoch 7/20 | Val: 0.9773 | Loss: 0.0725
  saved
Epoch 8/20 | Val: 0.9656 | Loss: 0.1223
Epoch 9/20 | Val: 0.9767 | Loss: 0.0806
Epoch 10/20 | Val: 0.9746 | Loss: 0.0970
Epoch 11/20 | Val: 0.9783 | Loss: 0.0770
Epoch 12/20 | Val: 0.9804 | Loss: 0.0693
  saved
Epoch 13/20 | Val: 0.9805 | Loss: 0.0710
Epoch 14/20 | Val: 0.9805 | Loss: 0.0702
Epoch 15/20 | Val: 0.9806 | Loss: 0.0662
  saved
Epoch 16/20 | Val: 0.9813 | Loss: 0.0693
Epoch 17/20 | Val: 0.9814 | Loss: 0.0689
Epoch 18/20 | Val: 0.9814 | Loss: 0.0691
Epoch 19/20 | Val: 0.9813 | Loss: 0.0695
Epoch 20/20 | Val: 0.9813 | Loss: 0.0696

Held-out RealWaste balanced acc: 81.6%
              precision    recall  f1-score   support

   cardboard       0.80      0.72      0.76       389


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from google.colab import files
uploaded = files.upload()   # upload waste-classification-main.zip

Mounted at /content/drive


Saving waste-classification-main.zip to waste-classification-main.zip


In [ ]:
import zipfile, importlib.util, torch, torch.nn as nn, random, os
import numpy as np
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import balanced_accuracy_score, classification_report
from collections import Counter

with zipfile.ZipFile('/content/waste-classification-main.zip', 'r') as z:
    z.extractall('/content/')

PROJECT_ROOT = '/content/waste-classification-main'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SAVE_DIR = '/content/drive/MyDrive/waste_experiments'
CLASSES = ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']
CLASS_TO_IDX = {cls: idx for idx, cls in enumerate(CLASSES)}

def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

models_module = load_module("models", f"{PROJECT_ROOT}/src/models.py")
get_model = models_module.get_model
print(f"Device: {device}")
print(f"Saved checkpoints: {os.listdir(SAVE_DIR)}")

Device: cuda
Saved checkpoints: ['efficientnet_best.pth', 'vit_best.pth', 'resnet50_combined_best.pth', 'efficientnet_combined_best.pth', 'vit_combined_best.pth', 'gradcam_resnet50.png', 'gradcam_efficientnet.png', 'gradcam_vit.png', 'resnet50_augmented_best.pth', 'resnet50_3dataset_best.pth', 'resnet50_partial_finetune_best.pth', 'efficientnet_partial_finetune_best.pth', 'vit_partial_finetune_best.pth']


In [ ]:
!pip install albumentations grad-cam kaggle -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 126.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
os.environ['KAGGLE_USERNAME'] = 'esmaeilmolapour'
os.environ['KAGGLE_KEY'] = 'YOUR_KAGGLE_KEY_HERE'

In [ ]:
!kaggle datasets download -d kneroma/tacotrashdataset \
    -p {PROJECT_ROOT}/data/raw/taco/ --unzip -q

for root, dirs, files in os.walk(f'{PROJECT_ROOT}/data/raw/taco'):
    depth = root.count(os.sep) - f'{PROJECT_ROOT}/data/raw/taco'.count(os.sep)
    if depth <= 2:
        print(root, '->', dirs[:10], '|', len(files), 'files' if files else '')

Dataset URL: https://www.kaggle.com/datasets/kneroma/tacotrashdataset
License(s): other
/content/waste-classification-main/data/raw/taco -> ['data'] | 3 files
/content/waste-classification-main/data/raw/taco/data -> ['batch_2', 'batch_12', 'batch_9', 'batch_4', 'batch_14', 'batch_13', 'batch_5', 'batch_15', 'batch_10', 'batch_8'] | 1 files
/content/waste-classification-main/data/raw/taco/data/batch_2 -> [] | 92 files
/content/waste-classification-main/data/raw/taco/data/batch_12 -> [] | 100 files
/content/waste-classification-main/data/raw/taco/data/batch_9 -> [] | 100 files
/content/waste-classification-main/data/raw/taco/data/batch_4 -> [] | 89 files
/content/waste-classification-main/data/raw/taco/data/batch_14 -> [] | 100 files
/content/waste-classification-main/data/raw/taco/data/batch_13 -> [] | 100 files
/content/waste-classification-main/data/raw/taco/data/batch_5 -> [] | 112 files
/content/waste-classification-main/data/raw/taco/data/batch_15 -> [] | 85 files
/content/waste-cl

In [ ]:
print(os.listdir(f'{PROJECT_ROOT}/data/raw/taco/data'))

['batch_2', 'batch_12', 'batch_9', 'batch_4', 'batch_14', 'batch_13', 'batch_5', 'batch_15', 'batch_10', 'batch_8', 'batch_11', 'annotations.json', 'batch_6', 'batch_7', 'batch_1', 'batch_3']


In [ ]:
import json

ann_path = f'{PROJECT_ROOT}/data/raw/taco/data/annotations.json'

with open(ann_path) as f:
    coco = json.load(f)

print(f"Images: {len(coco['images'])}")
print(f"Annotations: {len(coco['annotations'])}")
print(f"Categories: {len(coco['categories'])}")
print("\nAll categories:")
for cat in coco['categories']:
    print(f"  id={cat['id']}, name='{cat['name']}', supercategory='{cat['supercategory']}'")

supercats = sorted(set(cat['supercategory'] for cat in coco['categories']))
print(f"\nAll {len(supercats)} unique supercategories:")
for sc in supercats:
    print(f"  {sc}")

# Also check what a sample image entry looks like, since file_name format matters for locating images across batch folders
print("\nSample image entry:")
print(coco['images'][0])

Images: 1500
Annotations: 4784
Categories: 60

All categories:
  id=0, name='Aluminium foil', supercategory='Aluminium foil'
  id=1, name='Battery', supercategory='Battery'
  id=2, name='Aluminium blister pack', supercategory='Blister pack'
  id=3, name='Carded blister pack', supercategory='Blister pack'
  id=4, name='Other plastic bottle', supercategory='Bottle'
  id=5, name='Clear plastic bottle', supercategory='Bottle'
  id=6, name='Glass bottle', supercategory='Bottle'
  id=7, name='Plastic bottle cap', supercategory='Bottle cap'
  id=8, name='Metal bottle cap', supercategory='Bottle cap'
  id=9, name='Broken glass', supercategory='Broken glass'
  id=10, name='Food Can', supercategory='Can'
  id=11, name='Aerosol', supercategory='Can'
  id=12, name='Drink can', supercategory='Can'
  id=13, name='Toilet tube', supercategory='Carton'
  id=14, name='Other carton', supercategory='Carton'
  id=15, name='Egg carton', supercategory='Carton'
  id=16, name='Drink carton', supercategory='Car

In [ ]:
# Map at the FINE category level (not supercategory) since e.g. "Bottle"
# contains both glass and plastic bottles - mapping by supercategory would
# misclassify "Glass bottle" as the same bucket as "Clear plastic bottle".

TACO_CATEGORY_MAP = {
    'Aluminium foil': 'metal',
    'Battery': 'trash',
    'Aluminium blister pack': 'metal',
    'Carded blister pack': 'trash',
    'Other plastic bottle': 'plastic',
    'Clear plastic bottle': 'plastic',
    'Glass bottle': 'glass',
    'Plastic bottle cap': 'plastic',
    'Metal bottle cap': 'metal',
    'Broken glass': 'glass',
    'Food Can': 'metal',
    'Aerosol': 'metal',
    'Drink can': 'metal',
    'Toilet tube': 'cardboard',
    'Other carton': 'cardboard',
    'Egg carton': 'cardboard',
    'Drink carton': 'cardboard',
    'Corrugated carton': 'cardboard',
    'Meal carton': 'cardboard',
    'Pizza box': 'cardboard',
    'Paper cup': 'paper',
    'Disposable plastic cup': 'plastic',
    'Foam cup': 'trash',
    'Glass cup': 'glass',
    'Other plastic cup': 'plastic',
    'Food waste': 'trash',
    'Glass jar': 'glass',
    'Plastic lid': 'plastic',
    'Metal lid': 'metal',
    'Other plastic': 'plastic',
    'Magazine paper': 'paper',
    'Tissues': 'trash',
    'Wrapping paper': 'paper',
    'Normal paper': 'paper',
    'Paper bag': 'paper',
    'Plastified paper bag': 'paper',
    'Plastic film': 'plastic',
    'Six pack rings': 'plastic',
    'Garbage bag': 'plastic',
    'Other plastic wrapper': 'plastic',
    'Single-use carrier bag': 'plastic',
    'Polypropylene bag': 'plastic',
    'Crisp packet': 'plastic',
    'Spread tub': 'plastic',
    'Tupperware': 'plastic',
    'Disposable food container': 'plastic',
    'Foam food container': 'trash',
    'Other plastic container': 'plastic',
    'Plastic glooves': 'trash',
    'Plastic utensils': 'plastic',
    'Pop tab': 'metal',
    'Rope & strings': 'trash',
    'Scrap metal': 'metal',
    'Shoe': 'trash',
    'Squeezable tube': 'plastic',
    'Plastic straw': 'plastic',
    'Paper straw': 'paper',
    'Styrofoam piece': 'trash',
    'Unlabeled litter': 'trash',
    'Cigarette': 'trash',
}

# Sanity check: every category from the dataset must be in our map
dataset_cats = set(cat['name'] for cat in coco['categories'])
mapped_cats = set(TACO_CATEGORY_MAP.keys())
missing = dataset_cats - mapped_cats
extra = mapped_cats - dataset_cats
print(f"Missing from map: {missing}")
print(f"Extra in map (not in dataset): {extra}")
print("Mapping complete." if not missing and not extra else "FIX MAPPING BEFORE CONTINUING")

Missing from map: set()
Extra in map (not in dataset): set()
Mapping complete.


In [ ]:
import cv2

# Build lookup: category_id -> our unified class
cat_id_to_unified = {}
for cat in coco['categories']:
    unified = TACO_CATEGORY_MAP[cat['name']]
    cat_id_to_unified[cat['id']] = unified

# Build lookup: image_id -> file_name
image_id_to_file = {img['id']: img['file_name'] for img in coco['images']}

taco_data_root = f'{PROJECT_ROOT}/data/raw/taco/data'
taco_crops_dir = f'{PROJECT_ROOT}/data/raw/taco/crops'
os.makedirs(taco_crops_dir, exist_ok=True)

taco_samples = []
skipped = 0

for i, ann in enumerate(coco['annotations']):
    img_id = ann['image_id']
    cat_id = ann['category_id']
    bbox = ann['bbox']  # COCO format: [x, y, width, height]
    unified_label = cat_id_to_unified[cat_id]
    label_idx = CLASS_TO_IDX[unified_label]

    file_name = image_id_to_file[img_id]
    img_path = os.path.join(taco_data_root, file_name)

    if not os.path.exists(img_path):
        skipped += 1
        continue

    image = cv2.imread(img_path)
    if image is None:
        skipped += 1
        continue

    h, w = image.shape[:2]
    x, y, bw, bh = bbox
    x1, y1 = max(0, int(x)), max(0, int(y))
    x2, y2 = min(w, int(x + bw)), min(h, int(y + bh))

    if x2 <= x1 or y2 <= y1:
        skipped += 1
        continue

    crop = image[y1:y2, x1:x2]
    if crop.size == 0:
        skipped += 1
        continue

    crop_filename = f"{taco_crops_dir}/crop_{i:05d}.jpg"
    cv2.imwrite(crop_filename, crop)
    taco_samples.append((crop_filename, label_idx))

    if (i + 1) % 1000 == 0:
        print(f"  Processed {i+1}/{len(coco['annotations'])} annotations...")

print(f"\nTACO crops built: {len(taco_samples)}")
print(f"Skipped (missing file / invalid bbox): {skipped}")

# Class distribution
taco_labels = [l for _, l in taco_samples]
taco_counts = Counter(taco_labels)
print("\nClass distribution in TACO crops:")
for i, cls in enumerate(CLASSES):
    print(f"  {cls:10s}: {taco_counts[i]}")

  Processed 1000/4784 annotations...
  Processed 2000/4784 annotations...
  Processed 3000/4784 annotations...
  Processed 4000/4784 annotations...

TACO crops built: 4784
Skipped (missing file / invalid bbox): 0

Class distribution in TACO crops:
  cardboard : 251
  glass     : 254
  metal     : 550
  paper     : 204
  plastic   : 2108
  trash     : 1417


In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2

albumentations_eval_transform_rgb = A.Compose([
    A.Resize(256, 256),
    A.CenterCrop(224, 224),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

class TACODataset(Dataset):
    def __init__(self, samples, transform):
        self.samples = samples
        self.transform = transform
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        return self.transform(image=image)['image'], label

taco_dataset = TACODataset(taco_samples, albumentations_eval_transform_rgb)
taco_loader = DataLoader(taco_dataset, batch_size=32, shuffle=False, num_workers=2)

def evaluate_on_taco(model_name, checkpoint_path):
    model = get_model(model_name, num_classes=6)
    ckpt = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    model = model.to(device).eval()

    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in taco_loader:
            images = images.to(device)
            outputs = model(images)
            all_preds.extend(outputs.argmax(1).cpu().numpy())
            all_labels.extend(labels.numpy())

    acc = balanced_accuracy_score(all_labels, all_preds)
    report = classification_report(all_labels, all_preds, target_names=CLASSES, zero_division=0)
    print(f"\n{model_name} — TACO balanced accuracy: {acc*100:.1f}%")
    print(report)
    return acc

print("="*60)
print("EVALUATING FINE-TUNED MODELS ON TACO (independent, never seen)")
print("="*60)

resnet_taco = evaluate_on_taco('resnet50', f'{SAVE_DIR}/resnet50_partial_finetune_best.pth')
effnet_taco = evaluate_on_taco('efficientnet', f'{SAVE_DIR}/efficientnet_partial_finetune_best.pth')
vit_taco = evaluate_on_taco('vit', f'{SAVE_DIR}/vit_partial_finetune_best.pth')

print("\n" + "="*60)
print("SUMMARY — RealWaste held-out vs TACO (independent)")
print("="*60)
print(f"  ResNet-50:     RealWaste 76.4%  |  TACO {resnet_taco*100:.1f}%")
print(f"  EfficientNet:  RealWaste 79.9%  |  TACO {effnet_taco*100:.1f}%")
print(f"  ViT-Small:     RealWaste 81.6%  |  TACO {vit_taco*100:.1f}%")

EVALUATING FINE-TUNED MODELS ON TACO (independent, never seen)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]


resnet50 — TACO balanced accuracy: 33.3%
              precision    recall  f1-score   support

   cardboard       0.15      0.33      0.20       251
       glass       0.25      0.25      0.25       254
       metal       0.62      0.30      0.41       550
       paper       0.20      0.24      0.22       204
     plastic       0.69      0.25      0.36      2108
       trash       0.33      0.63      0.43      1417

    accuracy                           0.37      4784
   macro avg       0.37      0.33      0.31      4784
weighted avg       0.50      0.37      0.37      4784



model.safetensors:   0%|          | 0.00/49.3M [00:00<?, ?B/s]


efficientnet — TACO balanced accuracy: 34.1%
              precision    recall  f1-score   support

   cardboard       0.21      0.34      0.26       251
       glass       0.24      0.22      0.23       254
       metal       0.55      0.33      0.41       550
       paper       0.15      0.24      0.18       204
     plastic       0.76      0.21      0.33      2108
       trash       0.35      0.71      0.47      1417

    accuracy                           0.38      4784
   macro avg       0.37      0.34      0.31      4784
weighted avg       0.53      0.38      0.36      4784



model.safetensors:   0%|          | 0.00/88.2M [00:00<?, ?B/s]


vit — TACO balanced accuracy: 33.8%
              precision    recall  f1-score   support

   cardboard       0.20      0.37      0.26       251
       glass       0.38      0.36      0.37       254
       metal       0.76      0.30      0.43       550
       paper       0.16      0.12      0.14       204
     plastic       0.59      0.30      0.40      2108
       trash       0.31      0.58      0.40      1417

    accuracy                           0.38      4784
   macro avg       0.40      0.34      0.33      4784
weighted avg       0.48      0.38      0.38      4784


SUMMARY — RealWaste held-out vs TACO (independent)
  ResNet-50:     RealWaste 76.4%  |  TACO 33.3%
  EfficientNet:  RealWaste 79.9%  |  TACO 34.1%
  ViT-Small:     RealWaste 81.6%  |  TACO 33.8%


In [ ]:
widths, heights = [], []
for img_path, _ in taco_samples:
    img = cv2.imread(img_path)
    h, w = img.shape[:2]
    widths.append(w)
    heights.append(h)

widths = np.array(widths)
heights = np.array(heights)

print(f"Width  — min: {widths.min()}, median: {np.median(widths):.0f}, max: {widths.max()}")
print(f"Height — min: {heights.min()}, median: {np.median(heights):.0f}, max: {heights.max()}")
print(f"\nCrops smaller than 32x32 pixels: {((widths < 32) | (heights < 32)).sum()} / {len(widths)}")
print(f"Crops smaller than 64x64 pixels: {((widths < 64) | (heights < 64)).sum()} / {len(widths)}")
print(f"Crops smaller than 100x100 pixels: {((widths < 100) | (heights < 100)).sum()} / {len(widths)}")

Width  — min: 10, median: 180, max: 3758
Height — min: 4, median: 159, max: 3995

Crops smaller than 32x32 pixels: 650 / 4784
Crops smaller than 64x64 pixels: 1433 / 4784
Crops smaller than 100x100 pixels: 2024 / 4784


In [ ]:
MIN_SIZE = 100  # only keep crops where both dimensions are at least 100px

taco_samples_filtered = []
for img_path, label in taco_samples:
    img = cv2.imread(img_path)
    h, w = img.shape[:2]
    if w >= MIN_SIZE and h >= MIN_SIZE:
        taco_samples_filtered.append((img_path, label))

print(f"Original TACO samples: {len(taco_samples)}")
print(f"After filtering (>={MIN_SIZE}px both dims): {len(taco_samples_filtered)}")

filtered_labels = [l for _, l in taco_samples_filtered]
filtered_counts = Counter(filtered_labels)
print("\nClass distribution after filtering:")
for i, cls in enumerate(CLASSES):
    print(f"  {cls:10s}: {filtered_counts[i]}")

taco_dataset_filtered = TACODataset(taco_samples_filtered, albumentations_eval_transform_rgb)
taco_loader_filtered = DataLoader(taco_dataset_filtered, batch_size=32, shuffle=False, num_workers=2)

def evaluate_on_taco_filtered(model_name, checkpoint_path, loader):
    model = get_model(model_name, num_classes=6)
    ckpt = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    model = model.to(device).eval()

    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            all_preds.extend(outputs.argmax(1).cpu().numpy())
            all_labels.extend(labels.numpy())

    acc = balanced_accuracy_score(all_labels, all_preds)
    report = classification_report(all_labels, all_preds, target_names=CLASSES, zero_division=0)
    print(f"\n{model_name} — TACO (filtered, >={MIN_SIZE}px) balanced accuracy: {acc*100:.1f}%")
    print(report)
    return acc

print("="*60)
print(f"RE-EVALUATING ON FILTERED TACO (crops >= {MIN_SIZE}x{MIN_SIZE}px)")
print("="*60)

resnet_taco_f = evaluate_on_taco_filtered('resnet50', f'{SAVE_DIR}/resnet50_partial_finetune_best.pth', taco_loader_filtered)
effnet_taco_f = evaluate_on_taco_filtered('efficientnet', f'{SAVE_DIR}/efficientnet_partial_finetune_best.pth', taco_loader_filtered)
vit_taco_f = evaluate_on_taco_filtered('vit', f'{SAVE_DIR}/vit_partial_finetune_best.pth', taco_loader_filtered)

print("\n" + "="*60)
print("FINAL SUMMARY")
print("="*60)
print(f"  ResNet-50:     RealWaste 76.4%  |  TACO (all) 33.3%  |  TACO (filtered) {resnet_taco_f*100:.1f}%")
print(f"  EfficientNet:  RealWaste 79.9%  |  TACO (all) 34.1%  |  TACO (filtered) {effnet_taco_f*100:.1f}%")
print(f"  ViT-Small:     RealWaste 81.6%  |  TACO (all) 33.8%  |  TACO (filtered) {vit_taco_f*100:.1f}%")

Original TACO samples: 4784
After filtering (>=100px both dims): 2760

Class distribution after filtering:
  cardboard : 227
  glass     : 121
  metal     : 369
  paper     : 178
  plastic   : 1518
  trash     : 347
RE-EVALUATING ON FILTERED TACO (crops >= 100x100px)

resnet50 — TACO (filtered, >=100px) balanced accuracy: 38.8%
              precision    recall  f1-score   support

   cardboard       0.42      0.35      0.38       227
       glass       0.35      0.26      0.30       121
       metal       0.62      0.42      0.50       369
       paper       0.24      0.26      0.25       178
     plastic       0.72      0.32      0.44      1518
       trash       0.18      0.72      0.29       347

    accuracy                           0.38      2760
   macro avg       0.42      0.39      0.36      2760
weighted avg       0.57      0.38      0.41      2760


efficientnet — TACO (filtered, >=100px) balanced accuracy: 39.0%
              precision    recall  f1-score   support

   car

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2

albumentations_eval_transform = A.Compose([
    A.Resize(256, 256),
    A.CenterCrop(224, 224),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

class AlbumentationsDataset(Dataset):
    def __init__(self, samples, transform):
        self.samples = samples
        self.transform = transform
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        return self.transform(image=image)['image'], label

print("Ready.")

Ready.


In [ ]:
from google.colab import files
uploaded = files.upload()   # upload own_dataset_sorted.zip

Saving own_dataset_sorted.zip to own_dataset_sorted.zip


In [ ]:
import zipfile

with zipfile.ZipFile('/content/own_dataset_sorted.zip', 'r') as z:
    z.extractall(f'{PROJECT_ROOT}/data/raw/own_dataset/')

for root, dirs, filenames in os.walk(f'{PROJECT_ROOT}/data/raw/own_dataset'):
    if filenames:
        print(root, '->', len(filenames), 'files')

/content/waste-classification-main/data/raw/own_dataset -> 1 files
/content/waste-classification-main/data/raw/own_dataset/sorted/metal -> 17 files
/content/waste-classification-main/data/raw/own_dataset/sorted/glass -> 18 files
/content/waste-classification-main/data/raw/own_dataset/sorted/trash -> 7 files
/content/waste-classification-main/data/raw/own_dataset/sorted/plastic -> 56 files
/content/waste-classification-main/data/raw/own_dataset/sorted/cardboard -> 27 files
/content/waste-classification-main/data/raw/own_dataset/sorted/paper -> 41 files


In [ ]:
OWN_DATASET_MAP = {
    'cardboard': 'cardboard',
    'glass': 'glass',
    'metal': 'metal',
    'paper': 'paper',
    'plastic': 'plastic',
    'trash': 'trash',
}

def build_own_dataset_samples(root):
    samples = []
    for folder_name, unified_name in OWN_DATASET_MAP.items():
        class_dir = Path(root) / folder_name
        if not class_dir.exists():
            print(f"  Warning: {folder_name} not found")
            continue
        label_idx = CLASS_TO_IDX[unified_name]
        for img_file in class_dir.iterdir():
            if img_file.suffix.lower() in ['.jpg', '.jpeg', '.png']:
                samples.append((str(img_file), label_idx))
    return samples

own_samples = build_own_dataset_samples(f'{PROJECT_ROOT}/data/raw/own_dataset/sorted')
print(f"Own dataset: {len(own_samples)} images")

own_labels = [l for _, l in own_samples]
own_counts = Counter(own_labels)
print("\nClass distribution:")
for i, cls in enumerate(CLASSES):
    print(f"  {cls:10s}: {own_counts[i]}")

Own dataset: 166 images

Class distribution:
  cardboard : 27
  glass     : 18
  metal     : 17
  paper     : 41
  plastic   : 56
  trash     : 7


In [ ]:
own_dataset = AlbumentationsDataset(own_samples, albumentations_eval_transform)
own_loader = DataLoader(own_dataset, batch_size=32, shuffle=False, num_workers=2)

def evaluate_on_own(model_name, checkpoint_path, label=""):
    model = get_model(model_name, num_classes=6)
    ckpt = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    model = model.to(device).eval()

    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in own_loader:
            images = images.to(device)
            outputs = model(images)
            all_preds.extend(outputs.argmax(1).cpu().numpy())
            all_labels.extend(labels.numpy())

    acc = balanced_accuracy_score(all_labels, all_preds)
    report = classification_report(all_labels, all_preds, target_names=CLASSES, zero_division=0)
    print(f"\n{model_name} {label} — Own dataset balanced accuracy: {acc*100:.1f}%")
    print(report)
    return acc

print("="*60)
print("ZERO-SHOT MODELS (trained on TrashNet+GD only, never saw real-world data)")
print("="*60)
resnet_own_zs = evaluate_on_own('resnet50', f'{SAVE_DIR}/resnet50_combined_best.pth', "(zero-shot)")
effnet_own_zs = evaluate_on_own('efficientnet', f'{SAVE_DIR}/efficientnet_combined_best.pth', "(zero-shot)")
vit_own_zs = evaluate_on_own('vit', f'{SAVE_DIR}/vit_combined_best.pth', "(zero-shot)")

print("\n" + "="*60)
print("FINE-TUNED MODELS (15% RealWaste mixed into training)")
print("="*60)
resnet_own_ft = evaluate_on_own('resnet50', f'{SAVE_DIR}/resnet50_partial_finetune_best.pth', "(fine-tuned)")
effnet_own_ft = evaluate_on_own('efficientnet', f'{SAVE_DIR}/efficientnet_partial_finetune_best.pth', "(fine-tuned)")
vit_own_ft = evaluate_on_own('vit', f'{SAVE_DIR}/vit_partial_finetune_best.pth', "(fine-tuned)")

print("\n" + "="*60)
print("SUMMARY — Own Dataset (170 images, object-centric protocol)")
print("="*60)
print(f"  ResNet-50:     zero-shot {resnet_own_zs*100:.1f}%  |  fine-tuned {resnet_own_ft*100:.1f}%")
print(f"  EfficientNet:  zero-shot {effnet_own_zs*100:.1f}%  |  fine-tuned {effnet_own_ft*100:.1f}%")
print(f"  ViT-Small:     zero-shot {vit_own_zs*100:.1f}%  |  fine-tuned {vit_own_ft*100:.1f}%")

ZERO-SHOT MODELS (trained on TrashNet+GD only, never saw real-world data)


model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]


resnet50 (zero-shot) — Own dataset balanced accuracy: 36.3%
              precision    recall  f1-score   support

   cardboard       0.75      0.33      0.46        27
       glass       0.60      0.50      0.55        18
       metal       0.00      0.00      0.00        17
       paper       0.33      0.22      0.26        41
     plastic       0.78      0.12      0.22        56
       trash       0.07      1.00      0.13         7

    accuracy                           0.25       166
   macro avg       0.42      0.36      0.27       166
weighted avg       0.53      0.25      0.28       166



model.safetensors:   0%|          | 0.00/49.3M [00:00<?, ?B/s]


efficientnet (zero-shot) — Own dataset balanced accuracy: 29.8%
              precision    recall  f1-score   support

   cardboard       0.69      0.33      0.45        27
       glass       0.08      0.11      0.09        18
       metal       0.14      0.06      0.08        17
       paper       0.33      0.07      0.12        41
     plastic       0.80      0.21      0.34        56
       trash       0.07      1.00      0.13         7

    accuracy                           0.20       166
   macro avg       0.35      0.30      0.20       166
weighted avg       0.49      0.20      0.24       166



model.safetensors:   0%|          | 0.00/88.2M [00:00<?, ?B/s]


vit (zero-shot) — Own dataset balanced accuracy: 44.2%
              precision    recall  f1-score   support

   cardboard       0.68      0.70      0.69        27
       glass       0.31      0.44      0.36        18
       metal       0.00      0.00      0.00        17
       paper       0.75      0.07      0.13        41
     plastic       0.56      0.43      0.48        56
       trash       0.11      1.00      0.20         7

    accuracy                           0.37       166
   macro avg       0.40      0.44      0.31       166
weighted avg       0.52      0.37      0.36       166


FINE-TUNED MODELS (15% RealWaste mixed into training)

resnet50 (fine-tuned) — Own dataset balanced accuracy: 50.0%
              precision    recall  f1-score   support

   cardboard       0.61      0.70      0.66        27
       glass       0.69      0.50      0.58        18
       metal       0.10      0.06      0.07        17
       paper       0.52      0.29      0.38        41
     plastic 

In [ ]:
metal_paths = [p for p, l in own_samples if l == CLASS_TO_IDX['metal']]
print(f"Metal samples: {len(metal_paths)}")

for p in metal_paths[:5]:
    img = cv2.imread(p)
    if img is None:
        print(f"  FAILED TO LOAD: {p}")
    else:
        print(f"  {p.split('/')[-1]}: shape={img.shape}, dtype={img.dtype}")

Metal samples: 18
  metal_AM_009.jpeg: shape=(3260, 2444, 3), dtype=uint8
  metal_AM_002.JPG: shape=(4032, 3024, 3), dtype=uint8
  metal_AM_001.JPG: shape=(3024, 4032, 3), dtype=uint8
  metal_AM_013.jpeg: shape=(3260, 2444, 3), dtype=uint8
  metal_AM_012.jpeg: shape=(3260, 2444, 3), dtype=uint8


In [ ]:
metal_paths = [p for p, l in own_samples if l == CLASS_TO_IDX['metal']]
print(f"Metal samples: {len(metal_paths)}")

for p in metal_paths:
    img = cv2.imread(p)
    if img is None:
        print(f"  FAILED TO LOAD: {p}")
    else:
        print(f"  {p.split('/')[-1]}: shape={img.shape}, dtype={img.dtype}, min={img.min()}, max={img.max()}")

Metal samples: 17
  metal_AM_007.jpg: shape=(1280, 720, 3), dtype=uint8, min=0, max=255
  metal_AM_002.JPG: shape=(4032, 3024, 3), dtype=uint8, min=0, max=255
  metal_AM_006.JPG: shape=(4032, 3024, 3), dtype=uint8, min=0, max=255
  metal_AM_001.JPG: shape=(3024, 4032, 3), dtype=uint8, min=0, max=255
  metal_AM_013.jpeg: shape=(3260, 2444, 3), dtype=uint8, min=0, max=255
  metal_AM_012.jpeg: shape=(3260, 2444, 3), dtype=uint8, min=0, max=255
  metal_AM_003.JPG: shape=(3024, 4032, 3), dtype=uint8, min=0, max=255
  metal_AM_010.jpeg: shape=(3260, 2444, 3), dtype=uint8, min=0, max=255
  metal_AM_011.jpeg: shape=(3265, 2449, 3), dtype=uint8, min=0, max=255
  metal_AM_008.jpg: shape=(1280, 591, 3), dtype=uint8, min=0, max=255
  metal_AM_016.jpeg: shape=(3265, 2449, 3), dtype=uint8, min=0, max=255
  metal_AM_009.jpg: shape=(1280, 591, 3), dtype=uint8, min=0, max=255
  metal_AM_015.jpeg: shape=(3265, 2449, 3), dtype=uint8, min=0, max=255
  metal_AM_017.jpeg: shape=(2449, 3265, 3), dtype=uint8,

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, min(5, len(metal_paths)), figsize=(20, 4))
for i, p in enumerate(metal_paths[:5]):
    img = cv2.imread(p)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    axes[i].imshow(img_rgb)
    axes[i].set_title(p.split('/')[-1], fontsize=8)
    axes[i].axis('off')
plt.tight_layout()
plt.savefig('/content/metal_check.png', dpi=100)
plt.show()

In [ ]:
def show_predictions_for_class(model_name, checkpoint_path, class_name):
    model = get_model(model_name, num_classes=6)
    ckpt = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    model = model.to(device).eval()

    class_idx = CLASS_TO_IDX[class_name]
    paths = [p for p, l in own_samples if l == class_idx]

    print(f"\n{model_name} predictions for all '{class_name}' photos:")
    pred_counts = Counter()
    for p in paths:
        img = cv2.imread(p)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        tensor = albumentations_eval_transform(image=img)['image'].unsqueeze(0).to(device)
        with torch.no_grad():
            pred = model(tensor).argmax(1).item()
        pred_counts[CLASSES[pred]] += 1
        print(f"  {p.split('/')[-1]:25s} -> predicted: {CLASSES[pred]}")

    print(f"\n  Prediction distribution: {dict(pred_counts)}")

show_predictions_for_class('resnet50', f'{SAVE_DIR}/resnet50_partial_finetune_best.pth', 'metal')


resnet50 predictions for all 'metal' photos:
  metal_AM_007.jpg          -> predicted: cardboard
  metal_AM_002.JPG          -> predicted: trash
  metal_AM_006.JPG          -> predicted: trash
  metal_AM_001.JPG          -> predicted: plastic
  metal_AM_013.jpeg         -> predicted: trash
  metal_AM_012.jpeg         -> predicted: trash
  metal_AM_003.JPG          -> predicted: glass
  metal_AM_010.jpeg         -> predicted: trash
  metal_AM_011.jpeg         -> predicted: trash
  metal_AM_008.jpg          -> predicted: metal
  metal_AM_016.jpeg         -> predicted: trash
  metal_AM_009.jpg          -> predicted: cardboard
  metal_AM_015.jpeg         -> predicted: trash
  metal_AM_017.jpeg         -> predicted: trash
  metal_AM_014.jpeg         -> predicted: trash
  metal_AM_004.JPG          -> predicted: glass
  metal_AM_005.JPG          -> predicted: glass

  Prediction distribution: {'cardboard': 2, 'trash': 10, 'plastic': 1, 'glass': 3, 'metal': 1}


In [ ]:
def evaluate_excluding_class(model_name, checkpoint_path, exclude_class='metal'):
    model = get_model(model_name, num_classes=6)
    ckpt = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    model = model.to(device).eval()

    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in own_loader:
            images = images.to(device)
            outputs = model(images)
            all_preds.extend(outputs.argmax(1).cpu().numpy())
            all_labels.extend(labels.numpy())
    all_preds = np.array(all_preds); all_labels = np.array(all_labels)

    exclude_idx = CLASS_TO_IDX[exclude_class]
    mask = all_labels != exclude_idx
    acc_with = balanced_accuracy_score(all_labels, all_preds)
    acc_without = balanced_accuracy_score(all_labels[mask], all_preds[mask])

    print(f"{model_name}: with metal {acc_with*100:.1f}% | without metal {acc_without*100:.1f}%")
    return acc_with, acc_without

print("Fine-tuned models, own dataset, with vs without metal:")
evaluate_excluding_class('resnet50', f'{SAVE_DIR}/resnet50_partial_finetune_best.pth')
evaluate_excluding_class('efficientnet', f'{SAVE_DIR}/efficientnet_partial_finetune_best.pth')
evaluate_excluding_class('vit', f'{SAVE_DIR}/vit_partial_finetune_best.pth')

Fine-tuned models, own dataset, with vs without metal:


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


resnet50: with metal 50.0% | without metal 58.9%


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


efficientnet: with metal 42.1% | without metal 49.3%
vit: with metal 29.1% | without metal 35.0%


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


(np.float64(0.29141932292338796), np.float64(0.34970318750806556))

In [ ]:
def evaluate_zeroshot_excluding_metal(model_name, checkpoint_path):
    model = get_model(model_name, num_classes=6)
    ckpt = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    model = model.to(device).eval()

    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in own_loader:
            images = images.to(device)
            outputs = model(images)
            all_preds.extend(outputs.argmax(1).cpu().numpy())
            all_labels.extend(labels.numpy())
    all_preds = np.array(all_preds); all_labels = np.array(all_labels)

    mask = all_labels != CLASS_TO_IDX['metal']
    acc = balanced_accuracy_score(all_labels[mask], all_preds[mask])
    return acc

print("ZERO-SHOT models, own dataset, excluding metal:")
rn_zs_nomtl = evaluate_zeroshot_excluding_metal('resnet50', f'{SAVE_DIR}/resnet50_combined_best.pth')
ef_zs_nomtl = evaluate_zeroshot_excluding_metal('efficientnet', f'{SAVE_DIR}/efficientnet_combined_best.pth')
vt_zs_nomtl = evaluate_zeroshot_excluding_metal('vit', f'{SAVE_DIR}/vit_combined_best.pth')

print(f"  ResNet-50:     {rn_zs_nomtl*100:.1f}%")
print(f"  EfficientNet:  {ef_zs_nomtl*100:.1f}%")
print(f"  ViT-Small:     {vt_zs_nomtl*100:.1f}%")

print("\n" + "="*60)
print("COMPLETE PICTURE (excluding metal, 5 classes)")
print("="*60)
print(f"  ResNet-50:     zero-shot {rn_zs_nomtl*100:.1f}%  ->  fine-tuned 58.9%")
print(f"  EfficientNet:  zero-shot {ef_zs_nomtl*100:.1f}%  ->  fine-tuned 49.3%")
print(f"  ViT-Small:     zero-shot {vt_zs_nomtl*100:.1f}%  ->  fine-tuned 35.0%")

ZERO-SHOT models, own dataset, excluding metal:


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


  ResNet-50:     43.6%
  EfficientNet:  34.6%
  ViT-Small:     53.0%

COMPLETE PICTURE (excluding metal, 5 classes)
  ResNet-50:     zero-shot 43.6%  ->  fine-tuned 58.9%
  EfficientNet:  zero-shot 34.6%  ->  fine-tuned 49.3%
  ViT-Small:     zero-shot 53.0%  ->  fine-tuned 35.0%


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


In [ ]:
no_crop_transform = A.Compose([
    A.Resize(224, 224),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2
import torch.nn.functional as F

# New transform: direct resize to 224x224, no center crop
no_crop_transform = A.Compose([
    A.Resize(224, 224),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

# Rebuild own dataset with no-crop transform
own_dataset_nocrop = AlbumentationsDataset(own_samples, no_crop_transform)
own_loader_nocrop = DataLoader(own_dataset_nocrop, batch_size=32, shuffle=False, num_workers=2)

exclude_idx = CLASS_TO_IDX['metal']

def eval_simple(model_name, checkpoint_path, loader, label=""):
    model = get_model(model_name, num_classes=6)
    ckpt = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    model = model.to(device).eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            all_preds.extend(outputs.argmax(1).cpu().numpy())
            all_labels.extend(labels.numpy())
    all_preds = np.array(all_preds); all_labels = np.array(all_labels)
    mask = all_labels != exclude_idx
    acc = balanced_accuracy_score(all_labels[mask], all_preds[mask])
    print(f"  {model_name} {label}: {acc*100:.1f}%")
    return acc, model

print("="*60)
print("IMPROVEMENT #1: No-crop transform (Resize 224 directly)")
print("="*60)

# Zero-shot
print("\nZero-shot (no crop):")
rn_zs_nc, _ = eval_simple('resnet50', f'{SAVE_DIR}/resnet50_combined_best.pth', own_loader_nocrop, "zero-shot")
ef_zs_nc, _ = eval_simple('efficientnet', f'{SAVE_DIR}/efficientnet_combined_best.pth', own_loader_nocrop, "zero-shot")
vt_zs_nc, _ = eval_simple('vit', f'{SAVE_DIR}/vit_combined_best.pth', own_loader_nocrop, "zero-shot")

# Fine-tuned
print("\nFine-tuned (no crop):")
rn_ft_nc, _ = eval_simple('resnet50', f'{SAVE_DIR}/resnet50_partial_finetune_best.pth', own_loader_nocrop, "fine-tuned")
ef_ft_nc, _ = eval_simple('efficientnet', f'{SAVE_DIR}/efficientnet_partial_finetune_best.pth', own_loader_nocrop, "fine-tuned")
vt_ft_nc, _ = eval_simple('vit', f'{SAVE_DIR}/vit_partial_finetune_best.pth', own_loader_nocrop, "fine-tuned")

print("\nComparison vs center-crop baseline (5-class, excl metal):")
print(f"  ResNet-50:     crop 43.6/58.9%  ->  no-crop {rn_zs_nc*100:.1f}/{rn_ft_nc*100:.1f}%")
print(f"  EfficientNet:  crop 34.6/49.3%  ->  no-crop {ef_zs_nc*100:.1f}/{ef_ft_nc*100:.1f}%")
print(f"  ViT-Small:     crop 53.0/35.0%  ->  no-crop {vt_zs_nc*100:.1f}/{vt_ft_nc*100:.1f}%")

IMPROVEMENT #1: No-crop transform (Resize 224 directly)

Zero-shot (no crop):


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


  resnet50 zero-shot: 43.8%


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


  efficientnet zero-shot: 37.8%


...

  vit zero-shot: 47.7%

Fine-tuned (no crop):


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


  resnet50 fine-tuned: 54.7%


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


  efficientnet fine-tuned: 48.9%
  vit fine-tuned: 33.2%

Comparison vs center-crop baseline (5-class, excl metal):
  ResNet-50:     crop 43.6/58.9%  ->  no-crop 43.8/54.7%
  EfficientNet:  crop 34.6/49.3%  ->  no-crop 37.8/48.9%
  ViT-Small:     crop 53.0/35.0%  ->  no-crop 47.7/33.2%


In [ ]:
# TTA: average predictions over 5 augmented views per image
# Views: original, h-flip, small crop left, small crop right, slight rotation

tta_transforms = [
    A.Compose([A.Resize(224, 224), A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]), ToTensorV2()]),
    A.Compose([A.Resize(224, 224), A.HorizontalFlip(p=1.0), A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]), ToTensorV2()]),
    A.Compose([A.Resize(256, 256), A.CenterCrop(224, 224), A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]), ToTensorV2()]),
    A.Compose([A.Resize(256, 256), A.RandomCrop(224, 224), A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]), ToTensorV2()]),
    A.Compose([A.Resize(224, 224), A.Rotate(limit=(10, 10), p=1.0), A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]), ToTensorV2()]),
]

def eval_tta(model_name, checkpoint_path):
    model = get_model(model_name, num_classes=6)
    ckpt = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    model = model.to(device).eval()

    all_preds, all_labels = [], []
    for img_path, label in own_samples:
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        avg_logits = None
        for t in tta_transforms:
            tensor = t(image=img)['image'].unsqueeze(0).to(device)
            with torch.no_grad():
                logits = model(tensor)
            if avg_logits is None:
                avg_logits = logits
            else:
                avg_logits += logits
        avg_logits /= len(tta_transforms)
        pred = avg_logits.argmax(1).item()
        all_preds.append(pred)
        all_labels.append(label)

    all_preds = np.array(all_preds); all_labels = np.array(all_labels)
    mask = all_labels != exclude_idx
    acc = balanced_accuracy_score(all_labels[mask], all_preds[mask])
    print(f"  {model_name}: {acc*100:.1f}%")
    return acc

print("\n" + "="*60)
print("IMPROVEMENT #2: Test-Time Augmentation (5 views averaged)")
print("="*60)

print("\nFine-tuned + TTA:")
rn_tta = eval_tta('resnet50', f'{SAVE_DIR}/resnet50_partial_finetune_best.pth')
ef_tta = eval_tta('efficientnet', f'{SAVE_DIR}/efficientnet_partial_finetune_best.pth')
vt_tta_zs = eval_tta('vit', f'{SAVE_DIR}/vit_combined_best.pth')  # ViT: use zero-shot since fine-tuned was worse

print(f"\n  ResNet-50:     fine-tuned no-crop {rn_ft_nc*100:.1f}%  ->  TTA {rn_tta*100:.1f}%")
print(f"  EfficientNet:  fine-tuned no-crop {ef_ft_nc*100:.1f}%  ->  TTA {ef_tta*100:.1f}%")
print(f"  ViT-Small:     zero-shot no-crop {vt_zs_nc*100:.1f}%  ->  TTA {vt_tta_zs*100:.1f}%")


IMPROVEMENT #2: Test-Time Augmentation (5 views averaged)

Fine-tuned + TTA:


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


  resnet50: 60.5%


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


  efficientnet: 47.8%
  vit: 48.1%

  ResNet-50:     fine-tuned no-crop 54.7%  ->  TTA 60.5%
  EfficientNet:  fine-tuned no-crop 48.9%  ->  TTA 47.8%
  ViT-Small:     zero-shot no-crop 47.7%  ->  TTA 48.1%


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


In [ ]:
def eval_ensemble(checkpoints_list, label=""):
    """Average softmax outputs from multiple models."""
    models = []
    for model_name, checkpoint_path in checkpoints_list:
        model = get_model(model_name, num_classes=6)
        ckpt = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
        model.load_state_dict(ckpt['model_state_dict'])
        model = model.to(device).eval()
        models.append(model)

    all_preds, all_labels = [], []
    for img_path, label_idx in own_samples:
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        tensor = no_crop_transform(image=img)['image'].unsqueeze(0).to(device)

        avg_probs = None
        with torch.no_grad():
            for model in models:
                probs = F.softmax(model(tensor), dim=1)
                if avg_probs is None:
                    avg_probs = probs
                else:
                    avg_probs += probs
        avg_probs /= len(models)
        pred = avg_probs.argmax(1).item()
        all_preds.append(pred)
        all_labels.append(label_idx)

    all_preds = np.array(all_preds); all_labels = np.array(all_labels)
    mask = all_labels != exclude_idx
    acc = balanced_accuracy_score(all_labels[mask], all_preds[mask])
    return acc

print("\n" + "="*60)
print("IMPROVEMENT #3: Model Ensembles")
print("="*60)

# Ensemble A: all 3 fine-tuned
ens_all_ft = eval_ensemble([
    ('resnet50', f'{SAVE_DIR}/resnet50_partial_finetune_best.pth'),
    ('efficientnet', f'{SAVE_DIR}/efficientnet_partial_finetune_best.pth'),
    ('vit', f'{SAVE_DIR}/vit_partial_finetune_best.pth'),
])
print(f"  All 3 fine-tuned:                    {ens_all_ft*100:.1f}%")

# Ensemble B: ViT zero-shot + CNN fine-tuned (the smart combo)
ens_smart = eval_ensemble([
    ('resnet50', f'{SAVE_DIR}/resnet50_partial_finetune_best.pth'),
    ('efficientnet', f'{SAVE_DIR}/efficientnet_partial_finetune_best.pth'),
    ('vit', f'{SAVE_DIR}/vit_combined_best.pth'),  # zero-shot ViT
])
print(f"  ViT-zero-shot + CNN-fine-tuned:      {ens_smart*100:.1f}%")

# Ensemble C: all 3 zero-shot
ens_all_zs = eval_ensemble([
    ('resnet50', f'{SAVE_DIR}/resnet50_combined_best.pth'),
    ('efficientnet', f'{SAVE_DIR}/efficientnet_combined_best.pth'),
    ('vit', f'{SAVE_DIR}/vit_combined_best.pth'),
])
print(f"  All 3 zero-shot:                     {ens_all_zs*100:.1f}%")


IMPROVEMENT #3: Model Ensembles


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


  All 3 fine-tuned:                    44.7%


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


  ViT-zero-shot + CNN-fine-tuned:      51.2%
  All 3 zero-shot:                     44.0%


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


In [ ]:
print("\n" + "="*70)
print("COMPLETE RESULTS — OWN DATASET (5-class, excl. metal)")
print("="*70)
print(f"\n{'Method':<45} {'ResNet':>8} {'EffNet':>8} {'ViT':>8}")
print("-"*70)
print(f"{'Baseline (center-crop, zero-shot)':<45} {'43.6%':>8} {'34.6%':>8} {'53.0%':>8}")
print(f"{'Baseline (center-crop, fine-tuned)':<45} {'58.9%':>8} {'49.3%':>8} {'35.0%':>8}")
print(f"{'No-crop (zero-shot)':<45} {rn_zs_nc*100:>7.1f}% {ef_zs_nc*100:>7.1f}% {vt_zs_nc*100:>7.1f}%")
print(f"{'No-crop (fine-tuned)':<45} {rn_ft_nc*100:>7.1f}% {ef_ft_nc*100:>7.1f}% {vt_ft_nc*100:>7.1f}%")
print(f"{'TTA (best checkpoint per model)':<45} {rn_tta*100:>7.1f}% {ef_tta*100:>7.1f}% {vt_tta_zs*100:>7.1f}%")
print(f"\n{'Ensemble':<45} {'Acc':>8}")
print("-"*70)
print(f"{'All 3 fine-tuned':<45} {ens_all_ft*100:>7.1f}%")
print(f"{'ViT-zero-shot + CNN-fine-tuned (smart)':<45} {ens_smart*100:>7.1f}%")
print(f"{'All 3 zero-shot':<45} {ens_all_zs*100:>7.1f}%")


COMPLETE RESULTS — OWN DATASET (5-class, excl. metal)

Method                                          ResNet   EffNet      ViT
----------------------------------------------------------------------
Baseline (center-crop, zero-shot)                43.6%    34.6%    53.0%
Baseline (center-crop, fine-tuned)               58.9%    49.3%    35.0%
No-crop (zero-shot)                              43.8%    37.8%    47.7%
No-crop (fine-tuned)                             54.7%    48.9%    33.2%
TTA (best checkpoint per model)                  60.5%    47.8%    48.1%

Ensemble                                           Acc
----------------------------------------------------------------------
All 3 fine-tuned                                 44.7%
ViT-zero-shot + CNN-fine-tuned (smart)           51.2%
All 3 zero-shot                                  44.0%


In [ ]:
# Cell 1 - Mount + upload
from google.colab import drive
drive.mount('/content/drive')
from google.colab import files
files.upload()  # upload waste-classification-main.zip

Mounted at /content/drive


Saving waste-classification-main.zip to waste-classification-main.zip


<Output stripped — was 69.3MB, likely raw file-upload byte dump>

In [ ]:
# Cell 2 - Extract + basic setup
import zipfile, importlib.util, torch, torch.nn as nn, random, os
import numpy as np
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import balanced_accuracy_score, classification_report
from collections import Counter

with zipfile.ZipFile('/content/waste-classification-main.zip', 'r') as z:
    z.extractall('/content/')

PROJECT_ROOT = '/content/waste-classification-main'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SAVE_DIR = '/content/drive/MyDrive/waste_experiments_5class'
os.makedirs(SAVE_DIR, exist_ok=True)

# *** KEY CHANGE: 5 classes only, NO trash ***
CLASSES = ['cardboard', 'glass', 'metal', 'paper', 'plastic']
CLASS_TO_IDX = {cls: idx for idx, cls in enumerate(CLASSES)}

def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

models_module = load_module("models", f"{PROJECT_ROOT}/src/models.py")
get_model = models_module.get_model
print(f"Device: {device}")

Device: cuda


In [ ]:
# Cell 3 - Install
!pip install albumentations grad-cam kaggle -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 76.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
# Cell 4 - Download datasets
os.environ['KAGGLE_USERNAME'] = 'esmaeilmolapour'
os.environ['KAGGLE_KEY'] = 'YOUR_KAGGLE_KEY_HERE'
!kaggle datasets download -d feyzazkefe/trashnet -p {PROJECT_ROOT}/data/raw/trashnet --unzip -q
!kaggle datasets download -d mostafaabla/garbage-classification -p {PROJECT_ROOT}/data/raw/gd12/ --unzip -q
!kaggle datasets download -d joebeachcapital/realwaste -p {PROJECT_ROOT}/data/raw/realwaste/ --unzip -q
print("Done:", os.listdir(f'{PROJECT_ROOT}/data/raw/'))

Dataset URL: https://www.kaggle.com/datasets/feyzazkefe/trashnet
License(s): unknown
Dataset URL: https://www.kaggle.com/datasets/mostafaabla/garbage-classification
License(s): ODbL-1.0
Dataset URL: https://www.kaggle.com/datasets/joebeachcapital/realwaste
License(s): Attribution 4.0 International (CC BY 4.0)
Done: ['trashnet', 'gd12', 'realwaste']


In [ ]:
!kaggle datasets download -d minhle13/trashbox \
    -p {PROJECT_ROOT}/data/raw/trashbox/ --unzip -q

import os
for root, dirs, files in os.walk(f'{PROJECT_ROOT}/data/raw/trashbox'):
    if files:
        print(root, '->', len(files), 'files')
    else:
        print(root, '->', dirs[:5])

Dataset URL: https://www.kaggle.com/datasets/minhle13/trashbox
License(s): unknown
/content/waste-classification-main/data/raw/trashbox -> ['TrashBox_train_set']
/content/waste-classification-main/data/raw/trashbox/TrashBox_train_set -> ['e-waste', 'metal', 'glass', 'plastic', 'cardboard']
/content/waste-classification-main/data/raw/trashbox/TrashBox_train_set/e-waste -> 2406 files
/content/waste-classification-main/data/raw/trashbox/TrashBox_train_set/metal -> 2068 files
/content/waste-classification-main/data/raw/trashbox/TrashBox_train_set/glass -> 2022 files
/content/waste-classification-main/data/raw/trashbox/TrashBox_train_set/plastic -> 2135 files
/content/waste-classification-main/data/raw/trashbox/TrashBox_train_set/cardboard -> 1930 files
/content/waste-classification-main/data/raw/trashbox/TrashBox_train_set/medical -> 1565 files
/content/waste-classification-main/data/raw/trashbox/TrashBox_train_set/paper -> 2156 files


In [ ]:
import os
trashbox_root = f'{PROJECT_ROOT}/data/raw/trashbox'
for root, dirs, files_list in os.walk(trashbox_root):
    depth = root.replace(trashbox_root, '').count(os.sep)
    if depth <= 3:
        print(root, '->', dirs[:5] if dirs else f'{len(files_list)} files')

/content/waste-classification-main/data/raw/trashbox -> ['TrashBox_train_set']
/content/waste-classification-main/data/raw/trashbox/TrashBox_train_set -> ['e-waste', 'metal', 'glass', 'plastic', 'cardboard']
/content/waste-classification-main/data/raw/trashbox/TrashBox_train_set/e-waste -> 2406 files
/content/waste-classification-main/data/raw/trashbox/TrashBox_train_set/metal -> 2068 files
/content/waste-classification-main/data/raw/trashbox/TrashBox_train_set/glass -> 2022 files
/content/waste-classification-main/data/raw/trashbox/TrashBox_train_set/plastic -> 2135 files
/content/waste-classification-main/data/raw/trashbox/TrashBox_train_set/cardboard -> 1930 files
/content/waste-classification-main/data/raw/trashbox/TrashBox_train_set/medical -> 1565 files
/content/waste-classification-main/data/raw/trashbox/TrashBox_train_set/paper -> 2156 files


In [ ]:
import matplotlib.pyplot as plt
import cv2, random
from pathlib import Path

TRASHBOX_ROOT = f'{PROJECT_ROOT}/data/raw/trashbox/TrashBox_train_set'

for cls in ['cardboard', 'glass', 'metal', 'paper', 'plastic']:
    class_dir = Path(TRASHBOX_ROOT) / cls
    all_files = [f for f in class_dir.iterdir()
                 if f.suffix.lower() in ['.jpg','.jpeg','.png']]
    sample = random.sample(all_files, min(6, len(all_files)))

    fig, axes = plt.subplots(1, len(sample), figsize=(18, 3))
    for ax, f in zip(axes, sample):
        img = cv2.imread(str(f))
        if img is not None:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            ax.imshow(img)
        ax.axis('off')
    plt.suptitle(f'TrashBox — {cls} ({len(all_files)} images)', fontsize=12)
    plt.tight_layout()
    plt.savefig(f'/content/trashbox_{cls}.png', dpi=80, bbox_inches='tight')
    plt.show()
    print(f'{cls}: {len(all_files)} images')

cardboard: 1930 images


glass: 2022 images


metal: 2068 images


paper: 2156 images


plastic: 2135 images


In [ ]:
# Quick manual audit - look at 15-20 per class instead of 6
for cls in ['cardboard', 'glass', 'metal', 'paper', 'plastic']:
    class_dir = Path(TRASHBOX_ROOT) / cls
    all_files = [f for f in class_dir.iterdir() if f.suffix.lower() in ['.jpg','.jpeg','.png']]
    sample = random.sample(all_files, min(15, len(all_files)))
    fig, axes = plt.subplots(3, 5, figsize=(18, 10))
    for ax, f in zip(axes.flat, sample):
        img = cv2.imread(str(f))
        if img is not None:
            ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.axis('off')
    plt.suptitle(f'{cls} — larger sample')
    plt.savefig(f'/content/trashbox_big_{cls}.png', dpi=70, bbox_inches='tight')
    plt.show()

In [ ]:
for var in ['PROJECT_ROOT', 'device', 'CLASSES', 'get_model', 'SAVE_DIR']:
    print(var, ':', 'OK' if var in globals() else 'MISSING')

PROJECT_ROOT : OK
device : OK
CLASSES : OK
get_model : OK
SAVE_DIR : OK


In [ ]:
print(CLASSES)
print(PROJECT_ROOT)
print(SAVE_DIR)
print(os.listdir(f'{PROJECT_ROOT}/data/raw/'))

['cardboard', 'glass', 'metal', 'paper', 'plastic']
/content/waste-classification-main
/content/drive/MyDrive/waste_experiments_5class
['trashnet', 'gd12', 'trashbox', 'realwaste']


In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2
import torch.nn.functional as F

# Resize-pad instead of center-crop — preserves whole object, doesn't cut off-center items
def resize_pad_transform(train=False):
    tfms = [
        A.LongestMaxSize(max_size=224),
        A.PadIfNeeded(min_height=224, min_width=224, border_mode=cv2.BORDER_CONSTANT, value=(255,255,255)),
    ]
    if train:
        tfms += [
            A.HorizontalFlip(p=0.5),
            A.Rotate(limit=15, p=0.4),
            A.RandomBrightnessContrast(p=0.4),
        ]
    tfms += [
        A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
        ToTensorV2(),
    ]
    return A.Compose(tfms)

train_transform = resize_pad_transform(train=True)
eval_transform  = resize_pad_transform(train=False)

class WasteDataset(Dataset):
    def __init__(self, samples, transform):
        self.samples = samples
        self.transform = transform
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        return self.transform(image=image)['image'], label

print("Transforms + dataset class ready (resize-pad, no center-crop).")

Transforms + dataset class ready (resize-pad, no center-crop).


/tmp/ipykernel_854/3789802080.py:10: UserWarning: Argument(s) 'value' are not valid for transform PadIfNeeded
  A.PadIfNeeded(min_height=224, min_width=224, border_mode=cv2.BORDER_CONSTANT, value=(255,255,255)),


In [ ]:
def build_trashnet(root):
    samples = []
    for cls in CLASSES:
        class_dir = Path(root) / cls
        if not class_dir.exists(): continue
        for img in class_dir.iterdir():
            if img.suffix.lower() in ['.jpg','.jpeg','.png']:
                samples.append((str(img), CLASS_TO_IDX[cls]))
    return samples

GD_MAP_5CLASS = {
    'cardboard':'cardboard','brown-glass':'glass','green-glass':'glass','white-glass':'glass',
    'metal':'metal','paper':'paper','plastic':'plastic',
}
def build_gd(root):
    samples = []
    for folder, unified in GD_MAP_5CLASS.items():
        class_dir = Path(root) / folder
        if not class_dir.exists(): continue
        idx = CLASS_TO_IDX[unified]
        for img in class_dir.iterdir():
            if img.suffix.lower() in ['.jpg','.jpeg','.png']:
                samples.append((str(img), idx))
    return samples

REALWASTE_MAP_5CLASS = {
    'Cardboard':'cardboard','Glass':'glass','Metal':'metal','Paper':'paper','Plastic':'plastic',
}
def build_realwaste(root):
    samples = []
    for folder, unified in REALWASTE_MAP_5CLASS.items():
        class_dir = Path(root) / folder
        if not class_dir.exists():
            print(f"  Warning: {folder} not found"); continue
        idx = CLASS_TO_IDX[unified]
        for img in class_dir.iterdir():
            if img.suffix.lower() in ['.jpg','.jpeg','.png']:
                samples.append((str(img), idx))
    return samples

trashnet = build_trashnet(f'{PROJECT_ROOT}/data/raw/trashnet/dataset-resized')
gd = build_gd(f'{PROJECT_ROOT}/data/raw/gd12/garbage_classification')
realwaste_test = build_realwaste(f'{PROJECT_ROOT}/data/raw/realwaste/realwaste-main/RealWaste')

print(f"TrashNet: {len(trashnet)} | GD12: {len(gd)} | RealWaste (100% test): {len(realwaste_test)}")

TrashNet: 2390 | GD12: 5586 | RealWaste (100% test): 3092


In [ ]:
def resize_pad_transform(train=False):
    tfms = [
        A.LongestMaxSize(max_size=224),
        A.PadIfNeeded(min_height=224, min_width=224, border_mode=cv2.BORDER_CONSTANT, fill=255),
    ]
    if train:
        tfms += [
            A.HorizontalFlip(p=0.5),
            A.Rotate(limit=15, p=0.4),
            A.RandomBrightnessContrast(p=0.4),
        ]
    tfms += [
        A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
        ToTensorV2(),
    ]
    return A.Compose(tfms)

train_transform = resize_pad_transform(train=True)
eval_transform  = resize_pad_transform(train=False)
print("Transform fixed — white padding confirmed.")

Transform fixed — white padding confirmed.


In [ ]:
import numpy as np
import random as pyrandom

def extract_foreground_mask(image):
    """
    Rough foreground extraction using GrabCut, assuming object is roughly
    centered (true for TrashNet/GD12 studio photos). Not perfect, but doesn't
    need to be — it just needs to separate 'object' from 'clean background'
    well enough to recompose onto messy backgrounds.
    """
    h, w = image.shape[:2]
    mask = np.zeros((h, w), np.uint8)
    bgd_model = np.zeros((1, 65), np.float64)
    fgd_model = np.zeros((1, 65), np.float64)

    # Assume object occupies the central ~80% of the frame (matches TrashNet/GD style)
    rect = (int(w*0.05), int(h*0.05), int(w*0.9), int(h*0.9))
    try:
        cv2.grabCut(image, mask, rect, bgd_model, fgd_model, 3, cv2.GC_INIT_WITH_RECT)
        fg_mask = np.where((mask==2)|(mask==0), 0, 1).astype('uint8')
    except:
        fg_mask = np.ones((h, w), dtype='uint8')  # fallback: keep whole image
    return fg_mask

def make_synthetic_background(h, w):
    """Generate a varied synthetic background: solid color, gradient, or noise-textured,
    standing in for real-world clutter (floors, grass, concrete tones) without needing
    an external background image dataset."""
    choice = pyrandom.random()
    if choice < 0.4:
        # Textured solid color (concrete/floor-like grays and browns)
        base_color = pyrandom.choice([
            (120,120,120), (90,80,70), (140,130,110), (60,90,60), (100,100,90)
        ])
        bg = np.full((h, w, 3), base_color, dtype=np.uint8)
        noise = np.random.randint(-25, 25, (h, w, 3), dtype=np.int16)
        bg = np.clip(bg.astype(np.int16) + noise, 0, 255).astype(np.uint8)
    elif choice < 0.7:
        # Gradient background
        c1 = np.array(pyrandom.choice([(80,70,60),(100,110,90),(70,70,80)]))
        c2 = np.array(pyrandom.choice([(150,140,120),(120,130,100),(110,110,130)]))
        bg = np.linspace(c1, c2, h).astype(np.uint8)
        bg = np.tile(bg[:, None, :], (1, w, 1))
    else:
        # Random coarse noise texture (stand-in for cluttered/dirty surfaces)
        bg = np.random.randint(40, 200, (h, w, 3), dtype=np.uint8)
        bg = cv2.GaussianBlur(bg, (15, 15), 0)
    return bg

class BackgroundReplaceDataset(Dataset):
    """
    Wraps the base transform pipeline but, with probability p, replaces the
    background of clean-domain training images with a synthetic messy
    background before applying the standard transform. This directly targets
    the model's tendency to use background as a shortcut cue.
    """
    def __init__(self, samples, transform, bg_replace_prob=0.5):
        self.samples = samples
        self.transform = transform
        self.p = bg_replace_prob

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if pyrandom.random() < self.p:
            h, w = image.shape[:2]
            fg_mask = extract_foreground_mask(image)
            bg = make_synthetic_background(h, w)
            fg_mask_3ch = np.stack([fg_mask]*3, axis=-1)
            image = np.where(fg_mask_3ch == 1, image, bg).astype(np.uint8)

        return self.transform(image=image)['image'], label

print("Background-replacement augmentation ready.")

Background-replacement augmentation ready.


In [ ]:
import matplotlib.pyplot as plt

bg_replace_preview = BackgroundReplaceDataset(trashnet[:20], eval_transform, bg_replace_prob=1.0)

fig, axes = plt.subplots(2, 5, figsize=(18, 7))
for i, ax in enumerate(axes.flat):
    img_path, label = trashnet[i]
    orig = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    h, w = orig.shape[:2]
    fg_mask = extract_foreground_mask(orig)
    bg = make_synthetic_background(h, w)
    composited = np.where(np.stack([fg_mask]*3, -1)==1, orig, bg).astype(np.uint8)
    ax.imshow(composited)
    ax.set_title(CLASSES[label], fontsize=9)
    ax.axis('off')
plt.suptitle('Background-replacement preview')
plt.tight_layout()
plt.savefig('/content/bg_replace_preview.png', dpi=80)
plt.show()

In [ ]:
def border_background_replace(image, border_frac=0.18):
    """
    Replaces only the OUTER BORDER of the image with a synthetic messy
    background, leaving the central region (where the object reliably sits
    in TrashNet/GD12 studio photos) completely untouched. Much safer than
    pixel-level segmentation — no risk of damaging the object itself.
    """
    h, w = image.shape[:2]
    bg = make_synthetic_background(h, w)

    # Build a soft mask: 1.0 in center, fading to 0.0 at the border
    yy, xx = np.mgrid[0:h, 0:w]
    cx, cy = w/2, h/2
    # distance from center, normalized so border_frac controls fade start
    dist_x = np.abs(xx - cx) / (w/2)
    dist_y = np.abs(yy - cy) / (h/2)
    dist = np.maximum(dist_x, dist_y)  # 0 at center, 1 at edge

    fade_start = 1.0 - border_frac
    mask = np.clip((fade_start - dist) / border_frac + 1.0, 0, 1)
    mask_3ch = np.stack([mask]*3, axis=-1)

    composited = (image.astype(np.float32) * mask_3ch +
                  bg.astype(np.float32) * (1 - mask_3ch))
    return composited.astype(np.uint8)

class BorderBackgroundDataset(Dataset):
    def __init__(self, samples, transform, bg_replace_prob=0.5):
        self.samples = samples
        self.transform = transform
        self.p = bg_replace_prob
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        if pyrandom.random() < self.p:
            image = border_background_replace(image)
        return self.transform(image=image)['image'], label

print("Border-only background replacement ready (safer, no segmentation risk).")

Border-only background replacement ready (safer, no segmentation risk).


In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 7))
for i, ax in enumerate(axes.flat):
    img_path, label = trashnet[i]
    orig = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    composited = border_background_replace(orig)
    ax.imshow(composited)
    ax.set_title(CLASSES[label], fontsize=9)
    ax.axis('off')
plt.suptitle('Border-background-replacement preview (safer version)')
plt.tight_layout()
plt.savefig('/content/border_bg_preview.png', dpi=80)
plt.show()

In [ ]:
# Combine TrashNet + GD12 (5 classes), RealWaste stays 100% untouched as test
all_train_val = trashnet + gd
random.seed(42)
random.shuffle(all_train_val)

n = len(all_train_val)
n_train = int(0.85 * n)
train_samples = all_train_val[:n_train]
val_samples   = all_train_val[n_train:]

print(f"Train: {len(train_samples)} | Val: {len(val_samples)} | RealWaste test (untouched): {len(realwaste_test)}")

counts = Counter([l for _, l in train_samples])
print("Class distribution:")
for i, cls in enumerate(CLASSES):
    print(f"  {cls:10s}: {counts[i]}")

cc = np.array([counts[i] for i in range(5)], dtype=float)
weights = cc.sum() / (5 * cc)
print("Class weights:", {CLASSES[i]: round(weights[i],3) for i in range(5)})

# Label smoothing + weighted loss — reduces overconfidence (the root cause we diagnosed)
criterion = nn.CrossEntropyLoss(weight=torch.FloatTensor(weights).to(device), label_smoothing=0.1)

train_dataset = BorderBackgroundDataset(train_samples, train_transform, bg_replace_prob=0.5)
val_dataset   = WasteDataset(val_samples, eval_transform)
test_dataset  = WasteDataset(realwaste_test, eval_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)
print("DataLoaders ready.")

Train: 6779 | Val: 1197 | RealWaste test (untouched): 3092
Class distribution:
  cardboard : 1113
  glass     : 2132
  metal     : 1001
  paper     : 1395
  plastic   : 1138
Class weights: {'cardboard': np.float64(1.218), 'glass': np.float64(0.636), 'metal': np.float64(1.354), 'paper': np.float64(0.972), 'plastic': np.float64(1.191)}
DataLoaders ready.


In [ ]:
def freeze_backbone_partial(model, model_name):
    """Freeze early layers, only fine-tune later layers + head.
    Preserves generic ImageNet features that transfer better OOD."""
    if model_name == 'resnet50':
        for name, param in model.named_parameters():
            if not (name.startswith('layer3') or name.startswith('layer4') or name.startswith('fc')):
                param.requires_grad = False
    elif model_name == 'efficientnet':
        # freeze first ~60% of blocks, keep the rest + classifier trainable
        params = list(model.named_parameters())
        n_freeze = int(len(params) * 0.6)
        for name, param in params[:n_freeze]:
            param.requires_grad = False
    elif model_name == 'vit':
        params = list(model.named_parameters())
        n_freeze = int(len(params) * 0.6)
        for name, param in params[:n_freeze]:
            param.requires_grad = False
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"  Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")
    return model

def train_5class(model_name, num_epochs=20, freeze_backbone=False, weight_decay=2e-2):
    print(f"\n{'='*60}")
    print(f"Training {model_name} — 5-class, border-bg-aug, "
          f"{'FROZEN backbone' if freeze_backbone else 'full fine-tune'}")
    print('='*60)

    model = get_model(model_name, num_classes=5).to(device)
    if freeze_backbone:
        model = freeze_backbone_partial(model, model_name)

    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=1e-4, weight_decay=weight_decay
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

    best_val_bal_acc = 0.0  # *** select checkpoint by balanced accuracy, not loss ***

    for epoch in range(1, num_epochs+1):
        model.train()
        tp, tl = [], []
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            out = model(images)
            loss = criterion(out, labels)
            loss.backward()
            optimizer.step()
            tp.extend(out.argmax(1).detach().cpu().numpy())
            tl.extend(labels.cpu().numpy())
        train_acc = balanced_accuracy_score(tl, tp)

        model.eval()
        vl, vp, vlabels = 0, [], []
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                out = model(images)
                vl += criterion(out, labels).item()
                vp.extend(out.argmax(1).cpu().numpy())
                vlabels.extend(labels.cpu().numpy())
        val_bal_acc = balanced_accuracy_score(vlabels, vp)
        avg_vl = vl / len(val_loader)
        scheduler.step()

        print(f"Epoch {epoch}/{num_epochs} | Train: {train_acc:.4f} | Val BalAcc: {val_bal_acc:.4f} | Loss: {avg_vl:.4f}")

        if val_bal_acc > best_val_bal_acc:
            best_val_bal_acc = val_bal_acc
            suffix = 'frozen' if freeze_backbone else 'full'
            torch.save({'model_state_dict': model.state_dict()},
                       f'{SAVE_DIR}/{model_name}_5class_{suffix}_best.pth')
            print(f"  ✓ saved (val bal acc: {val_bal_acc:.4f})")

    # Zero-shot test on RealWaste — 100% untouched, per Sebastian
    model.eval(); op, ol = [], []
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            op.extend(model(images).argmax(1).cpu().numpy())
            ol.extend(labels.numpy())
    ood_acc = balanced_accuracy_score(ol, op)
    print(f"\n>>> Zero-shot RealWaste OOD (5-class): {ood_acc*100:.1f}% <<<")
    print(classification_report(ol, op, target_names=CLASSES, zero_division=0))
    return ood_acc

# Start with ResNet-50, full fine-tune, to get a baseline for the 5-class + all fixes combined
resnet_5class_ood = train_5class('resnet50', num_epochs=20, freeze_backbone=False)


Training resnet50 — 5-class, border-bg-aug, full fine-tune


model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]

Epoch 1/20 | Train: 0.6568 | Val BalAcc: 0.8644 | Loss: 0.7612
  ✓ saved (val bal acc: 0.8644)
Epoch 2/20 | Train: 0.8829 | Val BalAcc: 0.9263 | Loss: 0.5705
  ✓ saved (val bal acc: 0.9263)
Epoch 3/20 | Train: 0.9341 | Val BalAcc: 0.9482 | Loss: 0.5320
  ✓ saved (val bal acc: 0.9482)
Epoch 4/20 | Train: 0.9624 | Val BalAcc: 0.9602 | Loss: 0.5112
  ✓ saved (val bal acc: 0.9602)
Epoch 5/20 | Train: 0.9678 | Val BalAcc: 0.9671 | Loss: 0.4986
  ✓ saved (val bal acc: 0.9671)
Epoch 6/20 | Train: 0.9814 | Val BalAcc: 0.9697 | Loss: 0.4890
  ✓ saved (val bal acc: 0.9697)
Epoch 7/20 | Train: 0.9843 | Val BalAcc: 0.9780 | Loss: 0.4779
  ✓ saved (val bal acc: 0.9780)
Epoch 8/20 | Train: 0.9891 | Val BalAcc: 0.9783 | Loss: 0.4726
  ✓ saved (val bal acc: 0.9783)
Epoch 9/20 | Train: 0.9904 | Val BalAcc: 0.9773 | Loss: 0.4701
Epoch 10/20 | Train: 0.9926 | Val BalAcc: 0.9793 | Loss: 0.4668
  ✓ saved (val bal acc: 0.9793)
Epoch 11/20 | Train: 0.9933 | Val BalAcc: 0.9795 | Loss: 0.4683
  ✓ saved (val ba

In [ ]:
# Control run: identical setup, but NO border-background augmentation
train_dataset_control = WasteDataset(train_samples, train_transform)  # plain transform, no bg replace
train_loader_control = DataLoader(train_dataset_control, batch_size=32, shuffle=True, num_workers=2)

def train_5class_control(model_name, num_epochs=20, weight_decay=2e-2):
    print(f"\n{'='*60}")
    print(f"CONTROL: Training {model_name} — 5-class, NO bg augmentation, full fine-tune")
    print('='*60)

    model = get_model(model_name, num_classes=5).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

    best_val_bal_acc = 0.0

    for epoch in range(1, num_epochs+1):
        model.train()
        tp, tl = [], []
        for images, labels in train_loader_control:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            out = model(images)
            loss = criterion(out, labels)
            loss.backward()
            optimizer.step()
            tp.extend(out.argmax(1).detach().cpu().numpy())
            tl.extend(labels.cpu().numpy())
        train_acc = balanced_accuracy_score(tl, tp)

        model.eval()
        vl, vp, vlabels = 0, [], []
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                out = model(images)
                vl += criterion(out, labels).item()
                vp.extend(out.argmax(1).cpu().numpy())
                vlabels.extend(labels.cpu().numpy())
        val_bal_acc = balanced_accuracy_score(vlabels, vp)
        avg_vl = vl / len(val_loader)
        scheduler.step()

        print(f"Epoch {epoch}/{num_epochs} | Train: {train_acc:.4f} | Val BalAcc: {val_bal_acc:.4f} | Loss: {avg_vl:.4f}")

        if val_bal_acc > best_val_bal_acc:
            best_val_bal_acc = val_bal_acc
            torch.save({'model_state_dict': model.state_dict()},
                       f'{SAVE_DIR}/{model_name}_5class_control_best.pth')
            print(f"  ✓ saved (val bal acc: {val_bal_acc:.4f})")

    model.eval(); op, ol = [], []
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            op.extend(model(images).argmax(1).cpu().numpy())
            ol.extend(labels.numpy())
    ood_acc = balanced_accuracy_score(ol, op)
    print(f"\n>>> CONTROL Zero-shot RealWaste OOD (5-class, no bg-aug): {ood_acc*100:.1f}% <<<")
    print(classification_report(ol, op, target_names=CLASSES, zero_division=0))
    return ood_acc

resnet_control_ood = train_5class_control('resnet50', num_epochs=20)

print("\n" + "="*60)
print("ABLATION COMPARISON — ResNet-50, 5-class")
print("="*60)
print(f"  WITH border-bg augmentation:    58.0%")
print(f"  WITHOUT (control):               {resnet_control_ood*100:.1f}%")


CONTROL: Training resnet50 — 5-class, NO bg augmentation, full fine-tune


Epoch 1/20 | Train: 0.6587 | Val BalAcc: 0.8760 | Loss: 0.7337
  ✓ saved (val bal acc: 0.8760)
Epoch 2/20 | Train: 0.8977 | Val BalAcc: 0.9455 | Loss: 0.5517
  ✓ saved (val bal acc: 0.9455)
Epoch 3/20 | Train: 0.9463 | Val BalAcc: 0.9571 | Loss: 0.5186
  ✓ saved (val bal acc: 0.9571)
Epoch 4/20 | Train: 0.9672 | Val BalAcc: 0.9698 | Loss: 0.4935
  ✓ saved (val bal acc: 0.9698)
Epoch 5/20 | Train: 0.9722 | Val BalAcc: 0.9718 | Loss: 0.4812
  ✓ saved (val bal acc: 0.9718)
Epoch 6/20 | Train: 0.9832 | Val BalAcc: 0.9728 | Loss: 0.4776
  ✓ saved (val bal acc: 0.9728)
Epoch 7/20 | Train: 0.9859 | Val BalAcc: 0.9784 | Loss: 0.4667
  ✓ saved (val bal acc: 0.9784)
Epoch 8/20 | Train: 0.9893 | Val BalAcc: 0.9792 | Loss: 0.4710
  ✓ saved (val bal acc: 0.9792)
Epoch 9/20 | Train: 0.9917 | Val BalAcc: 0.9787 | Loss: 0.4691
Epoch 10/20 | Train: 0.9928 | Val BalAcc: 0.9792 | Loss: 0.4639
Epoch 11/20 | Train: 0.9934 | Val BalAcc: 0.9752 | Loss: 0.4647
Epoch 12/20 | Train: 0.9938 | Val BalAcc: 0.9798 

In [ ]:
# EfficientNet — with border-bg augmentation (the winning config)
effnet_5class_ood = train_5class('efficientnet', num_epochs=20, freeze_backbone=False)


Training efficientnet — 5-class, border-bg-aug, full fine-tune


model.safetensors:   0%|          | 0.00/49.3M [00:00<?, ?B/s]

Epoch 1/20 | Train: 0.7007 | Val BalAcc: 0.8622 | Loss: 0.7880
  ✓ saved (val bal acc: 0.8622)
Epoch 2/20 | Train: 0.9001 | Val BalAcc: 0.9179 | Loss: 0.6495
  ✓ saved (val bal acc: 0.9179)
Epoch 3/20 | Train: 0.9461 | Val BalAcc: 0.9454 | Loss: 0.5820
  ✓ saved (val bal acc: 0.9454)
Epoch 4/20 | Train: 0.9745 | Val BalAcc: 0.9573 | Loss: 0.5587
  ✓ saved (val bal acc: 0.9573)
Epoch 5/20 | Train: 0.9832 | Val BalAcc: 0.9681 | Loss: 0.5361
  ✓ saved (val bal acc: 0.9681)
Epoch 6/20 | Train: 0.9901 | Val BalAcc: 0.9632 | Loss: 0.5216
Epoch 7/20 | Train: 0.9929 | Val BalAcc: 0.9711 | Loss: 0.5135
  ✓ saved (val bal acc: 0.9711)
Epoch 8/20 | Train: 0.9952 | Val BalAcc: 0.9722 | Loss: 0.5055
  ✓ saved (val bal acc: 0.9722)
Epoch 9/20 | Train: 0.9965 | Val BalAcc: 0.9775 | Loss: 0.5016
  ✓ saved (val bal acc: 0.9775)
Epoch 10/20 | Train: 0.9972 | Val BalAcc: 0.9752 | Loss: 0.4977
Epoch 11/20 | Train: 0.9972 | Val BalAcc: 0.9755 | Loss: 0.4904
Epoch 12/20 | Train: 0.9978 | Val BalAcc: 0.9755 

In [ ]:
vit_5class_ood = train_5class('vit', num_epochs=20, freeze_backbone=False)

print("\n" + "="*60)
print("FULL 5-CLASS + BORDER-BG-AUG COMPARISON — ALL 3 MODELS")
print("="*60)
print(f"  ResNet-50:     {resnet_5class_ood*100:.1f}%")
print(f"  EfficientNet:  {effnet_5class_ood*100:.1f}%")
print(f"  ViT-Small:     {vit_5class_ood*100:.1f}%")


Training vit — 5-class, border-bg-aug, full fine-tune


model.safetensors:   0%|          | 0.00/88.2M [00:00<?, ?B/s]

Epoch 1/20 | Train: 0.9092 | Val BalAcc: 0.9712 | Loss: 0.5006
  ✓ saved (val bal acc: 0.9712)
Epoch 2/20 | Train: 0.9826 | Val BalAcc: 0.9623 | Loss: 0.4909
Epoch 3/20 | Train: 0.9928 | Val BalAcc: 0.9764 | Loss: 0.4666
  ✓ saved (val bal acc: 0.9764)
Epoch 4/20 | Train: 0.9897 | Val BalAcc: 0.9633 | Loss: 0.5004
Epoch 5/20 | Train: 0.9878 | Val BalAcc: 0.9724 | Loss: 0.4710
Epoch 6/20 | Train: 0.9882 | Val BalAcc: 0.9771 | Loss: 0.4683
  ✓ saved (val bal acc: 0.9771)
Epoch 7/20 | Train: 0.9915 | Val BalAcc: 0.9778 | Loss: 0.4711
  ✓ saved (val bal acc: 0.9778)
Epoch 8/20 | Train: 0.9927 | Val BalAcc: 0.9689 | Loss: 0.4849
Epoch 9/20 | Train: 0.9970 | Val BalAcc: 0.9793 | Loss: 0.4597
  ✓ saved (val bal acc: 0.9793)
Epoch 10/20 | Train: 0.9988 | Val BalAcc: 0.9781 | Loss: 0.4573
Epoch 11/20 | Train: 0.9993 | Val BalAcc: 0.9814 | Loss: 0.4518
  ✓ saved (val bal acc: 0.9814)
Epoch 12/20 | Train: 0.9996 | Val BalAcc: 0.9804 | Loss: 0.4554
Epoch 13/20 | Train: 0.9992 | Val BalAcc: 0.9804 

In [ ]:
import torch.nn.functional as F

def get_probs_and_labels(model, loader, device):
    """Returns softmax probabilities, predicted labels, true labels, and max-confidence per sample."""
    all_probs, all_preds, all_labels = [], [], []
    model.eval()
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            logits = model(images)
            probs = F.softmax(logits, dim=1)
            all_probs.append(probs.cpu().numpy())
            all_preds.extend(probs.argmax(1).cpu().numpy())
            all_labels.extend(labels.numpy())
    all_probs = np.concatenate(all_probs, axis=0)
    return all_probs, np.array(all_preds), np.array(all_labels)

def compute_ece(confidences, correct, n_bins=10):
    """Expected Calibration Error — how well confidence matches actual accuracy."""
    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    bin_details = []
    for i in range(n_bins):
        lo, hi = bin_boundaries[i], bin_boundaries[i+1]
        in_bin = (confidences > lo) & (confidences <= hi)
        prop_in_bin = in_bin.mean()
        if prop_in_bin > 0:
            acc_in_bin = correct[in_bin].mean()
            conf_in_bin = confidences[in_bin].mean()
            ece += np.abs(acc_in_bin - conf_in_bin) * prop_in_bin
            bin_details.append((lo, hi, prop_in_bin, acc_in_bin, conf_in_bin))
    return ece, bin_details

def compute_ood_auroc(id_confidences, ood_confidences):
    """AUROC for distinguishing in-distribution vs OOD using max softmax confidence.
    Higher AUROC = model's confidence is a better signal for 'this is OOD, I'm unsure'."""
    from sklearn.metrics import roc_auc_score
    labels = np.concatenate([np.ones_like(id_confidences), np.zeros_like(ood_confidences)])
    scores = np.concatenate([id_confidences, ood_confidences])
    return roc_auc_score(labels, scores)

def calibration_report(model_name, checkpoint_path, val_loader, ood_loader, device):
    print(f"\n{'='*60}")
    print(f"CALIBRATION ANALYSIS: {model_name}")
    print('='*60)

    model = get_model(model_name, num_classes=5)
    ckpt = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    model = model.to(device)

    # In-distribution (val set) calibration
    id_probs, id_preds, id_labels = get_probs_and_labels(model, val_loader, device)
    id_conf = id_probs.max(axis=1)
    id_correct = (id_preds == id_labels).astype(float)
    id_ece, id_bins = compute_ece(id_conf, id_correct)

    # OOD (RealWaste) calibration
    ood_probs, ood_preds, ood_labels = get_probs_and_labels(model, ood_loader, device)
    ood_conf = ood_probs.max(axis=1)
    ood_correct = (ood_preds == ood_labels).astype(float)
    ood_ece, ood_bins = compute_ece(ood_conf, ood_correct)

    # OOD detection AUROC (can the model's own confidence flag "I'm out of my depth"?)
    auroc = compute_ood_auroc(id_conf, ood_conf)

    print(f"\n  In-distribution (val):")
    print(f"    Mean confidence: {id_conf.mean():.3f} | Accuracy: {id_correct.mean():.3f} | ECE: {id_ece:.3f}")
    print(f"\n  Out-of-distribution (RealWaste):")
    print(f"    Mean confidence: {ood_conf.mean():.3f} | Accuracy: {ood_correct.mean():.3f} | ECE: {ood_ece:.3f}")
    print(f"\n  Overconfidence gap: {ood_conf.mean() - ood_correct.mean():+.3f}")
    print(f"  (positive = model is more confident than it is accurate = overconfident)")
    print(f"\n  OOD Detection AUROC: {auroc:.3f}")
    print(f"  (0.5 = confidence is useless for detecting OOD, 1.0 = perfect separation)")

    return {'id_ece': id_ece, 'ood_ece': ood_ece, 'auroc': auroc,
            'id_conf': id_conf.mean(), 'ood_conf': ood_conf.mean(),
            'ood_acc': ood_correct.mean()}

# Run on your 3 border-bg-aug checkpoints
results = {}
results['resnet50'] = calibration_report('resnet50',
    f'{SAVE_DIR}/resnet50_5class_full_best.pth', val_loader, test_loader, device)
results['efficientnet'] = calibration_report('efficientnet',
    f'{SAVE_DIR}/efficientnet_5class_full_best.pth', val_loader, test_loader, device)
results['vit'] = calibration_report('vit',
    f'{SAVE_DIR}/vit_5class_full_best.pth', val_loader, test_loader, device)

print("\n" + "="*60)
print("SUMMARY — CALIBRATION ACROSS ALL 3 MODELS")
print("="*60)
for name, r in results.items():
    print(f"  {name:12s} | OOD ECE: {r['ood_ece']:.3f} | Overconf gap: {r['ood_conf']-r['ood_acc']:+.3f} | AUROC: {r['auroc']:.3f}")


CALIBRATION ANALYSIS: resnet50

  In-distribution (val):
    Mean confidence: 0.889 | Accuracy: 0.982 | ECE: 0.097

  Out-of-distribution (RealWaste):
    Mean confidence: 0.733 | Accuracy: 0.602 | ECE: 0.132

  Overconfidence gap: +0.131
  (positive = model is more confident than it is accurate = overconfident)

  OOD Detection AUROC: 0.757
  (0.5 = confidence is useless for detecting OOD, 1.0 = perfect separation)

CALIBRATION ANALYSIS: efficientnet

  In-distribution (val):
    Mean confidence: 0.870 | Accuracy: 0.979 | ECE: 0.110

  Out-of-distribution (RealWaste):
    Mean confidence: 0.601 | Accuracy: 0.554 | ECE: 0.047

  Overconfidence gap: +0.046
  (positive = model is more confident than it is accurate = overconfident)

  OOD Detection AUROC: 0.885
  (0.5 = confidence is useless for detecting OOD, 1.0 = perfect separation)

CALIBRATION ANALYSIS: vit

  In-distribution (val):
    Mean confidence: 0.898 | Accuracy: 0.985 | ECE: 0.089

  Out-of-distribution (RealWaste):
    Mea

In [ ]:
import os
own_root = f'{PROJECT_ROOT}/data/raw/own_dataset/sorted'
print("Exists:", os.path.exists(own_root))
if os.path.exists(own_root):
    print(os.listdir(own_root))

Exists: False


In [ ]:
def build_own_5class(root):
    samples = []
    for cls in CLASSES:  # cardboard, glass, metal, paper, plastic - but we'll filter metal out below
        class_dir = Path(root) / cls
        if not class_dir.exists():
            continue
        for img in class_dir.iterdir():
            if img.suffix.lower() in ['.jpg','.jpeg','.png','.jpeg']:
                samples.append((str(img), CLASS_TO_IDX[cls]))
    return samples

own_samples_all = build_own_5class(f'{PROJECT_ROOT}/data/raw/own_dataset/sorted')

# Exclude metal (documented composition drift - small objects on grass, same issue as before)
own_samples_4class_eval = [(p, l) for p, l in own_samples_all if CLASSES[l] != 'metal']

print(f"Own dataset (5-class list, metal excluded for eval): {len(own_samples_4class_eval)}")
counts = Counter([l for _, l in own_samples_4class_eval])
for i, cls in enumerate(CLASSES):
    if cls != 'metal':
        print(f"  {cls:10s}: {counts[i]}")

own_dataset_eval = WasteDataset(own_samples_4class_eval, eval_transform)
own_loader_eval = DataLoader(own_dataset_eval, batch_size=32, shuffle=False, num_workers=2)

Own dataset (5-class list, metal excluded for eval): 142
  cardboard : 27
  glass     : 18
  paper     : 41
  plastic   : 56


In [ ]:
def evaluate_own_dataset(model_name, checkpoint_path, loader, device):
    model = get_model(model_name, num_classes=5)
    ckpt = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    model = model.to(device).eval()

    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            all_preds.extend(outputs.argmax(1).cpu().numpy())
            all_labels.extend(labels.numpy())

    acc = balanced_accuracy_score(all_labels, all_preds)
    present_classes = sorted(set(all_labels))
    present_names = [CLASSES[i] for i in present_classes]
    report = classification_report(all_labels, all_preds, labels=present_classes,
                                   target_names=present_names, zero_division=0)
    print(f"\n{model_name} — Own dataset (5-class, metal excluded) balanced acc: {acc*100:.1f}%")
    print(report)
    return acc

print("="*60)
print("OWN DATASET — NEW 5-CLASS + BORDER-BG-AUG CHECKPOINTS")
print("="*60)

resnet_own = evaluate_own_dataset('resnet50', f'{SAVE_DIR}/resnet50_5class_full_best.pth', own_loader_eval, device)
effnet_own = evaluate_own_dataset('efficientnet', f'{SAVE_DIR}/efficientnet_5class_full_best.pth', own_loader_eval, device)
vit_own = evaluate_own_dataset('vit', f'{SAVE_DIR}/vit_5class_full_best.pth', own_loader_eval, device)

print("\n" + "="*60)
print("CALIBRATION ON OWN DATASET")
print("="*60)
own_calib_resnet = calibration_report('resnet50', f'{SAVE_DIR}/resnet50_5class_full_best.pth', val_loader, own_loader_eval, device)
own_calib_effnet = calibration_report('efficientnet', f'{SAVE_DIR}/efficientnet_5class_full_best.pth', val_loader, own_loader_eval, device)
own_calib_vit = calibration_report('vit', f'{SAVE_DIR}/vit_5class_full_best.pth', val_loader, own_loader_eval, device)

print("\n" + "="*60)
print("SUMMARY — Own Dataset vs RealWaste (both zero-shot, new 5-class checkpoints)")
print("="*60)
print(f"  ResNet-50:     RealWaste 58.0%  |  Own dataset {resnet_own*100:.1f}%")
print(f"  EfficientNet:  RealWaste 53.8%  |  Own dataset {effnet_own*100:.1f}%")
print(f"  ViT-Small:     RealWaste 63.0%  |  Own dataset {vit_own*100:.1f}%")

OWN DATASET — NEW 5-CLASS + BORDER-BG-AUG CHECKPOINTS


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")



resnet50 — Own dataset (5-class, metal excluded) balanced acc: 45.2%
              precision    recall  f1-score   support

   cardboard       0.48      0.41      0.44        27
       glass       0.32      0.50      0.39        18
       paper       0.38      0.56      0.46        41
     plastic       0.68      0.34      0.45        56

   micro avg       0.45      0.44      0.44       142
   macro avg       0.47      0.45      0.43       142
weighted avg       0.51      0.44      0.44       142



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")



efficientnet — Own dataset (5-class, metal excluded) balanced acc: 41.2%
              precision    recall  f1-score   support

   cardboard       0.43      0.74      0.55        27
       glass       0.21      0.44      0.29        18
       paper       0.22      0.12      0.16        41
     plastic       0.86      0.34      0.49        56

   micro avg       0.40      0.37      0.38       142
   macro avg       0.43      0.41      0.37       142
weighted avg       0.51      0.37      0.38       142



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")



vit — Own dataset (5-class, metal excluded) balanced acc: 55.3%
              precision    recall  f1-score   support

   cardboard       0.83      0.70      0.76        27
       glass       0.44      0.61      0.51        18
       paper       0.55      0.15      0.23        41
     plastic       0.58      0.75      0.66        56

   micro avg       0.60      0.55      0.57       142
   macro avg       0.60      0.55      0.54       142
weighted avg       0.60      0.55      0.53       142


CALIBRATION ON OWN DATASET

CALIBRATION ANALYSIS: resnet50

  In-distribution (val):
    Mean confidence: 0.889 | Accuracy: 0.982 | ECE: 0.097

  Out-of-distribution (RealWaste):
    Mean confidence: 0.689 | Accuracy: 0.437 | ECE: 0.252

  Overconfidence gap: +0.252
  (positive = model is more confident than it is accurate = overconfident)

  OOD Detection AUROC: 0.873
  (0.5 = confidence is useless for detecting OOD, 1.0 = perfect separation)

CALIBRATION ANALYSIS: efficientnet

  In-distribut

In [ ]:
def train_5class_lr(model_name, lr, num_epochs=20, weight_decay=2e-2, tag=""):
    print(f"\n{'='*60}")
    print(f"Training {model_name} — lr={lr}, 5-class, border-bg-aug {tag}")
    print('='*60)

    model = get_model(model_name, num_classes=5).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

    best_val_bal_acc = 0.0

    for epoch in range(1, num_epochs+1):
        model.train()
        tp, tl = [], []
        for images, labels in train_loader:  # the border-bg-aug loader from EXP028
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            out = model(images)
            loss = criterion(out, labels)
            loss.backward()
            optimizer.step()
            tp.extend(out.argmax(1).detach().cpu().numpy())
            tl.extend(labels.cpu().numpy())
        train_acc = balanced_accuracy_score(tl, tp)

        model.eval()
        vl, vp, vlabels = 0, [], []
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                out = model(images)
                vl += criterion(out, labels).item()
                vp.extend(out.argmax(1).cpu().numpy())
                vlabels.extend(labels.cpu().numpy())
        val_bal_acc = balanced_accuracy_score(vlabels, vp)
        avg_vl = vl / len(val_loader)
        scheduler.step()

        print(f"Epoch {epoch}/{num_epochs} | Train: {train_acc:.4f} | Val BalAcc: {val_bal_acc:.4f} | Loss: {avg_vl:.4f}")

        if val_bal_acc > best_val_bal_acc:
            best_val_bal_acc = val_bal_acc
            torch.save({'model_state_dict': model.state_dict()},
                       f'{SAVE_DIR}/{model_name}_5class_lr{lr}_best.pth')
            print(f"  ✓ saved")

    model.eval(); op, ol = [], []
    with torch.no_grad():
        for images, labels in test_loader:  # RealWaste
            images = images.to(device)
            op.extend(model(images).argmax(1).cpu().numpy())
            ol.extend(labels.numpy())
    ood_acc = balanced_accuracy_score(ol, op)
    print(f"\n>>> {model_name} (lr={lr}) Zero-shot RealWaste OOD: {ood_acc*100:.1f}% <<<")
    print(classification_report(ol, op, target_names=CLASSES, zero_division=0))
    return ood_acc

# Lower LR — 5x smaller than the original 1e-4
vit_lowlr_ood = train_5class_lr('vit', lr=2e-5, num_epochs=20, tag="(lower LR test)")

print("\n" + "="*60)
print("VIT LEARNING RATE COMPARISON")
print("="*60)
print(f"  ViT lr=1e-4 (original, EXP029): 63.0%")
print(f"  ViT lr=2e-5 (this run):          {vit_lowlr_ood*100:.1f}%")


Training vit — lr=2e-05, 5-class, border-bg-aug (lower LR test)
Epoch 1/20 | Train: 0.8619 | Val BalAcc: 0.9590 | Loss: 0.5230
  ✓ saved
Epoch 2/20 | Train: 0.9751 | Val BalAcc: 0.9701 | Loss: 0.4873
  ✓ saved
Epoch 3/20 | Train: 0.9935 | Val BalAcc: 0.9800 | Loss: 0.4709
  ✓ saved
Epoch 4/20 | Train: 0.9956 | Val BalAcc: 0.9801 | Loss: 0.4639
  ✓ saved
Epoch 5/20 | Train: 0.9967 | Val BalAcc: 0.9842 | Loss: 0.4551
  ✓ saved
Epoch 6/20 | Train: 0.9975 | Val BalAcc: 0.9838 | Loss: 0.4511
Epoch 7/20 | Train: 0.9979 | Val BalAcc: 0.9859 | Loss: 0.4516
  ✓ saved
Epoch 8/20 | Train: 0.9990 | Val BalAcc: 0.9865 | Loss: 0.4507
  ✓ saved
Epoch 9/20 | Train: 0.9968 | Val BalAcc: 0.9802 | Loss: 0.4572
Epoch 10/20 | Train: 0.9979 | Val BalAcc: 0.9829 | Loss: 0.4536
Epoch 11/20 | Train: 0.9983 | Val BalAcc: 0.9803 | Loss: 0.4526
Epoch 12/20 | Train: 0.9984 | Val BalAcc: 0.9859 | Loss: 0.4470
Epoch 13/20 | Train: 0.9986 | Val BalAcc: 0.9871 | Loss: 0.4498
  ✓ saved
Epoch 14/20 | Train: 0.9994 | Va

In [ ]:
from scipy.stats import wilcoxon

def get_per_image_correctness(model_name, checkpoint_path, loader, device):
    """Returns a 0/1 array: was each individual image classified correctly?
    This gives us paired per-sample data for Wilcoxon, not just one aggregate number."""
    model = get_model(model_name, num_classes=5)
    ckpt = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    model = model.to(device).eval()

    correctness = []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            preds = model(images).argmax(1).cpu().numpy()
            labels = labels.numpy()
            correctness.extend((preds == labels).astype(int))
    return np.array(correctness)

# Get per-image correctness for all 3 models on RealWaste (same test set, same order -> paired)
print("Computing per-image correctness for Wilcoxon testing...")
resnet_correct = get_per_image_correctness('resnet50', f'{SAVE_DIR}/resnet50_5class_full_best.pth', test_loader, device)
effnet_correct = get_per_image_correctness('efficientnet', f'{SAVE_DIR}/efficientnet_5class_full_best.pth', test_loader, device)
vit_correct    = get_per_image_correctness('vit', f'{SAVE_DIR}/vit_5class_full_best.pth', test_loader, device)

print(f"\nSample sizes: ResNet={len(resnet_correct)}, EfficientNet={len(effnet_correct)}, ViT={len(vit_correct)}")
print(f"Accuracy check: ResNet={resnet_correct.mean()*100:.1f}%, EfficientNet={effnet_correct.mean()*100:.1f}%, ViT={vit_correct.mean()*100:.1f}%")

def run_wilcoxon(name_a, correct_a, name_b, correct_b):
    """Wilcoxon signed-rank test on paired per-image correctness (0/1) differences."""
    diff = correct_a.astype(int) - correct_b.astype(int)
    n_nonzero = np.sum(diff != 0)
    if n_nonzero == 0:
        print(f"  {name_a} vs {name_b}: IDENTICAL on every sample, cannot test")
        return
    stat, p = wilcoxon(correct_a, correct_b)
    sig = "SIGNIFICANT (p < 0.05)" if p < 0.05 else "not significant"
    print(f"  {name_a} vs {name_b}: statistic={stat:.1f}, p-value={p:.6f} -> {sig}")

print("\n" + "="*60)
print("WILCOXON SIGNED-RANK TEST — RealWaste (5-class, border-bg-aug)")
print("="*60)
run_wilcoxon('ResNet-50', resnet_correct, 'EfficientNet', effnet_correct)
run_wilcoxon('ResNet-50', resnet_correct, 'ViT-Small', vit_correct)
run_wilcoxon('EfficientNet', effnet_correct, 'ViT-Small', vit_correct)

# Same test on own dataset (smaller sample, this is the one that most needs it)
print("\n" + "="*60)
print("WILCOXON SIGNED-RANK TEST — Own Dataset (142 images, smaller sample)")
print("="*60)
resnet_correct_own = get_per_image_correctness('resnet50', f'{SAVE_DIR}/resnet50_5class_full_best.pth', own_loader_eval, device)
effnet_correct_own = get_per_image_correctness('efficientnet', f'{SAVE_DIR}/efficientnet_5class_full_best.pth', own_loader_eval, device)
vit_correct_own    = get_per_image_correctness('vit', f'{SAVE_DIR}/vit_5class_full_best.pth', own_loader_eval, device)

run_wilcoxon('ResNet-50', resnet_correct_own, 'EfficientNet', effnet_correct_own)
run_wilcoxon('ResNet-50', resnet_correct_own, 'ViT-Small', vit_correct_own)
run_wilcoxon('EfficientNet', effnet_correct_own, 'ViT-Small', vit_correct_own)

Computing per-image correctness for Wilcoxon testing...

Sample sizes: ResNet=3092, EfficientNet=3092, ViT=3092
Accuracy check: ResNet=60.2%, EfficientNet=55.4%, ViT=65.9%

WILCOXON SIGNED-RANK TEST — RealWaste (5-class, border-bg-aug)
  ResNet-50 vs EfficientNet: statistic=139256.0, p-value=0.000000 -> SIGNIFICANT (p < 0.05)
  ResNet-50 vs ViT-Small: statistic=130311.5, p-value=0.000000 -> SIGNIFICANT (p < 0.05)
  EfficientNet vs ViT-Small: statistic=129888.0, p-value=0.000000 -> SIGNIFICANT (p < 0.05)

WILCOXON SIGNED-RANK TEST — Own Dataset (142 images, smaller sample)
  ResNet-50 vs EfficientNet: statistic=556.5, p-value=0.165518 -> not significant
  ResNet-50 vs ViT-Small: statistic=724.5, p-value=0.042153 -> SIGNIFICANT (p < 0.05)
  EfficientNet vs ViT-Small: statistic=472.0, p-value=0.000640 -> SIGNIFICANT (p < 0.05)


In [ ]:
from statsmodels.stats.contingency_tables import mcnemar

def run_mcnemar(name_a, correct_a, name_b, correct_b):
    """McNemar's test - the statistically correct test for paired binary classification comparisons."""
    # Build 2x2 contingency table: both right, A right B wrong, A wrong B right, both wrong
    both_right = np.sum((correct_a == 1) & (correct_b == 1))
    a_right_b_wrong = np.sum((correct_a == 1) & (correct_b == 0))
    a_wrong_b_right = np.sum((correct_a == 0) & (correct_b == 1))
    both_wrong = np.sum((correct_a == 0) & (correct_b == 0))
    table = [[both_right, a_right_b_wrong], [a_wrong_b_right, both_wrong]]
    result = mcnemar(table, exact=False, correction=True)
    sig = "SIGNIFICANT (p < 0.05)" if result.pvalue < 0.05 else "not significant"
    print(f"  {name_a} vs {name_b}: statistic={result.statistic:.2f}, p-value={result.pvalue:.6f} -> {sig}")

print("\n" + "="*60)
print("McNEMAR'S TEST (more rigorous for paired binary outcomes) — RealWaste")
print("="*60)
run_mcnemar('ResNet-50', resnet_correct, 'EfficientNet', effnet_correct)
run_mcnemar('ResNet-50', resnet_correct, 'ViT-Small', vit_correct)
run_mcnemar('EfficientNet', effnet_correct, 'ViT-Small', vit_correct)

print("\n" + "="*60)
print("McNEMAR'S TEST — Own Dataset")
print("="*60)
run_mcnemar('ResNet-50', resnet_correct_own, 'EfficientNet', effnet_correct_own)
run_mcnemar('ResNet-50', resnet_correct_own, 'ViT-Small', vit_correct_own)
run_mcnemar('EfficientNet', effnet_correct_own, 'ViT-Small', vit_correct_own)


McNEMAR'S TEST (more rigorous for paired binary outcomes) — RealWaste
  ResNet-50 vs EfficientNet: statistic=25.90, p-value=0.000000 -> SIGNIFICANT (p < 0.05)
  ResNet-50 vs ViT-Small: statistic=38.39, p-value=0.000000 -> SIGNIFICANT (p < 0.05)
  EfficientNet vs ViT-Small: statistic=116.51, p-value=0.000000 -> SIGNIFICANT (p < 0.05)

McNEMAR'S TEST — Own Dataset
  ResNet-50 vs EfficientNet: statistic=1.56, p-value=0.212003 -> not significant
  ResNet-50 vs ViT-Small: statistic=3.63, p-value=0.056780 -> not significant
  EfficientNet vs ViT-Small: statistic=10.78, p-value=0.001028 -> SIGNIFICANT (p < 0.05)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from google.colab import files
files.upload()  # upload waste-classification-main.zip

Mounted at /content/drive


Saving waste-classification-main.zip to waste-classification-main.zip


<Output stripped — was 69.3MB, likely raw file-upload byte dump>

In [ ]:
import zipfile, importlib.util, torch, torch.nn as nn, random, os
import numpy as np
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import balanced_accuracy_score, classification_report
from collections import Counter

with zipfile.ZipFile('/content/waste-classification-main.zip', 'r') as z:
    z.extractall('/content/')

PROJECT_ROOT = '/content/waste-classification-main'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SAVE_DIR = '/content/drive/MyDrive/waste_experiments_5class'
OLD_SAVE_DIR = '/content/drive/MyDrive/waste_experiments'  # original 6-class checkpoints

CLASSES = ['cardboard', 'glass', 'metal', 'paper', 'plastic']
CLASS_TO_IDX = {cls: idx for idx, cls in enumerate(CLASSES)}
CLASSES_6 = ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']  # for old checkpoints

def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

models_module = load_module("models", f"{PROJECT_ROOT}/src/models.py")
get_model = models_module.get_model

print(f"Device: {device}")
print(f"5-class checkpoints:", os.listdir(SAVE_DIR))
print(f"Original 6-class checkpoints:", os.listdir(OLD_SAVE_DIR))

Device: cuda
5-class checkpoints: ['resnet50_5class_full_best.pth', 'resnet50_5class_control_best.pth', 'efficientnet_5class_full_best.pth', 'vit_5class_full_best.pth', 'vit_5class_lr2e-05_best.pth']
Original 6-class checkpoints: ['efficientnet_best.pth', 'vit_best.pth', 'resnet50_combined_best.pth', 'efficientnet_combined_best.pth', 'vit_combined_best.pth', 'gradcam_resnet50.png', 'gradcam_efficientnet.png', 'gradcam_vit.png', 'resnet50_augmented_best.pth', 'resnet50_3dataset_best.pth', 'resnet50_partial_finetune_best.pth', 'efficientnet_partial_finetune_best.pth', 'vit_partial_finetune_best.pth']


In [ ]:
!pip install albumentations grad-cam kaggle -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 83.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
os.environ['KAGGLE_USERNAME'] = 'esmaeilmolapour'
os.environ['KAGGLE_KEY'] = 'YOUR_KAGGLE_KEY_HERE'
!kaggle datasets download -d joebeachcapital/realwaste -p {PROJECT_ROOT}/data/raw/realwaste/ --unzip -q
print("Downloaded.")

Dataset URL: https://www.kaggle.com/datasets/joebeachcapital/realwaste
License(s): Attribution 4.0 International (CC BY 4.0)
Downloaded.


In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2

def resize_pad_transform():
    return A.Compose([
        A.LongestMaxSize(max_size=224),
        A.PadIfNeeded(min_height=224, min_width=224, border_mode=cv2.BORDER_CONSTANT, fill=255),
        A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
        ToTensorV2(),
    ])

eval_transform = resize_pad_transform()

REALWASTE_MAP_5CLASS = {
    'Cardboard':'cardboard','Glass':'glass','Metal':'metal','Paper':'paper','Plastic':'plastic',
}
REALWASTE_MAP_6CLASS = {
    'Cardboard':'cardboard','Glass':'glass','Metal':'metal','Paper':'paper','Plastic':'plastic',
    'Miscellaneous Trash':'trash','Food Organics':'trash','Textile Trash':'trash','Vegetation':'trash',
}

def build_realwaste(root, class_map, class_to_idx):
    samples = []
    for folder, unified in class_map.items():
        class_dir = Path(root) / folder
        if not class_dir.exists(): continue
        idx = class_to_idx[unified]
        for img in class_dir.iterdir():
            if img.suffix.lower() in ['.jpg','.jpeg','.png']:
                samples.append((str(img), idx, unified))  # keep original class name too
    return samples

REALWASTE_ROOT = f'{PROJECT_ROOT}/data/raw/realwaste/realwaste-main/RealWaste'
realwaste_5class = build_realwaste(REALWASTE_ROOT, REALWASTE_MAP_5CLASS, CLASS_TO_IDX)
print(f"RealWaste samples for GradCAM: {len(realwaste_5class)}")

RealWaste samples for GradCAM: 3092


In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.reshape_transforms import vit_reshape_transform
import matplotlib.pyplot as plt

INV_NORMALIZE_MEAN = np.array([0.485, 0.456, 0.406])
INV_NORMALIZE_STD = np.array([0.229, 0.224, 0.225])

def get_target_layer(model, model_name):
    if model_name == 'resnet50':
        return [model.layer4[-1]]
    elif model_name == 'efficientnet':
        return [model.conv_head]
    elif model_name == 'vit':
        return [model.blocks[-1].norm1]

def load_checkpoint_model(model_name, checkpoint_path, num_classes):
    model = get_model(model_name, num_classes=num_classes)
    ckpt = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    return model.to(device).eval()

def samples_by_class(samples):
    """Group samples by class index - the stratification fix."""
    by_class = {}
    for path, idx, orig_name in samples:
        by_class.setdefault(idx, []).append((path, idx))
    return by_class

print("GradCAM utilities ready (stratified sampling fix included).")

GradCAM utilities ready (stratified sampling fix included).


In [ ]:
def generate_gradcam_stratified(model_name, checkpoint_path, samples, class_names,
                                 num_classes, output_path, n_per_class=1):
    print(f"Generating GradCAM for {model_name} -> {output_path}")
    model = load_checkpoint_model(model_name, checkpoint_path, num_classes)
    target_layers = get_target_layer(model, model_name)
    reshape = vit_reshape_transform if model_name == 'vit' else None
    cam = GradCAM(model=model, target_layers=target_layers, reshape_transform=reshape)

    by_class = samples_by_class(samples)
    correct_samples = {i: [] for i in range(len(class_names))}
    wrong_samples = {i: [] for i in range(len(class_names))}

    for class_idx, items in by_class.items():
        random.shuffle(items)
        for img_path, true_label in items:
            have_c = len(correct_samples[class_idx]) >= n_per_class
            have_w = len(wrong_samples[class_idx]) >= n_per_class
            if have_c and have_w:
                break
            img = cv2.imread(img_path)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            tensor = eval_transform(image=img)['image'].unsqueeze(0).to(device)
            with torch.no_grad():
                pred = model(tensor).argmax(1).item()
            if pred == true_label and not have_c:
                correct_samples[class_idx].append((true_label, pred, tensor))
            elif pred != true_label and not have_w:
                wrong_samples[class_idx].append((true_label, pred, tensor))

    flat_correct = [s for i in range(len(class_names)) for s in correct_samples[i]]
    flat_wrong = [s for i in range(len(class_names)) for s in wrong_samples[i]]
    n_cols = max(len(flat_correct), len(flat_wrong), 1)

    fig, axes = plt.subplots(4, n_cols, figsize=(n_cols*4, 16))
    if n_cols == 1: axes = axes.reshape(4,1)
    fig.suptitle(f'GradCAM — {model_name} (5-class, border-bg-aug) on RealWaste\n'
                 f'Rows 1-2: Correct | Rows 3-4: Wrong', fontsize=13, fontweight='bold')

    for col in range(n_cols):
        for row_pair, samples_list, color in [(0, flat_correct, 'green'), (2, flat_wrong, 'red')]:
            ax_top, ax_bot = axes[row_pair, col], axes[row_pair+1, col]
            if col >= len(samples_list):
                ax_top.axis('off'); ax_bot.axis('off'); continue
            true_label, pred, tensor = samples_list[col]
            gcam = cam(input_tensor=tensor, targets=[ClassifierOutputTarget(pred)])[0]
            rgb = tensor.squeeze().cpu().numpy().transpose(1,2,0)
            rgb = np.clip(rgb * INV_NORMALIZE_STD + INV_NORMALIZE_MEAN, 0, 1)
            viz = show_cam_on_image(rgb, gcam, use_rgb=True)
            mark = '✓' if color=='green' else '✗'
            ax_top.imshow(rgb); ax_top.set_title(f'True: {class_names[true_label]}', fontsize=9, color=color); ax_top.axis('off')
            ax_bot.imshow(viz); ax_bot.set_title(f'Pred: {class_names[pred]} {mark}', fontsize=9, color=color); ax_bot.axis('off')

    plt.tight_layout()
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved: {output_path}")

# Generate for all 3 new 5-class models
generate_gradcam_stratified('resnet50', f'{SAVE_DIR}/resnet50_5class_full_best.pth',
    realwaste_5class, CLASSES, 5, '/content/gradcam_resnet50_5class.png')
generate_gradcam_stratified('efficientnet', f'{SAVE_DIR}/efficientnet_5class_full_best.pth',
    realwaste_5class, CLASSES, 5, '/content/gradcam_efficientnet_5class.png')
generate_gradcam_stratified('vit', f'{SAVE_DIR}/vit_5class_full_best.pth',
    realwaste_5class, CLASSES, 5, '/content/gradcam_vit_5class.png')

Generating GradCAM for resnet50 -> /content/gradcam_resnet50_5class.png


model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]

Saved: /content/gradcam_resnet50_5class.png
Generating GradCAM for efficientnet -> /content/gradcam_efficientnet_5class.png


model.safetensors:   0%|          | 0.00/49.3M [00:00<?, ?B/s]

Saved: /content/gradcam_efficientnet_5class.png
Generating GradCAM for vit -> /content/gradcam_vit_5class.png


model.safetensors:   0%|          | 0.00/88.2M [00:00<?, ?B/s]

Saved: /content/gradcam_vit_5class.png


In [ ]:
def before_after_comparison(model_name, old_checkpoint, new_checkpoint,
                             old_classes, new_classes, sample_paths_labels,
                             n_examples=5, output_path='/content/before_after.png'):
    """
    Shows the same RealWaste images through both the OLD (6-class, no bg-aug)
    and NEW (5-class, border-bg-aug) checkpoints, GradCAM side by side.
    """
    old_model = load_checkpoint_model(model_name, old_checkpoint, len(old_classes))
    new_model = load_checkpoint_model(model_name, new_checkpoint, len(new_classes))

    old_target = get_target_layer(old_model, model_name)
    new_target = get_target_layer(new_model, model_name)
    reshape = vit_reshape_transform if model_name == 'vit' else None

    old_cam = GradCAM(model=old_model, target_layers=old_target, reshape_transform=reshape)
    new_cam = GradCAM(model=new_model, target_layers=new_target, reshape_transform=reshape)

    fig, axes = plt.subplots(3, n_examples, figsize=(n_examples*4, 12))
    fig.suptitle(f'Before / After Border-Background Augmentation — {model_name}\n'
                 f'Row 1: Original | Row 2: OLD model (6-class, no aug) | Row 3: NEW model (5-class, border-bg-aug)',
                 fontsize=13, fontweight='bold')

    for col, (img_path, class_name) in enumerate(sample_paths_labels[:n_examples]):
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        tensor = eval_transform(image=img)['image'].unsqueeze(0).to(device)
        rgb = tensor.squeeze().cpu().numpy().transpose(1,2,0)
        rgb = np.clip(rgb * INV_NORMALIZE_STD + INV_NORMALIZE_MEAN, 0, 1)

        with torch.no_grad():
            old_pred = old_model(tensor).argmax(1).item()
            new_pred = new_model(tensor).argmax(1).item()

        old_gcam = old_cam(input_tensor=tensor, targets=[ClassifierOutputTarget(old_pred)])[0]
        new_gcam = new_cam(input_tensor=tensor, targets=[ClassifierOutputTarget(new_pred)])[0]
        old_viz = show_cam_on_image(rgb, old_gcam, use_rgb=True)
        new_viz = show_cam_on_image(rgb, new_gcam, use_rgb=True)

        axes[0, col].imshow(rgb); axes[0, col].set_title(f'True: {class_name}', fontsize=9); axes[0, col].axis('off')
        axes[1, col].imshow(old_viz); axes[1, col].set_title(f'OLD pred: {old_classes[old_pred]}', fontsize=9, color='gray'); axes[1, col].axis('off')
        axes[2, col].imshow(new_viz); axes[2, col].set_title(f'NEW pred: {new_classes[new_pred]}', fontsize=9, color='blue'); axes[2, col].axis('off')

    plt.tight_layout()
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved: {output_path}")

# Pick 5 diverse examples across classes for the comparison
sample_pool = []
seen_classes = set()
for path, idx, orig_name in realwaste_5class:
    if idx not in seen_classes:
        sample_pool.append((path, CLASSES[idx]))
        seen_classes.add(idx)
    if len(seen_classes) == 5:
        break

before_after_comparison(
    'resnet50',
    f'{OLD_SAVE_DIR}/resnet50_combined_best.pth', f'{SAVE_DIR}/resnet50_5class_full_best.pth',
    CLASSES_6, CLASSES, sample_pool, n_examples=5,
    output_path='/content/before_after_resnet50.png'
)

Saved: /content/before_after_resnet50.png


In [ ]:
for var in ['realwaste_5class', 'eval_transform', 'CLASSES', 'CLASS_TO_IDX', 'get_model', 'device', 'SAVE_DIR']:
    print(var, ':', 'OK' if var in globals() else 'MISSING')

realwaste_5class : OK
eval_transform : OK
CLASSES : OK
CLASS_TO_IDX : OK
get_model : OK
device : OK
SAVE_DIR : OK


In [ ]:
class RealWasteEvalDataset(Dataset):
    def __init__(self, samples, transform):
        self.samples = samples  # (path, label_idx, orig_class_name) tuples
        self.transform = transform
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        img_path, label, _ = self.samples[idx]
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        return self.transform(image=image)['image'], label

test_dataset = RealWasteEvalDataset(realwaste_5class, eval_transform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)
print(f"test_loader ready: {len(realwaste_5class)} samples")

test_loader ready: 3092 samples


In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

def get_predictions(model_name, checkpoint_path, loader, num_classes, device):
    model = get_model(model_name, num_classes=num_classes)
    ckpt = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    model = model.to(device).eval()

    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            preds = model(images).argmax(1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())
    return np.array(all_labels), np.array(all_preds)

def plot_confusion_matrix(labels, preds, class_names, title, ax):
    cm = confusion_matrix(labels, preds, labels=range(len(class_names)))
    # Normalize by row (true class) so each row shows % of that true class's predictions
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names,
                vmin=0, vmax=1, ax=ax, cbar_kws={'label': 'Fraction of true class'})
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_title(title, fontsize=11, fontweight='bold')

# Get predictions for all 3 models on RealWaste (5-class, border-bg-aug checkpoints)
labels_r, preds_r = get_predictions('resnet50', f'{SAVE_DIR}/resnet50_5class_full_best.pth', test_loader, 5, device)
labels_e, preds_e = get_predictions('efficientnet', f'{SAVE_DIR}/efficientnet_5class_full_best.pth', test_loader, 5, device)
labels_v, preds_v = get_predictions('vit', f'{SAVE_DIR}/vit_5class_full_best.pth', test_loader, 5, device)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
plot_confusion_matrix(labels_r, preds_r, CLASSES, 'ResNet-50', axes[0])
plot_confusion_matrix(labels_e, preds_e, CLASSES, 'EfficientNet-B3', axes[1])
plot_confusion_matrix(labels_v, preds_v, CLASSES, 'ViT-Small (lr=2e-5)', axes[2])

fig.suptitle('Confusion Matrices — RealWaste (Zero-Shot, 5-class, Border-BG-Aug)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('/content/confusion_matrices_realwaste.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: confusion_matrices_realwaste.png")

Saved: confusion_matrices_realwaste.png


In [ ]:
def get_reliability_data(model_name, checkpoint_path, loader, num_classes, device, n_bins=10):
    """Returns per-bin (confidence, accuracy, count) for a reliability diagram."""
    model = get_model(model_name, num_classes=num_classes)
    ckpt = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    model = model.to(device).eval()

    all_conf, all_correct = [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            probs = torch.softmax(model(images), dim=1)
            conf, preds = probs.max(1)
            all_conf.extend(conf.cpu().numpy())
            all_correct.extend((preds.cpu().numpy() == labels.numpy()).astype(int))

    all_conf = np.array(all_conf)
    all_correct = np.array(all_correct)

    bin_edges = np.linspace(0, 1, n_bins + 1)
    bin_centers, bin_accs, bin_counts = [], [], []
    for i in range(n_bins):
        lo, hi = bin_edges[i], bin_edges[i+1]
        mask = (all_conf > lo) & (all_conf <= hi)
        if mask.sum() > 0:
            bin_centers.append(all_conf[mask].mean())
            bin_accs.append(all_correct[mask].mean())
            bin_counts.append(mask.sum())
        else:
            bin_centers.append((lo+hi)/2)
            bin_accs.append(np.nan)
            bin_counts.append(0)
    return np.array(bin_centers), np.array(bin_accs), np.array(bin_counts)

# Compute for all 3 models on RealWaste (5-class, border-bg-aug)
rn_c, rn_a, rn_n = get_reliability_data('resnet50', f'{SAVE_DIR}/resnet50_5class_full_best.pth', test_loader, 5, device)
ef_c, ef_a, ef_n = get_reliability_data('efficientnet', f'{SAVE_DIR}/efficientnet_5class_full_best.pth', test_loader, 5, device)
vt_c, vt_a, vt_n = get_reliability_data('vit', f'{SAVE_DIR}/vit_5class_lr2e-05_best.pth', test_loader, 5, device)

fig, ax = plt.subplots(figsize=(8, 8))
ax.plot([0,1], [0,1], 'k--', alpha=0.5, label='Perfect calibration')
ax.plot(rn_c, rn_a, 'o-', label='ResNet-50', color='#1f77b4', markersize=6)
ax.plot(ef_c, ef_a, 's-', label='EfficientNet-B3', color='#2ca02c', markersize=6)
ax.plot(vt_c, vt_a, '^-', label='ViT-Small', color='#d62728', markersize=6)

ax.set_xlabel('Confidence (mean predicted probability)', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Reliability Diagram — RealWaste (OOD, 5-class)', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
ax.set_xlim(0,1); ax.set_ylim(0,1)

plt.tight_layout()
plt.savefig('/content/reliability_diagram.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: reliability_diagram.png")

Saved: reliability_diagram.png
